# TabICLv2 Classifier — DIMER E2E tabular classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabicl-classifier-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabicl-classifier-pipeline/blob/main/tutorials/tabiclv2_classifier_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-jingang%2FTabICL-ffcc4d?style=flat)](https://huggingface.co/jingang/TabICL) [![Upstream](https://img.shields.io/badge/Upstream-soda--inria%2Ftabicl-181717?style=flat&logo=github&logoColor=white)](https://github.com/soda-inria/tabicl) [![arXiv](https://img.shields.io/badge/arXiv-2602.11139-b31b1b.svg)](https://arxiv.org/abs/2602.11139)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 3.0 — **standalone** (§4)  
**Notebook kind:** `TUTORIAL`  
**Capability:** end-to-end TabICLv2 tabular classification: immutable checkpoint acquisition, validated support data with class coverage, in-context evaluation and bounded gradient fine-tuning against trivial and classical baselines, new-data inference with class probabilities, a DIMER-style serving bundle and its fresh reload

**This notebook is standalone.** Section 2 carries the repository's package (2 modules under `src/tabicl_classifier_pipeline/`, at revision `ade4356ae264`), the stage runner, the hash-locked requirements (90 packages) and the pinned checkpoint manifest, and verifies every carried file against its SHA-256 before use, so the notebook keeps working after export even if the repository changes or disappears. Nothing is installed into the notebook kernel: a pinned `uv` (checked by size and SHA-256) builds an isolated Python environment from the lock with `--require-hashes`, and every stage runs there in its own process, so the hosted runtime's own packages are never replaced and no restart is needed. Its only external dependencies are PyPI, the managed CPython build that `uv` downloads, and the Hugging Face Hub at the immutable revision `4dcd344ece2c00be9e831fdd35bed57b5ad83e19` (~110 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/3.0-tabular); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh Linux x86_64 runtime (a T4 GPU is the documented runtime) builds an isolated Python environment from the carried hash-locked requirements without touching the notebook kernel's own packages, then runs each stage below in its own process: it stages and digest-verifies the pinned TabICLv2 checkpoint, fetches the pinned MAGIC Gamma Telescope split archive (3,000 support, 1,000 validation and a 1,500-row fresh test) by immutable commit and verifies its SHA-256, validates and encodes the tables, evaluates the pretrained TabICLv2 classifier **adapted by in-context conditioning on the support rows**, runs a **pre-registered three-config fine-tuning grid** that takes real optimizer steps and keeps the pretrained model unless a config beats it on validation log loss by more than 0.005, compares LightGBM, a random forest and a majority-class baseline, scores the fresh test **only under the pre-registered settings** (a different setting writes an exploratory record and reads nothing), scores new rows, exports a DIMER-style serving bundle and reloads it in a fresh process, and finally, in the closing external-artifact section, downloads the pinned sample artifact that a recorded earlier run of this notebook exported (a GitHub release asset, accepted only at its pinned SHA-256 and never this run's export), validates it, rebuilds serving state from its files alone and scores eight MAGIC rows. On a runtime without CUDA the grid is skipped with a printed note and the run is exploratory. Set `DIMER_MASK_FRESH_TEST=1` to skip the fresh test. No repository clone, DIMER worker or service, credential, upload dialog, configuration edit or runtime restart is required (NOTEBOOK_SPEC 3.0 §5). Recorded runs of this notebook are listed in `docs/release-verification.md`; this revision's own run is recorded there once it exists.

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` and `DATA_SOURCE` to `Upload CSV` or `Upload pre-split train/val/test` in Section 4 and run Sections 4–9 again. Give the files by path (`BYOD_CSV_PATH`, or `BYOD_TRAIN_PATH` / `BYOD_VAL_PATH` / optional `BYOD_TEST_PATH` — these work in Colab, Kaggle and Jupyter), or leave the paths empty in Colab to use upload dialogs. Name your label column in `TARGET_COLUMN`. A missing label column is refused naming it; rows with a missing label are dropped **and counted in the input manifest**; a column that is numeric except for a few stray strings is refused naming the column and the values (list it in `TEXT_COLUMNS` if it really is categorical). `RUN_NEW_DATA_INFERENCE` in Section 8 scores your own unlabelled rows; in Section 10, `USE_OWN_ROWS` scores your own rows with the external artifact, and `ARTIFACT_SOURCE` can load your own artifact ZIP instead of the pinned sample (both off by default). Files stay inside this runtime. BYOD is optional, never part of the default path, and always exploratory: a BYOD run never reads the fresh test.

TabICLv2 (Qu et al.) is an **in-context tabular foundation model**: a transformer pretrained on synthetic tables to classify a query row by reading labelled example rows — the **support rows** or **context** — alongside it. `fit` registers the (encoded) support rows as context and every `predict_proba` call feeds context plus query rows through the transformer and reads off a class distribution per query row; no gradient step happens there. `n_estimators=8` averages eight passes with different feature and row permutations. The DIMER pipeline's production workflow goes one step further: it **fine-tunes** the checkpoint by gradient descent on the support rows (AdamW, early stopping on the holdout) and keeps the fine-tuned model only if it beats the pretrained one on the holdout. This notebook runs both: conditioning, and a pre-registered fine-tuning grid on a GPU — and it shows honestly that **fine-tuning a strong in-context model often does not help**.

The upstream project supplies the model and the checkpoint; the carried package adds the pinned snapshot scheme, the table preparation, class-coverage and encoding rules, the metric set, the `validate_inputs` / `majority_class_baseline` / `evaluation_report` helpers, and the serving-bundle safety checks. `predict` applies an implicit `argmax` over class probabilities that are raw ensemble outputs, **not calibrated probabilities**; the package ships no acceptance threshold. The default sample is the UCI MAGIC Gamma Telescope table [3]: harder than the earlier Breast Cancer default, with a 1,500-row fresh test that resolves differences of a few tenths of a percent. Its metrics are tutorial evidence, not a benchmark or production claim.

**Learning objectives:** by the end of this notebook you will be able to —

1. **Explain** the difference between in-context conditioning (registering support rows) and gradient fine-tuning, read a fine-tuning curve, the optimizer-step count and a parameter delta, and say why a pretrained model that tabicl restores is a finding, not a failure (Section 6).
2. **Diagnose** an invalid table from a validation refusal, and say what a dropped-row count in the input manifest means (Sections 4, 5).
3. **Compare** TabICLv2 with a majority-class baseline, LightGBM and a random forest on the same partitions, using error counts and the one-row resolution (Section 7).
4. **Interpret** accuracy, balanced accuracy, weighted F1, log loss and ROC-AUC, and why a blend weight chosen on validation is selection-biased (Section 7); read a pre-registered fresh-test comparison with its interval, and say why changing a setting makes a run exploratory (Section 7b).
5. **Apply** the selected model to new rows and read class-ordered, uncalibrated probabilities under an explicit `argmax` rule (Section 8).
6. **Verify** that the exported bundle rebuilds an equivalent classifier in a fresh process (Section 9).
7. **Validate and serve** an artifact made outside this run: check its pinned digest before opening it, read its provenance and format, rebuild serving state from its files, and say how that differs from the fresh reload (Section 10).
8. **Predict**, run and **explain** how a three-class table with 36 holdout rows changes the one-row resolution and ROC-AUC, in an optional activity (Section 11).
9. **Write** an evidence-based conclusion that names the baselines, the selection basis and the limits of one public table (Conclusion).

**This notebook does not demonstrate:** regression, forecasting, calibrated probabilities, or any deployment threshold. `prediction` is the `argmax` over uncalibrated class probabilities; fine-tuning runs only on CUDA, and its selection is evidence on one holdout, not a guarantee of better generalisation.

> **⚠️ Notice.** For research, training and evaluation only. Not validated for production,
> clinical, diagnostic or safety-critical use. Running this notebook downloads third-party
> model weights and datasets under their own licenses (see the model card) and uses compute
> on your own Colab or Kaggle account. Data you bring stays in your runtime, and you are
> responsible for having the right to use it. Results here are tutorial evidence, not
> benchmarks. Everything is provided "as is", without warranty of any kind. To the maximum
> extent permitted by law, the maintainers and the DIMER platform accept no liability for
> its use.

## How to use this notebook

**Who this notebook is for.** Learners who can run cells in a hosted notebook (Google Colab or Jupyter) and read short Python and pandas, and who want to see a tabular foundation model used the way a production pipeline uses it: conditioned, optionally fine-tuned with honest holdout selection, compared with strong classical models, and packaged as a reusable bundle. No experience with transformers is assumed; each term is explained where it is first needed, and the glossary below collects them.

**Running it.** In Colab choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. The default path needs no edit, no upload, no account, no token and no runtime restart. Section 2 builds an isolated environment, which takes the longest (PyTorch and its CUDA libraries are a few GB); Section 6 then runs the pre-registered fine-tuning grid on the GPU (the whole notebook took about 7.5 minutes on a hosted T4 in the 2026-10-20 run of record). You can also run one cell at a time with *Shift + Enter*.

**Where the code runs.** The notebook kernel installs nothing and imports no model library. Each learner cell calls `run_stage('…')`, which runs one stage of the carried stage runner in its own process with the isolated environment's Python, streams what it prints, and stops the notebook with the stage's own error message if it fails. Stages hand results to each other only through files: the verified checkpoint, the partitions and fitted encoders, the fine-tuned checkpoint if one was selected, and JSON records.

**Two kinds of cell.** *Learner cells* (Sections 4–11) are the machine-learning workflow. *Infrastructure cells* (Sections 1–3: the runtime check, the carried code, the isolated environment and the checkpoint staging) are collapsed and titled **Infrastructure**. You may run them without studying their implementation: they exist for reproducibility and provenance, not as prerequisite machine-learning knowledge.

**Form controls.** `DATA_SOURCE`, `USE_BYOD`, `TARGET_COLUMN`, the `BYOD_*_PATH` fields and `TEXT_COLUMNS` (Section 4); `RUN_FINE_TUNING`, `GRID_EPOCHS_CAP`, `GRID_LEARNING_RATE` and `RANDOM_STATE` (Section 6); `RUN_NEW_DATA_INFERENCE` and `NEW_DATA_PATH` (Section 8); `ARTIFACT_SOURCE`, `ARTIFACT_ZIP_PATH`, `EXPECTED_ZIP_SHA256`, `USE_OWN_ROWS` and `EXTERNAL_ROWS_PATH` (Section 10); `RUN_ACTIVITY` and `ACTIVITY_DATA_SOURCE` (Section 11). Leave them at their defaults for the first run: changing any of them makes the run exploratory (validation only; the fresh test is not read). The environment variable `DIMER_MASK_FRESH_TEST=1` skips the fresh test; stages run with `DIMER_ISOLATED_ENV` set to the isolated environment's path.

**Section tags.** **[Concept]** — what the model does and why. **[Evaluation practice]** — how the evidence is produced and how to read it. **[Engineering]** — reproducibility, provenance and packaging.

**Predict, then check.** Before Sections 6 and 7 a **Predict before running** prompt asks you to commit to an expectation; **What to notice** follows each stage; a collapsed **Check your reasoning** answer follows each checkpoint. The stages switch on PyTorch's deterministic algorithms and fixed seeds; Section 6 prints a grid hash for information: on a T4 it can differ between two runs of the same notebook revision, while the selection and the fresh-test values match exactly.

## The task: Input → Model → Output

| Stage | Input | Model / system | Output |
|---|---|---|---|
| **Validate** | the pinned MAGIC split | `validate_inputs`, support-fitted categorical encoders | an input manifest; 3,000 support and 1,000 validation rows (the 1,500-row fresh test is not loaded yet) |
| **Condition** | the support rows | TabICLv2 reading them as context (`n_estimators=8`) | class probabilities for any query row |
| **Fine-tune (GPU)** | support rows; validation for early stopping | a pre-registered three-config grid, 6 optimizer steps per epoch | a selection on validation log loss; pretrained kept unless beaten by > 0.005 |
| **Fresh test** | 1,500 never-scored rows | the pre-registered reading rule, score-once and fail-closed | Δ log loss (selected − pretrained) with a paired bootstrap interval |
| **Evaluate** | validation rows | `classification_metrics`, majority baseline, LightGBM, random forest, an in-memory blend | metrics, error counts, an evaluation report |
| **Package** | selected checkpoint + support rows + settings | ZIP bundle, `safe_extract_zip`, `verify_artifact_bundle` | a bundle that rebuilds an equivalent classifier |
| **Serve elsewhere** | a pinned artifact from an earlier recorded run + eight MAGIC rows | digest check, `safe_extract_zip`, `validate_artifact_runtime`, `verify_artifact_bundle` | predictions under `outputs/external_artifact/` |

## Roadmap

| Section | Tag | What happens | What you read |
|---|---|---|---|
| 1. Check the runtime | [Engineering] | Linux x86_64, GPU and disk; a run directory | the machine |
| 2. Carry the code, build the environment | [Engineering] | carried files verified; an isolated hash-locked environment | versions |
| 3. Pin, stage and verify the model | [Engineering] | the 110 MB checkpoint downloaded at a fixed revision, digest-checked | identity and digest |
| 4. Load the sample or your own data | [Concept] | the pinned MAGIC split (or your files) | partition sizes, digests, the mask |
| 5. Validate and encode | [Evaluation practice] | input manifest; a refusal probe; encoders; majority baseline | the manifest |
| 6. Condition and fine-tune | [Concept] | pretrained metrics; the pre-registered grid; validation selection | curves, steps, parameter deltas, the selection |
| 7. Baselines, blend and report | [Evaluation practice] | LightGBM, random forest, blend; the evaluation report | the validation comparison |
| 7b. Fresh test | [Evaluation practice] | score-once, pre-registered reading rule | the principal result and its `test_look` |
| 8. New-data inference | [Engineering] | eight rows (or your file) scored | the output contract |
| 9. Export and fresh reload | [Engineering] | the bundle exported and rebuilt in a new process | the reload check and the digest |
| Interpretation and conclusion | [Evaluation practice] | limits and an evidence-based conclusion | your conclusion |
| 10. Using an artifact made elsewhere | [Engineering] | the pinned sample artifact verified, rebuilt and used to score MAGIC rows | provenance, digests, predictions |
| 11. Optional activity | [Concept] | the three-class Wine table (off by default) | your comparison |
| Troubleshooting | [Engineering] | common failures and what to do | when something fails |
| References | | sources and repository documents | |

**Fast path.** Run all, then read Sections 6, 7b, 9 and 10 and the conclusion.

<details>
<summary><strong>Glossary</strong> — open when a term is unfamiliar</summary>

| Term | Meaning in this notebook |
|---|---|
| **In-context learning / conditioning** | Predicting from labelled rows shown to the model at prediction time; `fit` only registers them. |
| **Support rows / context** | The labelled rows the model reads; here the 3,000-row support partition. |
| **`n_estimators`** | How many passes with different feature/row permutations are averaged into one prediction. |
| **Fine-tuning** | Updating the checkpoint's weights by gradient descent on the support rows. |
| **Early stopping** | Stopping fine-tuning when the validation metric stops improving; tabicl then restores the best state. |
| **Validation (holdout)** | The 1,000 rows that early stopping and the grid selection look at. |
| **Fresh test** | 1,500 rows no choice has seen, scored only under the pre-registered settings. |
| **Exposed development test** | A test whose numbers were already seen in earlier runs (the Breast Cancer 114 rows). |
| **Pre-registration** | Writing down settings, selection and reading rule before running, so no result can steer them. |
| **Exploratory run** | Any run whose settings differ from the pre-registration: compared on validation only. |
| **Optimizer step / parameter delta** | One AdamW update; the L2 norm of how far the weights moved from the pretrained ones. |
| **Paired bootstrap interval** | Resampling the same rows for both models to bound their difference. |
| **Selection bias** | A score looks better than it will on new data because a choice was made by looking at it. |
| **One-row resolution** | 1 / (rows in the partition): the smallest possible change in accuracy. 1,000 rows → 0.001. |
| **Majority-class baseline** | Always predict the most frequent support class. |
| **LightGBM / random forest** | Strong classical tree ensembles trained on the same support rows. |
| **Blend** | A weighted average of TabICLv2's and LightGBM's probabilities, weight chosen on the holdout. |
| **Accuracy / balanced accuracy / weighted F1** | Discrete correctness under the `argmax` rule; balanced accuracy averages per-class recall. |
| **Log loss** | Average negative log probability of the true class; lower is better; punishes confident mistakes. |
| **ROC-AUC (one-vs-rest)** | Ranking quality, threshold-free; for several classes, averaged over one-class-vs-rest problems. |
| **Uncalibrated probability** | A score not guaranteed to match observed frequencies. |
| **Categorical encoder** | Ordinal codes fitted on the support rows only; unseen values get an 'unknown' code and are counted. |
| **Serving bundle** | A ZIP of `artifact.json`, `checkpoints/best.ckpt` and `training_context.parquet`. |
| **Digest (SHA-256)** | A fingerprint of a file's bytes. |
| **Hash-locked environment / stage** | The isolated Python environment every stage runs in; one workflow step run as its own process. |
| **BYOD** | Bring Your Own Data. |

</details>

## Prerequisites

- **Runtime:** a fresh **Linux x86_64** runtime — Google Colab with a **T4 GPU** is the documented runtime (the pre-registered fine-tuning grid needs CUDA, and only a Tesla T4 run scores the fresh test); Kaggle or a Linux Jupyter kernel also work, and a CPU-only runtime runs everything except the grid, which it skips with a printed note (that run is exploratory). The kernel's own Python version does not matter: the notebook installs nothing into it, and runs every stage with CPython 3.12.12 in an isolated environment built from 90 hash-locked packages (`tabicl` 2.1.1 with its fine-tuning extras, `torch` 2.14.0 with its CUDA libraries, `numpy` 2.5.2, `pandas` 3.0.5, `scikit-learn` 1.9.0, `lightgbm` 4.7.0). About 0.2 GB of disk is needed for the checkpoint and about 8 GB for the isolated environment.
- **Knowledge:** basic pandas, what a train/holdout/test split is, and how to read a printed Python dictionary. The metrics, in-context conditioning and fine-tuning are explained where they are first used, and the glossary collects them.
- **Model file:** one checkpoint, `tabicl-classifier-v2-20260212.ckpt` (110 MB, released under BSD-3-Clause). It is a PyTorch/Lightning checkpoint: `tabicl` deserialises it, so only the digest-verified file is ever loaded.
- **Data:** the default sample is the UCI MAGIC Gamma Telescope table (19,020 simulated events, 10 numeric features, two classes, CC BY 4.0) as the repository's pinned split archive: 3,000 support, 1,000 validation and a 1,500-row fresh test, plus a 13,520-row reserve the notebook never reads; it is fetched by immutable commit and its SHA-256 is verified (`examples/sample-data/DATASET_CARD.md`). `Sample: Breast Cancer` (the earlier default, scikit-learn's bundled table) and `Sample: Wine` need no download; `Sample: Palmer Penguins` fetches the repository's pinned CC0 archive. BYOD (one CSV with a label column, or pre-split `train`/`val`/`test` CSVs) is off by default so the sample path runs top to bottom without interaction. The tutorial's own floors are `MIN_TRAIN_ROWS` = 50 support rows and `MIN_EVAL_ROWS` = 2 evaluation rows, with no class ceiling; the production DIMER validator described in the model card is stricter. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** `huggingface.co` (no account or token), to fetch the pinned `jingang/TabICL` checkpoint (~110 MB) at revision `4dcd344ece2c…`, accepted only at the pinned size and SHA-256; PyPI (`pypi.org`, `files.pythonhosted.org`), for the pinned `uv` wheel and the 90 hash-locked packages; and the managed CPython build (python-build-standalone) that `uv` downloads for the isolated environment. No repository clone and no credentials are required; nothing is installed from this repository.

## 1. Check the runtime · [Engineering]

> **Infrastructure.** The code cells in Sections 1–3 are collapsed and titled **Infrastructure**. You may run them without studying their implementation; they exist for reproducibility and provenance, not as prerequisite machine-learning knowledge. The learning activities start in Section 4.

**Input:** a fresh hosted runtime. **System:** checks that it is Linux x86_64 with enough free disk, looks for a CUDA GPU, and creates a run directory under `dimer_runs/tabiclv2_classifier/` for the carried code, the stage logs and the hand-off state between stages. Learner-facing files are written to `outputs/`. **Output:** one dictionary naming the accelerator, the kernel's Python version and the directories. Running this cell again in the same session keeps the run directory, so the cells after it keep working; tick `NEW_RUN_DIRECTORY` for a fresh one and run Sections 2 and 3 again. The verified checkpoint is kept in `weights/` and reused.

In [ ]:
# @title Infrastructure: check the runtime, accelerator and disk; create (or keep) the run directory
NEW_RUN_DIRECTORY = False  # @param {type:"boolean"}
import base64
import hashlib
import json
import os
import platform
import shutil
import subprocess
import tempfile
import time
import uuid
from pathlib import Path

SESSION_START = time.perf_counter()
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle, or a Linux x86_64 Jupyter kernel): its locked environment is built for manylinux x86_64 wheels. A GPU is recommended, not required.')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    accelerator = gpu.stdout.strip() if gpu.returncode == 0 else ''
except FileNotFoundError:
    accelerator = ''
if not accelerator:
    accelerator = 'none (CPU only)'
    print('No CUDA GPU detected. The notebook still runs on the CPU, more slowly (see Prerequisites). For the documented runtime choose Runtime > Change runtime type > T4 GPU before Run all.')
STEM = 'tabiclv2_classifier'
# Re-running this cell keeps the run directory of this session, so the cells after it keep working; tick
# NEW_RUN_DIRECTORY (then run the cells below again) for a fresh one.
if NEW_RUN_DIRECTORY or not isinstance(globals().get('ROOT'), Path) or not globals()['ROOT'].is_dir():
    ROOT = Path.cwd() / 'dimer_runs' / STEM / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUTS = Path.cwd() / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
WEIGHTS = Path.cwd() / 'weights'
WEIGHTS.mkdir(exist_ok=True)
ENV_ROOT = Path(tempfile.gettempdir()) / 'dimer_uv'
staged_gib = sum(p.stat().st_size for p in WEIGHTS.rglob('*') if p.is_file()) / 1024**3
need = {'weights': max(0.0, 0.2 - staged_gib), 'environment': 8.0}
free = {'weights': shutil.disk_usage(WEIGHTS).free / 1024**3, 'environment': shutil.disk_usage(tempfile.gettempdir()).free / 1024**3}
if os.stat(WEIGHTS).st_dev == os.stat(tempfile.gettempdir()).st_dev:
    short = free['weights'] < need['weights'] + need['environment']
else:
    short = free['weights'] < need['weights'] or free['environment'] < need['environment']
if short:
    raise RuntimeError(f'Not enough free disk: need about {need} GiB, free {free} GiB. Start a fresh runtime (see Troubleshooting).')
print({'accelerator': accelerator, 'kernel_python': platform.python_version(), 'run_directory': str(ROOT), 'outputs': str(OUTPUTS), 'weights': str(WEIGHTS), 'environment_root': str(ENV_ROOT), 'free_gib': {k: round(v, 1) for k, v in free.items()}})

## 2. Carry the code and build the isolated environment · [Engineering]

> **Infrastructure.** The next two code cells are collapsed. The first **is** the repository's code, carried so that this notebook works on its own; the second builds the environment every stage runs in.

The first cell holds, as text, the files the workflow needs: the package's 2 modules under `src/tabicl_classifier_pipeline/` (at revision `ade4356ae264`), the stage runner `tutorial_stages.py`, the hash-locked `requirements.txt` (90 packages), the pinned checkpoint manifest and the licence. It writes each file into the run directory and checks its SHA-256 against `CARRIED_HASHES`, stopping on any mismatch. The text is the repository's files byte for byte; the repository's parity tests fail whenever the two diverge, so what runs here is what the repository tests. Nothing in this cell runs a model.

**Expected result:** `carried_files`, `verified: True`, and the repository revision the notebook was generated from.

In [ ]:
# @title Infrastructure: write and verify the carried package, stage runner, lock, manifest and samples
CARRIED_FILES = {'src/tabicl_classifier_pipeline/__init__.py': 'from .api import (\n    ARTIFACT_FORMAT,\n    BASE_CHECKPOINT_NAME,\n    BASE_MODEL_REPO,\n    BASE_MODEL_REVISION,\n    BASE_MODEL_SHA256,\n    DECISION_RULE,\n    DEFAULT_N_ESTIMATORS,\n    DEFAULT_RANDOM_STATE,\n    DEFAULT_WEIGHTS_DIR,\n    INPUT_SCHEMA,\n    MANIFEST_NAME,\n    MAX_ARTIFACT_EXPANDED_BYTES,\n    MAX_FEATURES,\n    MAX_TRAIN_ROWS,\n    METRIC_IDS,\n    MIN_CLASSES,\n    MIN_EVAL_ROWS,\n    MIN_ROWS_PER_CLASS,\n    MIN_TRAIN_ROWS,\n    MODEL_ID,\n    MODEL_KEY,\n    MODEL_LICENSE,\n    MODEL_REVISION,\n    TabICLClassificationPipeline,\n    align_probabilities,\n    align_to_schema,\n    apply_categorical_encoder,\n    check_stratifiable,\n    classification_metrics,\n    compare_metric,\n    condition_classifier,\n    create_classifier,\n    create_finetuned_classifier,\n    download_output,\n    evaluation_report,\n    fine_tune_classifier,\n    fit_categorical_encoder,\n    majority_class_baseline,\n    manifest_member_path,\n    predict_labels,\n    predict_probabilities,\n    prepare_classification_table,\n    raw_csv_header,\n    read_csv_payload,\n    read_inference_csv,\n    read_single_input,\n    renormalise_probabilities,\n    runtime_identity,\n    safe_extract_zip,\n    sha256_file,\n    stage_missing_files,\n    validate_artifact_runtime,\n    validate_inputs,\n    verify_artifact_bundle,\n    verify_snapshot,\n)\n\n__all__ = [\n    "ARTIFACT_FORMAT",\n    "BASE_CHECKPOINT_NAME",\n    "BASE_MODEL_REPO",\n    "BASE_MODEL_REVISION",\n    "BASE_MODEL_SHA256",\n    "DECISION_RULE",\n    "DEFAULT_N_ESTIMATORS",\n    "DEFAULT_RANDOM_STATE",\n    "DEFAULT_WEIGHTS_DIR",\n    "INPUT_SCHEMA",\n    "MANIFEST_NAME",\n    "MAX_ARTIFACT_EXPANDED_BYTES",\n    "MAX_FEATURES",\n    "MAX_TRAIN_ROWS",\n    "METRIC_IDS",\n    "MIN_CLASSES",\n    "MIN_EVAL_ROWS",\n    "MIN_ROWS_PER_CLASS",\n    "MIN_TRAIN_ROWS",\n    "MODEL_ID",\n    "MODEL_KEY",\n    "MODEL_LICENSE",\n    "MODEL_REVISION",\n    "TabICLClassificationPipeline",\n    "align_probabilities",\n    "align_to_schema",\n    "apply_categorical_encoder",\n    "check_stratifiable",\n    "classification_metrics",\n    "compare_metric",\n    "condition_classifier",\n    "create_classifier",\n    "create_finetuned_classifier",\n    "download_output",\n    "evaluation_report",\n    "fine_tune_classifier",\n    "fit_categorical_encoder",\n    "majority_class_baseline",\n    "manifest_member_path",\n    "predict_labels",\n    "predict_probabilities",\n    "renormalise_probabilities",\n    "prepare_classification_table",\n    "raw_csv_header",\n    "read_csv_payload",\n    "read_inference_csv",\n    "read_single_input",\n    "runtime_identity",\n    "safe_extract_zip",\n    "sha256_file",\n    "stage_missing_files",\n    "validate_artifact_runtime",\n    "validate_inputs",\n    "verify_artifact_bundle",\n    "verify_snapshot",\n]\n', 'src/tabicl_classifier_pipeline/api.py': '# ruff: noqa: E501  -- long docstrings, messages and single-line test fixtures are kept readable\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport importlib.metadata\nimport io\nimport json\nimport math\nimport os\nimport shutil\nimport stat\nimport sys\nimport zipfile\nfrom collections.abc import Callable, Mapping, Sequence\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\n\nARTIFACT_FORMAT = "tabicl-dimer-classifier-v1"\n\n# Fleet snapshot identity (DIMER Notebook Specification 1.1, ST3/MOD13). The pinned upstream checkpoint is\n# unchanged; these are the fleet-standard names for the same repository, revision, license and snapshot key.\n# The BASE_* spellings below stay as the package\'s published names and alias these constants.\nMODEL_ID = "jingang/TabICL"\nMODEL_REVISION = "4dcd344ece2c00be9e831fdd35bed57b5ad83e19"\nMODEL_LICENSE = "bsd-3-clause"\nMODEL_KEY = "tabicl-classifier-v2"\nMANIFEST_NAME = "dimer-base-manifest.json"\nDEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\n\nBASE_MODEL_REPO = MODEL_ID\nBASE_CHECKPOINT_NAME = "tabicl-classifier-v2-20260212.ckpt"\nBASE_MODEL_REVISION = MODEL_REVISION\nBASE_MODEL_SHA256 = "bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0"\n\n# Operational ceilings the tutorials enforce before any model execution (DAT22).\nMIN_TRAIN_ROWS = 50  # labelled support rows required to condition the classifier\nMIN_EVAL_ROWS = 2  # labelled rows required for a holdout / test partition\nMAX_TRAIN_ROWS = 50_000  # support rows per conditioning call\nMAX_FEATURES = 2_000  # feature columns per table\nMAX_ARTIFACT_EXPANDED_BYTES = 1024 * 1024 * 1024  # 1 GiB expanded ZIP size for a serving artifact\nDEFAULT_N_ESTIMATORS = 8\nDEFAULT_RANDOM_STATE = 42\nMIN_CLASSES = 2  # distinct target classes in the support rows\nMIN_ROWS_PER_CLASS = 2  # rows per class for a stratified holdout split\nDECISION_RULE = "argmax"  # `predict` is the argmax over `predict_proba`; no threshold is shipped\nMETRIC_IDS = ("accuracy", "balanced_accuracy", "f1_weighted", "log_loss", "roc_auc")  # the ids `classification_metrics` reports\n\n\ndef runtime_identity() -> dict[str, str]:\n    import torch\n\n    return {\n        "pythonVersion": sys.version.split()[0],\n        "tabiclVersion": importlib.metadata.version("tabicl"),\n        "torchVersion": torch.__version__,\n        "device": "cuda" if torch.cuda.is_available() else "cpu",\n    }\n\n\ndef create_classifier(\n    *,\n    model_path: str | Path,\n    n_estimators: int,\n    random_state: int,\n    device: str,\n    allow_auto_download: bool = False,\n    support_many_classes: bool = True,\n):\n    """Construct the supported TabICLv2 classification serving estimator."""\n    from tabicl import TabICLClassifier\n\n    return TabICLClassifier(\n        model_path=str(model_path),\n        allow_auto_download=allow_auto_download,\n        n_estimators=n_estimators,\n        random_state=random_state,\n        device=device,\n        support_many_classes=support_many_classes,\n    )\n\n\ndef create_finetuned_classifier(**kwargs: Any):\n    """Construct the supported TabICLv2 classification fine-tuning estimator."""\n    from tabicl import FinetunedTabICLClassifier\n\n    kwargs.setdefault("allow_auto_download", False)\n    kwargs.setdefault("support_many_classes", True)\n    return FinetunedTabICLClassifier(**kwargs)\n\n\ndef fine_tune_classifier(model: Any, X: Any, y: Any, **kwargs: Any) -> Any:\n    """Run the upstream fine-tuning operation through the repository API."""\n    return model.fit(X, y, **kwargs)\n\n\ndef condition_classifier(model: Any, X: Any, y: Any) -> Any:\n    """Register the serving support context required by TabICL inference (no gradient step)."""\n    model.fit(X, y)\n    return model\n\n\ndef predict_labels(model: Any, X: Any) -> np.ndarray:\n    """Return one-dimensional class labels (the argmax of the class probabilities)."""\n    values = np.asarray(model.predict(X))\n    if values.ndim != 1:\n        values = values.reshape(-1)\n    return values\n\n\nMAX_PROBABILITY_ROW_SUM_DEVIATION = 1e-3  # TabICL\'s float32/fp16 ensemble average drifts from 1 by ~1e-7; more is a defect\n\n\ndef renormalise_probabilities(proba: Any) -> tuple[np.ndarray, float]:\n    """Return ``(rows divided by their sum, max |row sum - 1| before renormalising)``.\n\n    TabICL averages ensemble members in reduced precision, so rows can miss 1 by rounding; scikit-learn\'s metrics then\n    warn that ``y_prob`` values do not sum to one. A deviation above ``MAX_PROBABILITY_ROW_SUM_DEVIATION`` is refused\n    rather than hidden."""\n    values = np.asarray(proba, dtype=float)\n    sums = values.sum(axis=1, keepdims=True)\n    if values.size and not float(np.min(values)) >= 0.0:\n        raise RuntimeError(f"class probabilities contain a negative or undefined entry (min = {float(np.min(values)):.3g})")\n    deviation = float(np.max(np.abs(sums - 1.0))) if values.size else 0.0\n    if not deviation <= MAX_PROBABILITY_ROW_SUM_DEVIATION:\n        raise RuntimeError(f"class probabilities do not sum to one (max |row sum - 1| = {deviation:.3g})")\n    return values / sums, deviation\n\n\ndef predict_probabilities(model: Any, X: Any) -> np.ndarray:\n    """Return finite class probabilities, one column per entry of ``model.classes_`` in that order."""\n    values = np.asarray(model.predict_proba(X), dtype=float)\n    if values.ndim != 2 or values.shape[1] != len(model.classes_):\n        raise RuntimeError("TabICL produced probabilities of an unexpected shape")\n    if not np.isfinite(values).all():\n        raise RuntimeError("TabICL produced non-finite class probabilities")\n    return renormalise_probabilities(values)[0]\n\n\ndef align_probabilities(proba: Any, model_classes: Sequence[Any], classes: Sequence[Any]) -> np.ndarray:\n    """Reorder probability columns from ``model_classes`` order into ``classes`` order (strict: every class in\n    ``classes`` must be a column, so blending two models can never permute labels silently)."""\n    proba = np.asarray(proba, dtype=float)\n    lookup = {label: index for index, label in enumerate(model_classes)}\n    missing = [label for label in classes if label not in lookup]\n    if missing:\n        raise RuntimeError(f"model never saw classes: {missing}")\n    return proba[:, [lookup[label] for label in classes]]\n\n\ndef read_single_input(*, env_var: str, label: str) -> tuple[str, bytes]:\n    """Read one user-supplied file from an explicit path or Colab upload dialog."""\n    explicit = os.environ.get(env_var, "").strip()\n    if explicit:\n        path = Path(explicit).expanduser().resolve()\n        if not path.is_file():\n            raise FileNotFoundError(f"{label} path does not exist: {path}")\n        return path.name, path.read_bytes()\n\n    try:\n        from google.colab import files  # type: ignore\n    except ModuleNotFoundError as exc:\n        raise RuntimeError(\n            f"{label} requires either a Colab upload or environment variable {env_var}"\n        ) from exc\n\n    uploaded = files.upload()\n    if len(uploaded) != 1:\n        raise ValueError(f"Upload exactly one {label}")\n    name, payload = next(iter(uploaded.items()))\n    return str(name), bytes(payload)\n\n\ndef download_output(path: str | Path) -> Path:\n    """Download in Colab; otherwise retain the file at its explicit Jupyter path."""\n    resolved = Path(path).resolve()\n    try:\n        from google.colab import files  # type: ignore\n    except ModuleNotFoundError:\n        print(f"Output retained at {resolved}")\n        return resolved\n    files.download(str(resolved))\n    return resolved\n\n\ndef _base_version(value: str) -> str:\n    return value.split("+")[0]\n\n\ndef validate_artifact_runtime(\n    manifest: dict[str, Any],\n    *,\n    expected_artifact_format: str = ARTIFACT_FORMAT,\n    expected_tabicl_version: str = "2.1.1",\n    expected_torch_version: str = "2.14.0",\n) -> dict[str, Any]:\n    """Validate and return artifact/runtime provenance before model deserialization."""\n    if manifest.get("artifactFormat") != expected_artifact_format:\n        raise ValueError(f"Unsupported artifactFormat: {manifest.get(\'artifactFormat\')}")\n    if manifest.get("tabiclVersion") != expected_tabicl_version:\n        raise ValueError("Artifact TabICL version does not match the supported runtime")\n\n    observed = runtime_identity()\n    if _base_version(observed["torchVersion"]) != expected_torch_version:\n        raise RuntimeError(\n            f"Runtime torch {observed[\'torchVersion\']} is incompatible with the "\n            f"release-verified torch {expected_torch_version}"\n        )\n\n    producer = manifest.get("runtime")\n    compatibility = {\n        "artifactFormat": manifest.get("artifactFormat"),\n        "baseCheckpoint": manifest.get("baseCheckpoint"),\n        "baseModelRevision": manifest.get("baseModelRevision"),\n        "baseModelSha256": manifest.get("baseModelSha256"),\n        "tabiclVersion": manifest.get("tabiclVersion"),\n        "producerRuntime": producer,\n        "consumerRuntime": observed,\n        "policy": "TabICL exact; consumer torch base version 2.14.0; device may differ",\n    }\n\n    if producer:\n        producer_torch = str(producer.get("torchVersion", ""))\n        if producer_torch and _base_version(producer_torch) != expected_torch_version:\n            raise RuntimeError(\n                f"Artifact was produced with torch {producer_torch}; expected release family "\n                f"{expected_torch_version}"\n            )\n    else:\n        compatibility["warning"] = (\n            "Legacy artifact has no runtime block; TabICL/version and digest checks apply, "\n            "but producer runtime compatibility cannot be proven."\n        )\n    return compatibility\n\n\n# ---------------------------------------------------------------------------\n# Fleet snapshot scheme (NOTEBOOK_SPEC 1.1 ST3/ST4, MOD13): manifest-driven verification and staging.\n# ---------------------------------------------------------------------------\n\n\ndef sha256_file(path: str | Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n    """Check a local pinned snapshot against its manifest; raise naming the first mismatch.\n\n    The manifest is the parity anchor the standalone tutorials carry inline (ST3). The package\'s own\n    ``BASE_MODEL_SHA256`` is not replaced by it: the manifest entry for ``BASE_CHECKPOINT_NAME`` must equal\n    that constant, so the two can never diverge silently.\n    """\n    root = Path(path or DEFAULT_WEIGHTS_DIR)\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as handle:\n        manifest = json.load(handle)\n    if manifest.get("modelId") != MODEL_ID:\n        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n    if manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n    entries = manifest.get("files", [])\n    declared = {entry["path"]: entry["sha256"] for entry in entries}\n    if declared.get(BASE_CHECKPOINT_NAME) != BASE_MODEL_SHA256:\n        raise ValueError(\n            f"manifest {BASE_CHECKPOINT_NAME} sha256 {declared.get(BASE_CHECKPOINT_NAME)!r} "\n            f"!= BASE_MODEL_SHA256 {BASE_MODEL_SHA256!r}"\n        )\n    for entry in entries:\n        file_path = root / entry["path"]\n        if not file_path.is_file():\n            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n        size = file_path.stat().st_size\n        if size != entry["bytes"]:\n            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n        digest = sha256_file(file_path)\n        if digest != entry["sha256"]:\n            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n    return {"path": str(root), **manifest}\n\n\ndef _hub_download(relative_path: str, root: Path) -> None:\n    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n    from huggingface_hub import hf_hub_download\n\n    hf_hub_download(repo_id=MODEL_ID, filename=relative_path, revision=MODEL_REVISION, local_dir=str(root))\n\n\ndef stage_missing_files(\n    path: str | Path | None = None,\n    *,\n    allow_download: bool = False,\n    downloader: Callable[[str, Path], None] | None = None,\n) -> list[str]:\n    """Fetch manifest-listed files that are absent locally (a clone commits the manifest but\n    git-ignores the checkpoint). Returns the relative paths fetched; ``verify_snapshot`` still runs after."""\n    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n    manifest_path = root / MANIFEST_NAME\n    if not manifest_path.is_file():\n        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n    with open(manifest_path, encoding="utf-8") as handle:\n        manifest = json.load(handle)\n    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n        raise ValueError(\n            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n        )\n    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n    if not missing:\n        return []\n    if not allow_download:\n        raise FileNotFoundError(\n            f"snapshot at {root} is missing {missing}; "\n            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n        )\n    fetch = downloader or _hub_download\n    for relative_path in missing:\n        fetch(relative_path, root)\n    return missing\n\n\nclass TabICLClassificationPipeline:\n    """Serving wrapper: the digest-verified pinned checkpoint behind the upstream ``TabICLClassifier``.\n\n    ``from_pretrained`` stages and verifies the snapshot and constructs the estimator through\n    ``create_classifier`` (no auto-download); ``fit`` registers the support context (in-context learning,\n    no gradient training), ``predict_proba`` returns finite class probabilities in ``classes_`` order and\n    ``predict`` the ``argmax`` label (``DECISION_RULE``; no threshold is shipped).\n    """\n\n    def __init__(\n        self,\n        estimator: Any,\n        *,\n        model_path: Path,\n        n_estimators: int,\n        random_state: int,\n        device: str,\n        source: str = "local-snapshot",\n    ) -> None:\n        self.estimator = estimator\n        self.model_path = Path(model_path)\n        self.n_estimators = n_estimators\n        self.random_state = random_state\n        self.device = device\n        self.source = source\n        self.is_fitted = False\n\n    @classmethod\n    def from_pretrained(\n        cls,\n        weights_dir: str | Path | None = None,\n        *,\n        allow_download: bool = False,\n        n_estimators: int = DEFAULT_N_ESTIMATORS,\n        random_state: int = DEFAULT_RANDOM_STATE,\n        device: str | None = None,\n    ) -> TabICLClassificationPipeline:\n        """Stage what is missing (at ``MODEL_REVISION``), re-hash every manifest entry, then build the\n        estimator on the verified checkpoint. ``tabicl`` deserialises the checkpoint on the first ``fit``."""\n        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)\n        stage_missing_files(root, allow_download=allow_download)\n        verify_snapshot(root)\n        if device is None:\n            import torch\n\n            device = "cuda" if torch.cuda.is_available() else "cpu"\n        model_path = root / BASE_CHECKPOINT_NAME\n        estimator = create_classifier(\n            model_path=model_path,\n            n_estimators=n_estimators,\n            random_state=random_state,\n            device=device,\n            allow_auto_download=False,\n        )\n        return cls(\n            estimator,\n            model_path=model_path,\n            n_estimators=n_estimators,\n            random_state=random_state,\n            device=device,\n        )\n\n    @property\n    def classes_(self) -> list[Any]:\n        if not self.is_fitted:\n            raise RuntimeError("Pipeline is not conditioned; call fit(X, y) with the support rows first")\n        return list(self.estimator.classes_)\n\n    def fit(self, X: Any, y: Any) -> TabICLClassificationPipeline:\n        condition_classifier(self.estimator, X, y)\n        self.is_fitted = True\n        return self\n\n    def predict_proba(self, X: Any) -> np.ndarray:\n        if not self.is_fitted:\n            raise RuntimeError("Pipeline is not conditioned; call fit(X, y) with the support rows first")\n        return predict_probabilities(self.estimator, X)\n\n    def predict(self, X: Any) -> np.ndarray:\n        if not self.is_fitted:\n            raise RuntimeError("Pipeline is not conditioned; call fit(X, y) with the support rows first")\n        return predict_labels(self.estimator, X)\n\n\n# ---------------------------------------------------------------------------\n# Table preparation, encoding, metrics (extracted from the tutorials so both notebooks share one code path).\n# ---------------------------------------------------------------------------\n\n\ndef _check_classification_table(\n    frame: pd.DataFrame,\n    target_column: str,\n    *,\n    min_rows: int,\n    max_rows: int = MAX_TRAIN_ROWS,\n    max_features: int = MAX_FEATURES,\n    min_classes: int = MIN_CLASSES,\n) -> tuple[pd.DataFrame, int]:\n    """The checks `prepare_classification_table` applies; returns the cleaned table and the dropped-row count."""\n    if not isinstance(frame, pd.DataFrame):\n        raise TypeError("frame must be a pandas.DataFrame")\n    if frame.columns.duplicated().any():\n        dupes = sorted(set(frame.columns[frame.columns.duplicated()]))\n        raise ValueError(f"table contains duplicate column names: {dupes}")\n    if target_column not in frame.columns:\n        raise KeyError(f"missing target {target_column!r}")\n    labelled = frame[target_column].notna()\n    dropped = int((~labelled).sum())\n    out = frame.loc[labelled].copy().reset_index(drop=True)\n    if len(out) < min_rows:\n        raise ValueError(f"need at least {min_rows} labelled rows, got {len(out)}")\n    features = [column for column in out.columns if column != target_column]\n    if not features:\n        raise ValueError("No feature columns")\n    if len(features) > max_features or len(out) > max_rows:\n        raise ValueError(\n            f"Operational row/feature ceiling exceeded: rows={len(out)} (MAX_TRAIN_ROWS={max_rows}), "\n            f"features={len(features)} (MAX_FEATURES={max_features})"\n        )\n    if out[target_column].nunique() < min_classes:\n        raise ValueError(f"Need at least {min_classes} target classes, got {out[target_column].nunique()}")\n    return out, dropped\n\n\ndef prepare_classification_table(\n    frame: pd.DataFrame, target_column: str, *, min_rows: int = MIN_TRAIN_ROWS, min_classes: int = MIN_CLASSES\n) -> tuple[pd.DataFrame, int]:\n    """Drop rows whose target is missing (the count is returned so the notebook can report it — DAT23),\n    enforce the row/feature ceilings and the class-count floor."""\n    return _check_classification_table(frame, target_column, min_rows=min_rows, min_classes=min_classes)\n\n\ndef check_stratifiable(frame: pd.DataFrame, target_column: str, *, min_rows_per_class: int = MIN_ROWS_PER_CLASS) -> dict[Any, int]:\n    """Every class needs at least ``min_rows_per_class`` rows for a stratified split; returns the class counts."""\n    counts = frame[target_column].value_counts()\n    if counts.size < MIN_CLASSES or int(counts.min()) < min_rows_per_class:\n        raise ValueError(\n            f"Every class needs >= {min_rows_per_class} rows for a stratified holdout: {counts.to_dict()}"\n        )\n    return {label: int(n) for label, n in counts.items()}\n\n\ndef align_to_schema(\n    frame: pd.DataFrame,\n    feature_columns: Sequence[str],\n    target_column: str,\n    train_classes: Sequence[Any] | None = None,\n) -> pd.DataFrame:\n    """Require the training schema exactly and (when ``train_classes`` is given) no class unseen in training;\n    order columns like train."""\n    expected = set(feature_columns) | {target_column}\n    if set(frame.columns) != expected:\n        raise ValueError(\n            f"schema does not match train; missing={sorted(expected - set(frame.columns))}, "\n            f"extra={sorted(set(frame.columns) - expected)}"\n        )\n    if train_classes is not None:\n        unseen = sorted(set(frame[target_column]) - set(train_classes), key=str)\n        if unseen:\n            raise ValueError(f"target classes unseen in training: {unseen}")\n    return frame[[*feature_columns, target_column]].reset_index(drop=True)\n\n\ndef fit_categorical_encoder(frame: pd.DataFrame, feature_columns: Sequence[str]) -> dict[str, list[str]]:\n    """Ordinal maps for non-numeric columns, fitted on the training split only."""\n    encoders: dict[str, list[str]] = {}\n    for column in feature_columns:\n        series = frame[column]\n        if pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):\n            continue\n        encoders[column] = sorted({str(value) for value in series.dropna().unique()})\n    return encoders\n\n\ndef apply_categorical_encoder(\n    frame: pd.DataFrame, encoders: Mapping[str, Sequence[str]]\n) -> tuple[pd.DataFrame, dict[str, int]]:\n    """Apply training-fitted ordinal maps; unseen or missing values get the extra \'unknown\' code.\n\n    Returns the encoded frame and, per column, how many unseen values were mapped to the unknown code\n    (DAT20: the notebook reports them)."""\n    out = frame.copy()\n    unseen_counts: dict[str, int] = {}\n    for column, categories in encoders.items():\n        lookup = {category: index for index, category in enumerate(categories)}\n        unknown = len(categories)\n        encoded, unseen = [], 0\n        for value in out[column]:\n            if pd.isna(value):\n                encoded.append(unknown)\n                continue\n            key = str(value)\n            if key not in lookup:\n                unseen += 1\n            encoded.append(lookup.get(key, unknown))\n        out[column] = encoded\n        if unseen:\n            unseen_counts[column] = unseen\n    return out, unseen_counts\n\n\ndef _missing_counts(frame: pd.DataFrame) -> dict[str, int]:\n    return {str(column): int(n) for column, n in frame.isna().sum().items() if n > 0}\n\n\ndef _check_inference_frame(frame: pd.DataFrame, feature_columns: Sequence[str]) -> pd.DataFrame:\n    """The checks `read_inference_csv` applies to an inference table (after the raw-header check)."""\n    if frame.columns.duplicated().any():\n        dupes = sorted(set(frame.columns[frame.columns.duplicated()]))\n        raise ValueError(f"Inference CSV contains duplicate column names: {dupes}")\n    if "prediction" in frame.columns or any(str(column).startswith("probability_") for column in frame.columns):\n        raise ValueError("Inference CSV already contains prediction/probability output columns")\n    missing = [column for column in feature_columns if column not in frame.columns]\n    if missing:\n        raise ValueError(f"Inference CSV missing features: {missing}")\n    return frame\n\n\ndef raw_csv_header(payload: bytes) -> list[str]:\n    """First non-empty CSV row, read before pandas can rename duplicate names."""\n    reader = csv.reader(io.StringIO(payload.decode("utf-8-sig")))\n    for row in reader:\n        if row and any(cell.strip() for cell in row):\n            return row\n    raise ValueError("CSV has no header")\n\n\ndef read_csv_payload(payload: bytes, label: str) -> pd.DataFrame:\n    """Read a CSV payload, refusing duplicate header names before pandas renames them (DAT16)."""\n    header = raw_csv_header(payload)\n    dupes = sorted({name for name in header if header.count(name) > 1})\n    if dupes:\n        raise ValueError(f"{label} contains duplicate column names: {dupes}")\n    return pd.read_csv(io.BytesIO(payload))\n\n\ndef read_inference_csv(payload: bytes, feature_columns: Sequence[str]) -> pd.DataFrame:\n    """Read an unlabelled inference CSV: unique header, no `prediction`/`probability_*` columns, every feature present."""\n    return _check_inference_frame(read_csv_payload(payload, "Inference CSV"), feature_columns)\n\n\ndef classification_metrics(y_true: Any, y_pred: Any, y_proba: Any, classes: Sequence[Any]) -> dict[str, float]:\n    """The repository\'s metric set: accuracy, balanced_accuracy, f1_weighted (discrete correctness under the\n    argmax rule), log_loss (probability quality; lower is better) and roc_auc (ranking quality; binary or\n    one-vs-rest; NaN when undefined, e.g. a single class present)."""\n    from sklearn.metrics import (\n        accuracy_score,\n        balanced_accuracy_score,\n        f1_score,\n        log_loss,\n        roc_auc_score,\n    )\n\n    y = np.asarray(y_true).reshape(-1)\n    pred = np.asarray(y_pred).reshape(-1)\n    proba = np.asarray(y_proba, dtype=float)\n    labels = list(classes)\n    if y.shape != pred.shape or y.size == 0:\n        raise ValueError("y_true and y_pred must be non-empty and the same length")\n    if proba.ndim != 2 or proba.shape != (y.size, len(labels)):\n        raise ValueError("y_proba must have one row per example and one column per class")\n    proba = renormalise_probabilities(proba)[0]  # rows that miss 1 by rounding only (R96-m10)\n    out = {\n        "accuracy": float(accuracy_score(y, pred)),\n        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),\n        "f1_weighted": float(f1_score(y, pred, average="weighted")),\n        "log_loss": float(log_loss(y, proba, labels=labels)),\n    }\n    try:\n        if len(labels) == 2:\n            out["roc_auc"] = float(roc_auc_score(y, proba[:, 1]))\n        else:\n            out["roc_auc"] = float(roc_auc_score(y, proba, multi_class="ovr", labels=labels))\n    except ValueError:\n        out["roc_auc"] = float("nan")\n    return out\n\n\ndef majority_class_baseline(train_targets: Any, holdout_targets: Any) -> dict[str, float]:\n    """The trivial baseline: always predict the most frequent support class, with the support class shares as\n    the probabilities (so log_loss is defined; roc_auc is NaN because constant scores rank nothing)."""\n    train = pd.Series(np.asarray(train_targets).reshape(-1))\n    holdout = np.asarray(holdout_targets).reshape(-1)\n    if train.size == 0 or holdout.size == 0:\n        raise ValueError("targets must be non-empty")\n    shares = train.value_counts(normalize=True).sort_index()\n    classes = list(shares.index)\n    unseen = sorted(set(holdout) - set(classes), key=str)\n    if unseen:\n        raise ValueError(f"holdout has target classes unseen in training: {unseen}")\n    majority = shares.idxmax()\n    proba = np.tile(shares.to_numpy(dtype=float), (holdout.size, 1))\n    metrics = classification_metrics(holdout, np.full(holdout.shape, majority, dtype=object), proba, classes)\n    metrics["roc_auc"] = float("nan")\n    return metrics\n\n\ndef compare_metric(candidate: Mapping[str, float], reference: Mapping[str, float], name: str) -> bool:\n    """True when `candidate` beats `reference` on metric `name` (lower log_loss, higher everything else)."""\n    candidate_value, reference_value = float(candidate[name]), float(reference[name])\n    if not (math.isfinite(candidate_value) and math.isfinite(reference_value)):\n        raise ValueError(f"{name} unavailable for selection")\n    if name == "log_loss":\n        return candidate_value < reference_value\n    return candidate_value > reference_value\n\n\n# ---------------------------------------------------------------------------\n# Role stages (DAT24 / EVAL21).\n# ---------------------------------------------------------------------------\n\nINPUT_SCHEMA: dict[str, Any] = {\n    "input": "pandas.DataFrame, one row per example; numeric/categorical features plus a categorical target",\n    "columns": "unique names; non-numeric feature columns are ordinal-encoded with maps fitted on training",\n    "target": (\n        "any label type; rows whose target is missing are dropped and counted; at least MIN_CLASSES distinct "\n        "classes, and every holdout/test class must be present in the support rows"\n    ),\n    "train_rows": [MIN_TRAIN_ROWS, MAX_TRAIN_ROWS],\n    "eval_rows": [MIN_EVAL_ROWS, None],\n    "features": [1, MAX_FEATURES],\n    "classes": [MIN_CLASSES, None],\n    "rows_per_class_for_stratified_split": [MIN_ROWS_PER_CLASS, None],\n    "inference_input": (\n        "every fitted feature column present; no `prediction` or `probability_*` column; extras pass through"\n    ),\n    "decision_rule": DECISION_RULE,\n    "preprocessing": (\n        "no scaling by the package; unseen or missing categorical values map to the fitted \'unknown\' code; "\n        "the model is conditioned on the (encoded) training rows at prediction time"\n    ),\n}\n\n\ndef validate_inputs(\n    frame: pd.DataFrame,\n    target_column: str | None = "target",\n    *,\n    feature_columns: Sequence[str] | None = None,\n    min_rows: int = MIN_TRAIN_ROWS,\n    names: Sequence[str] | None = None,\n) -> dict[str, Any]:\n    """Validation stage: return the input manifest (schema, observed table properties, verdict).\n\n    With a ``target_column`` the table is checked exactly as ``prepare_classification_table`` checks it (the\n    number of dropped missing-target rows is recorded, not hidden; class counts are observed); with\n    ``target_column=None`` it is an inference table checked against ``feature_columns`` exactly as\n    ``read_inference_csv`` checks it. Rejection is reported by raising the same error the core function raises.\n    """\n    if names is not None and len(names) != 1:\n        raise ValueError("names must have exactly one entry (the table\'s id)")\n    table_id = names[0] if names else "table-0"\n    if target_column is None:\n        if feature_columns is None:\n            raise ValueError("feature_columns is required to validate an inference table")\n        checked = _check_inference_frame(frame, list(feature_columns))\n        entry: dict[str, Any] = {\n            "id": table_id,\n            "mode": "inference",\n            "rows": len(checked),\n            "feature_columns": list(feature_columns),\n            "extra_columns": [column for column in checked.columns if column not in feature_columns],\n            "missing_value_columns": _missing_counts(checked[list(feature_columns)]),\n        }\n    else:\n        cleaned, dropped = _check_classification_table(frame, target_column, min_rows=min_rows)\n        features = [column for column in cleaned.columns if column != target_column]\n        encoders = fit_categorical_encoder(cleaned, features)\n        counts = cleaned[target_column].value_counts()\n        entry = {\n            "id": table_id,\n            "mode": "fit",\n            "rows": len(cleaned),\n            "dropped_missing_target_rows": dropped,\n            "feature_columns": features,\n            "categorical_columns": sorted(encoders),\n            "missing_value_columns": _missing_counts(cleaned[features]),\n            "classes": [str(label) for label in counts.sort_index().index],\n            "class_counts": {str(label): int(n) for label, n in counts.sort_index().items()},\n            "majority_class_share": float(counts.max() / counts.sum()),\n        }\n    return {\n        "schema": dict(INPUT_SCHEMA),\n        "inputs": [entry],\n        "target_column": target_column,\n        "min_rows": min_rows if target_column is not None else None,\n        "verdict": "accepted",\n        "findings": [],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n\n\ndef evaluation_report(\n    metrics: Mapping[str, float] | None,\n    *,\n    baseline: Mapping[str, float] | None = None,\n    independent_test: Mapping[str, float] | None = None,\n    n_holdout: int | None = None,\n    n_test: int | None = None,\n    class_labels: Sequence[Any] | None = None,\n    target_column: str | None = None,\n    selection: str | None = None,\n    sample_kind: str = "sample",\n    estimation: str = "single seeded stratified holdout; no dispersion estimate",\n) -> dict[str, Any]:\n    """Evaluation stage: a machine-readable report even when nothing is measurable.\n\n    ``metrics`` / ``independent_test`` are dicts from ``classification_metrics`` and ``baseline`` from\n    ``majority_class_baseline``; the verdict is ``sample-sanity``. Without metrics (no labelled rows) the\n    verdict is ``not-measurable`` and the report says what labelled data would make the task measurable.\n    """\n\n    def _entries(source: Mapping[str, float]) -> list[dict[str, Any]]:\n        unknown = sorted(set(source) - set(METRIC_IDS))\n        if unknown:\n            raise ValueError(f"unknown metric ids {unknown}; classification_metrics reports {list(METRIC_IDS)}")\n        return [\n            {\n                "id": metric_id,\n                "value": None if not math.isfinite(float(source[metric_id])) else float(source[metric_id]),\n                "units": "nats" if metric_id == "log_loss" else "unitless",\n                "higher_is_better": metric_id != "log_loss",\n            }\n            for metric_id in METRIC_IDS\n            if metric_id in source\n        ]\n\n    base: dict[str, Any] = {\n        "task": "tabular classification by in-context conditioning on labelled support rows",\n        "decision_rule": DECISION_RULE,\n        "score_semantics": "class probabilities in classes_ order, uncalibrated; no threshold shipped",\n        "sample_kind": sample_kind,\n        "n_holdout": n_holdout,\n        "n_test": n_test,\n        "class_labels": None if class_labels is None else [str(label) for label in class_labels],\n        "target_column": target_column,\n        "selection": selection,\n        "baselines": [],\n        "model_id": MODEL_ID,\n        "model_revision": MODEL_REVISION,\n    }\n    if metrics is None:\n        return {\n            **base,\n            "metrics": [],\n            "independent_test": [],\n            "verdict": "not-measurable",\n            "reason": "no labelled holdout rows were supplied for the scored table",\n            "needs": (\n                "a labelled holdout table whose classes all appear in the support rows, scored with "\n                "`classification_metrics` (accuracy, balanced_accuracy, f1_weighted, log_loss, roc_auc) against "\n                "`majority_class_baseline`; an independent test partition from the deployment domain for any "\n                "generalisable claim"\n            ),\n        }\n    reported = [{**entry, "estimation": estimation} for entry in _entries(metrics)]\n    test_entries: list[dict[str, Any]] = []\n    if independent_test is not None:\n        test_estimation = "independent test partition, single run"\n        test_entries = [{**e, "estimation": test_estimation} for e in _entries(independent_test)]\n    baselines = [] if baseline is None else [{"id": "majority_class", "metrics": _entries(baseline)}]\n    rows = "an unstated number of" if n_holdout is None else str(n_holdout)\n    return {\n        **base,\n        "metrics": reported,\n        "independent_test": test_entries,\n        "baselines": baselines,\n        "verdict": "sample-sanity",\n        "reason": f"{rows} labelled holdout row(s) from one seeded stratified split; tutorial evidence, not a benchmark",\n        "needs": (\n            "an independent, domain-representative labelled test set for any generalisable quality claim; the "\n            "class probabilities are uncalibrated and any decision threshold must be chosen on the caller\'s data"\n        ),\n    }\n\n\n# ---------------------------------------------------------------------------\n# Serving-artifact ZIP handling shared by the producer\'s fresh-reload check and the closing external-artifact section.\n# ---------------------------------------------------------------------------\n\n\ndef safe_extract_zip(\n    zip_path: str | Path, dest: str | Path, *, max_expanded_bytes: int = MAX_ARTIFACT_EXPANDED_BYTES\n) -> Path:\n    """Extract a ZIP member by member after every member passed the path, symlink and size checks (AINF3).\n\n    Members are copied individually (never ``extractall``) so a member that fails a check is never written.\n    """\n    dest = Path(dest)\n    dest.mkdir(parents=True, exist_ok=True)\n    root = dest.resolve()\n    with zipfile.ZipFile(zip_path) as archive:\n        expanded_bytes = 0\n        for info in archive.infolist():\n            if "\\\\" in info.filename:\n                raise ValueError(f"Ambiguous backslash ZIP member: {info.filename}")\n            expanded_bytes += info.file_size\n            if expanded_bytes > max_expanded_bytes:\n                raise ValueError(f"Artifact exceeds {max_expanded_bytes} expanded bytes")\n            name = info.filename\n            parts = Path(name).parts\n            mode = info.external_attr >> 16\n            if name.startswith("/") or ".." in parts or stat.S_ISLNK(mode):\n                raise ValueError(f"Unsafe ZIP member: {info.filename}")\n            target = (dest / Path(name)).resolve()\n            if root != target and root not in target.parents:\n                raise ValueError("ZIP member escapes destination")\n        for info in archive.infolist():\n            target = (dest / Path(info.filename)).resolve()\n            if info.is_dir():\n                target.mkdir(parents=True, exist_ok=True)\n                continue\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(info) as source, target.open("wb") as destination:\n                shutil.copyfileobj(source, destination)\n    return root\n\n\ndef manifest_member_path(root: str | Path, value: Any, field: str) -> Path:\n    """Resolve a manifest path inside the bundle root, refusing absolute paths and traversal."""\n    rel = Path(str(value))\n    if rel.is_absolute() or ".." in rel.parts:\n        raise ValueError(f"Unsafe {field} path in artifact.json: {value!r}")\n    root_resolved = Path(root).resolve()\n    target = (root_resolved / rel).resolve()\n    if root_resolved != target and root_resolved not in target.parents:\n        raise ValueError(f"{field} path escapes artifact root: {value!r}")\n    return target\n\n\ndef verify_artifact_bundle(root: str | Path, manifest: Mapping[str, Any]) -> dict[str, Path]:\n    """Check an extracted bundle\'s allowlist, sizes and digests against its manifest; return member paths."""\n    root = Path(root)\n    ckpt = manifest_member_path(root, manifest["checkpoint"], "checkpoint")\n    context_path = manifest_member_path(root, manifest["trainingContext"], "trainingContext")\n    payload_files = manifest.get("payloadFiles")\n    if payload_files is not None:\n        expected_files = {"artifact.json", *payload_files}\n        actual_files = {p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()}\n        if actual_files != expected_files:\n            unexpected = sorted(actual_files ^ expected_files)\n            raise RuntimeError(f"Unexpected or missing artifact files: {unexpected}")\n    if manifest.get("sizes"):\n        if ckpt.stat().st_size != manifest["sizes"]["checkpoint"]:\n            raise RuntimeError("Checkpoint size mismatch")\n        if context_path.stat().st_size != manifest["sizes"]["trainingContext"]:\n            raise RuntimeError("Training-context size mismatch")\n    if sha256_file(ckpt) != manifest["digests"]["checkpointSha256"]:\n        raise RuntimeError("Checkpoint digest mismatch")\n    if sha256_file(context_path) != manifest["digests"]["trainingContextSha256"]:\n        raise RuntimeError("Training-context digest mismatch")\n    return {"checkpoint": ckpt, "training_context": context_path}\n', 'tutorial_stages.py': '"""Stage runner for the standalone TabICLv2 classifier E2E tutorial (NOTEBOOK_SPEC 2.5 §25.13 isolated-environment pattern).\n\nThe tutorial notebook carries this file verbatim (as ``tutorial_stages.py`` in its run directory, beside the carried\npackage under ``src/``) and runs every stage with the interpreter of an isolated, hash-locked environment::\n\n    python -u tutorial_stages.py --root RUN_DIR --outputs OUTPUTS --weights WEIGHTS --stage data --options \'{...}\'\n\nNothing is installed into the notebook kernel. Each stage is a separate process and starts from files only: the\nverified checkpoint under ``--weights``, the partitions, encoders and records of earlier stages under\n``RUN_DIR/state``, and the learner-facing exports under ``--outputs``. In-context conditioning (``fit``) registers the\nsupport rows and is repeated by every stage that needs the model; a fine-tuned candidate is kept as a checkpoint file.\nOn failure a stage writes ``RUN_DIR/state/<stage>.error.json``, which the notebook re-raises in the kernel.\n\nStages: weights → data → validate → condition → report → fresh_test → predict → export → reload, then the closing\nexternal-artifact section (NOTEBOOK_SPEC 3.0 §19.1) external_artifact → external_reconstruct → external_predict, plus the\noptional ``activity``. The external-artifact stages load the pinned sample artifact (a release asset produced by a recorded\nearlier run of this notebook, pinned by URL and SHA-256), never this run\'s export, and read none of the main path\'s state.\nThe fresh test of the default MAGIC sample is read only by ``fresh_test``, only under the pre-registered settings (docs/magic-finetune-preregistration.md), and never when ``DIMER_MASK_FRESH_TEST=1``.\nThe ``data`` and ``validate`` stages import no model library, so CI exercises them directly.\n"""\n# ruff: noqa: E501  -- the printed dictionaries are the learner-facing output; they are kept on one line each\nfrom __future__ import annotations\n\nimport argparse\nimport contextlib\nimport hashlib\nimport importlib\nimport io\nimport json\nimport math\nimport os\nimport random\nimport re\nimport shutil\nimport sys\nimport traceback\nfrom pathlib import Path\nfrom typing import Any\n\nSTEM = "tabiclv2_classifier"\nPACKAGE = "tabicl_classifier_pipeline"\nSEED = 42\nVALIDATION_SPLIT = 0.20\nNEW_ROWS = 8\nRELOAD_RTOL, RELOAD_ATOL = 1e-5, 1e-7\nMAGIC_SOURCE = "Sample: MAGIC Gamma Telescope"\nREFERENCE_SOURCE = "Sample: Breast Cancer"  # the earlier default; its 114-row test is an exposed development test\nSAMPLE_SOURCES = (MAGIC_SOURCE, REFERENCE_SOURCE, "Sample: Wine", "Sample: Palmer Penguins")\nDEFAULT_SOURCE = MAGIC_SOURCE\nUPLOAD_SOURCES = ("Upload CSV", "Upload pre-split train/val/test")\nPENGUINS_URL = "https://raw.githubusercontent.com/kurtvalcorza/tabicl-classifier-pipeline/169e60fa8d956aa389c144ac9c7988b92db84a79/examples/sample-data/palmer-penguins.zip"\nPENGUINS_SHA256 = "fe894295ccc0a447dc020f5d57798968b95eae3da1a4cbe09a638e6b1bd0ba44"\nNUMERIC_LIKE_SHARE = 0.9  # an object column whose values are >= 90 % numbers is treated as a numeric column with typos\nMIN_SELECTION_HOLDOUT_ROWS = 50\n\n# --- MAGIC Gamma Telescope (UCI 159, CC BY 4.0): pinned split archive, committed at an immutable revision (R96-D1) ---\nMAGIC_COMMIT = "fc2a5adb68981e01bb00cae3ea5ea2f377412383"\nMAGIC_URL = f"https://raw.githubusercontent.com/kurtvalcorza/tabicl-classifier-pipeline/{MAGIC_COMMIT}/examples/sample-data/magic-gamma.zip"\nMAGIC_SHA256 = "2c6108e6d3e53cd9fcc8417df6cc8018ea2faa70a3e9b8e58fd3ef4e1463497d"\nMAGIC_SOURCE_SHA256 = "a52d3a378e43df7b678c4c0b2f03a66a058bbfeb27c95ccd037a767e7d172984"\nMAGIC_MEMBERS = {\n    "train.csv": "0a5f1692662ad23807339a91ebfeef26cc22871ad5df535bd284aea70d16a246",\n    "val.csv": "36bc23405d7a5bee814c21297140ffd70436c147c236cbaf2881613a1ebdb0b4",\n    "test.csv": "62c3eff6bb323ac150ec60eb4f26d5ed92ce20fe79ed3baa83f32b76e4482b51",\n}\nMAGIC_ROW_IDS_SHA256 = {\n    "train": "009082b81a485027433be0fcb0de99855a480b425ebbb9e70bd37242307210cd",\n    "val": "22dbeb1022a76e2cab51dd7a2df9869efa71430867d975f21ac59369e84d17b3",\n    "test": "8847cab4e87c8ff30dd6a27045464f7891b2e14dd39149c5e5648348f190f3e4",\n    "reserve": "c1b30483d5024fa6f94893a824dcbacf60ca942b941faa790edbb541c97208b3",\n}\nMAGIC_FRESH_TEST_ROWS = 1500\n\n# --- External-artifact section (NOTEBOOK_SPEC 3.0 §19.1, XAS1–XAS7, SART1–SART11) ---------------------------------------\n# The default artifact of the closing section is a release asset of this repository: the deterministic export\n# (`outputs/tabiclv2_classifier_artifact.zip`) of a recorded earlier hosted run of this notebook\'s default path (SART8,\n# SART9). It is pinned by URL and SHA-256 and never replaced; a changed artifact gets a new tag and a new pin (SART7,\n# SART11). The export of the current run is never used as the external artifact (XAS2).\nSAMPLE_ARTIFACT_TAG = "sample-artifact-magic-v1"\nSAMPLE_ARTIFACT_URL = "https://github.com/kurtvalcorza/tabicl-classifier-pipeline/releases/download/sample-artifact-magic-v1/tabiclv2_classifier_artifact.zip"  # literal: validators allow only this exact URL\nSAMPLE_ARTIFACT_SHA256 = "2dc82d55161382602fa7a5d1e217e3a33b6c6188ba6ddf4f7ff6b087bf67114c"\nSAMPLE_ARTIFACT_PRODUCER = {\n    "notebook": "tutorials/tabiclv2_classifier_colab.ipynb (default path: MAGIC Gamma Telescope sample)",\n    "commit": "383ca046184b2a3aead0c9dd94503338bfbd1253",\n    "run": "recorded hosted Colab T4 run; the release notes of the tag name the run record (SART8)",\n    "release_tag": SAMPLE_ARTIFACT_TAG,\n}\nSAMPLE_ARTIFACT_MAX_BYTES = 512 * 1024 * 1024  # a download larger than this is refused before it is hashed or opened\nEXTERNAL_DIR = "external_artifact"  # under outputs/: the section\'s predictions and provenance (XAS6)\nEXTERNAL_SAMPLE_ROWS = 8\nSHA256_HEX = re.compile(r"^[0-9a-f]{64}$")\n\n# --- pre-registered fine-tuning grid, selection and reading rule (docs/magic-finetune-preregistration.md) ---\nGRID_COMMON = {"weight_decay": 0.01, "grad_clip": 1.0, "max_data_size": 500, "use_lr_scheduler": False, "finetune_ctx_query_ratio": 0.2, "early_stopping": True, "eval_metric": "log_loss", "min_delta": 1e-4, "n_estimators_inference": 8}\nGRID = (\n    {"id": "C1", "role": "production-like", "learning_rate": 1e-5, "epochs": 30, "patience": 8, "n_estimators_finetune": 2, "n_estimators_validation": 2, "time_limit": 1800},\n    {"id": "C2", "role": "larger steps", "learning_rate": 5e-5, "epochs": 10, "patience": 3, "n_estimators_finetune": 2, "n_estimators_validation": 2, "time_limit": 600},\n    {"id": "C3", "role": "smaller steps", "learning_rate": 1e-6, "epochs": 10, "patience": 3, "n_estimators_finetune": 2, "n_estimators_validation": 2, "time_limit": 600},\n)\nSELECTION_TOLERANCE = 0.005  # validation log loss a config must beat the pretrained model by; otherwise keep pretrained\nSELECTION_TIE = 1e-4\nINFERENCE_ESTIMATORS = 8\nBOOTSTRAP = {"seed": 20261019, "n_boot": 10000}\nPRACTICAL_DELTA = 0.005  # "improved" needs the CI upper bound < 0 and delta <= -PRACTICAL_DELTA\nDETERMINISM = {"CUBLAS_WORKSPACE_CONFIG": ":4096:8", "cudnn_deterministic": True, "cudnn_benchmark": False, "deterministic_algorithms": True, "warn_only": False, "seed": SEED, "sdpa_backend": "MATH", "flash_attn3": False}\n# Amendment 2: every stage that runs the model does so inside deterministic_kernels() (MATH attention, FlashAttention-3 off).\nMODEL_STAGES = frozenset(("condition", "report", "fresh_test", "predict", "export", "reload", "activity", "external_reconstruct", "external_predict"))\nDETERMINISM_WARNING = re.compile(r"determinis|non-?deterministic|memory.efficient|mem_efficient|efficient attention", re.IGNORECASE)\n_CAUGHT_WARNINGS: list = []  # the warnings recorded by the active deterministic_kernels() context\nREQUIRED_DEVICE = "Tesla T4"\nMASK_ENV = "DIMER_MASK_FRESH_TEST"\n# The evaluation of record: the first hosted T4 run under the pre-registered settings (amendment 2), recorded in\n# docs/release-verification.md with its evidence. Because a record exists, every pre-registered scoring of the fresh test, the\n# first in a new runtime included, is a "reproduction" and prints matches_record / differs against the printed values below.\n# The record kept the pretrained model, which is also what a CPU run (no grid) exports.\n# Issue #28: on a Tesla T4 the fp16 fine-tuning grid hash varies run to run even at an identical revision. Two hosted runs of the\n# same notebook blob (f21920ab) with the same carried stage runner (ee759496...), at commits d4172f5 and edbe94d, printed\n# 2aa9e0cd... and 4b1bf536...; their selection and every fresh-test value matched the record exactly. So grid_hash is reported\n# (record, this run, equal or not) as information only and never decides matches_record. The record still names the SHA-256 of\n# the stage runner that produced it, which is reported the same way.\nRECORDED_SELECTION: str = "pretrained"\nRECORDED_EVALUATION: dict[str, Any] | None = {\n    "label": "pre-registered evaluation (this runtime)",\n    "date": "2026-10-20",\n    "runner": "Colab CLI sequential execution, fresh hosted Tesla T4 session",\n    "commit": "9ea0888d9df24a9568613ec9c51dd1fe65cea443",\n    "notebook_blob": "947feb162abbf8126a8579188268024da3ca3ba8",\n    "evidence": "docs/execution-evidence/2026-10-20-9ea0888/",\n    "device": "Tesla T4",\n    "selected": RECORDED_SELECTION,\n    "grid_hash": "4b1bf5366b121dd2365c6c9b02408e1b825c247bb6636fee6d8fe35697b20692",\n    # SHA-256 of the carried tutorial_stages.py (UTF-8, LF) that produced the record: NOTEBOOK_SOURCE[\'files\'][\'tutorial_stages.py\']\n    # in blob 947feb16, equal to tools/tutorial_stages.py at 15c1ae4 (unchanged at 9ea0888).\n    "stage_runner_sha256": "cc604098aa235653c2ef5b36cffbca37332539a9ac74e1b52912e1ff71184823",\n    "reading": {\n        "primary.delta": 0.0,\n        "primary.ci95": [0.0, 0.0],\n        "primary.verdict": "same model (Δ = 0)",\n        "accuracy.pretrained": 0.8993,\n        "accuracy.pretrained_wilson95": [0.8831, 0.9136],\n        "accuracy.selected": 0.8993,\n        "accuracy.difference": 0.0,\n        "roc_auc.pretrained": 0.95,\n        "roc_auc.selected": 0.95,\n        "sensitivity.excluded_rows": 4,\n        "sensitivity.n": 1496,\n        "sensitivity.primary.delta": 0.0,\n        "sensitivity.primary.verdict": "same model (Δ = 0)",\n    },\n}\n\n\ndef printed_reading(reading: dict[str, Any], sensitivity: dict[str, Any]) -> dict[str, Any]:\n    """The fresh-test values in the form the record stores them (rounded as printed)."""\n    r = rounded({"reading": reading, "sensitivity": sensitivity})\n    rd, sv = r["reading"], r["sensitivity"]\n    return {\n        "primary.delta": rd["primary"]["delta"],\n        "primary.ci95": list(rd["primary"]["ci95"]),\n        "primary.verdict": rd["primary"]["verdict"],\n        "accuracy.pretrained": rd["accuracy"]["pretrained"],\n        "accuracy.pretrained_wilson95": list(rd["accuracy"]["pretrained_wilson95"]),\n        "accuracy.selected": rd["accuracy"]["selected"],\n        "accuracy.difference": rd["accuracy"]["difference"],\n        "roc_auc.pretrained": rd["roc_auc"]["pretrained"],\n        "roc_auc.selected": rd["roc_auc"]["selected"],\n        "sensitivity.excluded_rows": sv["excluded_rows"],\n        "sensitivity.n": sv["n"],\n        "sensitivity.primary.delta": sv["primary"]["delta"],\n        "sensitivity.primary.verdict": sv["primary"]["verdict"],\n    }\n\n\ndef stage_runner_sha256() -> str:\n    """SHA-256 of this stage runner as the notebook stamps it (UTF-8, LF): NOTEBOOK_SOURCE[\'files\'][\'tutorial_stages.py\']."""\n    return hashlib.sha256(Path(__file__).read_text(encoding="utf-8").replace("\\r\\n", "\\n").encode("utf-8")).hexdigest()\n\n\nGRID_HASH_NOTE = "informational only, not compared: on a T4 the fp16 grid hash varies run to run at an identical revision (issue #28)"\n\n\ndef compare_with_record(selected: str, grid_hash_value: str, device: str | None, values: dict[str, Any], runner_sha256: str | None = None) -> dict[str, Any] | None:\n    """A reproduction against the evaluation of record: the fields that differ, by name. None when no record exists.\n\n    The selection, the device and every fresh-test value must match the record exactly. The grid hash is reported, with the\n    record\'s value, this run\'s value and whether they are equal, but is informational and never a difference, whatever the stage\n    runner (issue #28: on a T4 the fp16 grid hash has varied run to run at an identical revision). The stage-runner digests are\n    reported the same way."""\n    if RECORDED_EVALUATION is None:\n        return None\n    current = runner_sha256 if runner_sha256 is not None else stage_runner_sha256()\n    recorded_runner = RECORDED_EVALUATION.get("stage_runner_sha256")\n    differs = [k for k, v in (("selected", selected), ("device", device)) if RECORDED_EVALUATION.get(k) != v]\n    recorded = RECORDED_EVALUATION.get("reading") or {}\n    differs += [f"reading.{k}" for k in recorded if values.get(k) != recorded[k]]\n    recorded_grid = RECORDED_EVALUATION.get("grid_hash")\n    return {\n        "reproduction_of": f"{RECORDED_EVALUATION.get(\'date\')} hosted T4, commit {str(RECORDED_EVALUATION.get(\'commit\'))[:7]}",\n        "stage_runner_sha256": {"record": recorded_runner, "this_run": current},\n        "same_stage_runner": recorded_runner == current,\n        "grid_hash": {"informational": GRID_HASH_NOTE, "record": recorded_grid, "this_run": grid_hash_value, "equal": recorded_grid == grid_hash_value},\n        "matches_record": not differs,\n        "differs": differs,\n    }\nTEST_LOOKS = ("pre-registered evaluation (this runtime)", "reproduction", "not scored (exploratory)")\nFRESH_DIR = "fresh-test"  # under outputs/; not removed by clear_outputs, so a re-run in the same runtime is a reproduction\n\n\ndef fresh_test_masked() -> bool:\n    """Re-read on every call (a mask copied once at install fails open)."""\n    return os.environ.get(MASK_ENV, "").strip().lower() in {"1", "true", "yes"}\n\n\ndef atomic_write_bytes(path: Path, data: bytes) -> Path:\n    """Write ``path`` via a temporary file and ``os.replace`` so an interrupted write never leaves a truncated file."""\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".tmp")\n    tmp.write_bytes(data)\n    os.replace(tmp, path)\n    return path\n\n\ndef atomic_write_json(path: Path, value: Any) -> Path:\n    return atomic_write_bytes(path, json.dumps(value, indent=2, ensure_ascii=False, default=str).encode("utf-8"))\n\n\ndef apply_determinism() -> dict[str, Any]:\n    """Pre-registered determinism flags. ``CUBLAS_WORKSPACE_CONFIG`` is set by ``main`` before torch is imported. An op\n    with no deterministic CUDA kernel raises (no ``warn_only``) and stops the stage, naming the op."""\n    import numpy as np\n    import torch\n\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n    torch.use_deterministic_algorithms(True)\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    return {"CUBLAS_WORKSPACE_CONFIG": os.environ.get("CUBLAS_WORKSPACE_CONFIG"), "cudnn_deterministic": bool(torch.backends.cudnn.deterministic), "cudnn_benchmark": bool(torch.backends.cudnn.benchmark), "deterministic_algorithms": bool(torch.are_deterministic_algorithms_enabled()), "warn_only": bool(torch.is_deterministic_algorithms_warn_only_enabled()), "seed": SEED, **attention_backend()}\n\n\ndef attention_backend() -> dict[str, Any]:\n    """The attention path as PyTorch and tabicl report it now (observed, not assumed): ``sdpa_backend`` is "MATH" only when\n    the math kernel is the sole enabled SDPA backend; ``flash_attn3`` is tabicl\'s own FlashAttention-3 switch."""\n    import torch\n\n    cuda = torch.backends.cuda\n    enabled = {"FLASH": cuda.flash_sdp_enabled(), "MEM_EFFICIENT": cuda.mem_efficient_sdp_enabled(), "MATH": cuda.math_sdp_enabled(), "CUDNN": bool(getattr(cuda, "cudnn_sdp_enabled", lambda: False)())}\n    names = sorted(name for name, on in enabled.items() if on)\n    try:\n        tabicl_attention = importlib.import_module("tabicl._model.attention")\n        flash3 = bool(getattr(tabicl_attention, "HAS_FLASH_ATTN3", False) and getattr(tabicl_attention, "_use_flash_attn3", False))\n    except ImportError:\n        flash3 = None\n    return {"sdpa_backend": "+".join(names) or "NONE", "flash_attn3": flash3}\n\n\ndef determinism_warnings() -> list[str]:\n    """Determinism-related warnings captured so far in this stage (e.g. a kernel that only warns)."""\n    return sorted({str(w.message) for w in _CAUGHT_WARNINGS if DETERMINISM_WARNING.search(str(w.message))})\n\n\n@contextlib.contextmanager\ndef deterministic_kernels():\n    """Amendment 2: the MATH SDPA backend and tabicl\'s FlashAttention-3 path off around a whole model stage (fine-tuning\n    and every inference), the pre-registered determinism flags, and every Python warning captured; determinism-related\n    warnings are printed, so a non-deterministic kernel that only warns shows up in the log."""\n    import warnings\n\n    from torch.nn.attention import SDPBackend, sdpa_kernel\n\n    tabicl_attention = importlib.import_module("tabicl._model.attention")  # the upstream attention switch, not the model API\n\n    with contextlib.ExitStack() as stack:\n        caught = stack.enter_context(warnings.catch_warnings(record=True))\n        warnings.simplefilter("always")\n        stack.enter_context(sdpa_kernel(SDPBackend.MATH))\n        stack.enter_context(tabicl_attention.flash_attn3_toggle(False))\n        # tabicl\'s inference re-toggles FA3 from its own `use_fa3` per forward pass; switching the capability flag off\n        # too makes the path unreachable for the whole stage (flash_attn_interface is not in the lock anyway).\n        stack.callback(setattr, tabicl_attention, "HAS_FLASH_ATTN3", getattr(tabicl_attention, "HAS_FLASH_ATTN3", False))\n        tabicl_attention.HAS_FLASH_ATTN3 = False\n        globals()["_CAUGHT_WARNINGS"] = caught  # determinism_warnings() reads the live record during the stage\n        apply_determinism()\n        try:\n            yield\n        finally:\n            seen = set()\n            for w in caught:\n                text = f"{w.category.__name__}: {w.message}"\n                if text in seen:\n                    continue\n                seen.add(text)\n                prefix = "DETERMINISM WARNING" if DETERMINISM_WARNING.search(str(w.message)) else "warning"\n                print(f"{prefix}: {text}", flush=True)\n            globals()["_CAUGHT_WARNINGS"] = list(caught)\n\n\ndef device_name() -> str:\n    import torch\n\n    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"\n\n\nclass Run:\n    """Paths of one run: carried sources and state under ``root``; learner-facing files under ``outputs``."""\n\n    def __init__(self, root: Path, outputs: Path, weights: Path, options: dict[str, Any]) -> None:\n        self.root = root\n        self.out = outputs\n        self.weights = weights\n        self.options = options\n        self.state = root / "state"\n        self.out.mkdir(parents=True, exist_ok=True)\n        self.state.mkdir(parents=True, exist_ok=True)\n\n    def write_state(self, name: str, value: Any) -> Path:\n        return atomic_write_json(self.state / name, value)\n\n    def read_state(self, name: str, needed_by: str) -> Any:\n        path = self.state / name\n        if not path.is_file():\n            raise RuntimeError(f"{name} is missing: run the stage that writes it before \'{needed_by}\' (run the notebook from Section 4)")\n        try:\n            return json.loads(path.read_text(encoding="utf-8"))\n        except json.JSONDecodeError as exc:\n            raise RuntimeError(f"{name} is incomplete or corrupt (an interrupted stage?): re-run the stage that writes it before \'{needed_by}\' (run the notebook from Section 4)") from exc\n\n    def write_frame(self, name: str, frame) -> None:\n        buffer = io.BytesIO()\n        frame.to_parquet(buffer, index=True, engine="pyarrow")\n        atomic_write_bytes(self.state / name, buffer.getvalue())\n\n    def read_frame(self, name: str, needed_by: str):\n        import pandas as pd\n\n        path = self.state / name\n        if not path.is_file():\n            raise RuntimeError(f"{name} is missing: run the stage that writes it before \'{needed_by}\' (run the notebook from Section 4)")\n        return pd.read_parquet(path, engine="pyarrow")\n\n    def write_output(self, name: str, value: Any) -> Path:\n        return atomic_write_json(self.out / name, value)\n\n\ndef package(root: Path):\n    src = str(root / "src")\n    if src not in sys.path:\n        sys.path.insert(0, src)\n    return importlib.import_module(PACKAGE)\n\n\ndef sha256_file(path: Path) -> str:\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef rounded(value: Any, digits: int = 4) -> Any:\n    if isinstance(value, float):\n        return round(value, digits) if math.isfinite(value) else value\n    if isinstance(value, dict):\n        return {k: rounded(v, digits) for k, v in value.items()}\n    if isinstance(value, list | tuple):\n        return [rounded(v, digits) for v in value]\n    return value\n\n\n# --------------------------------------------------------------------------------------------------\n# data (TIC-m1, TIC-m2)\n# --------------------------------------------------------------------------------------------------\n\n\ndef stratified_60_20_20(frame, target: str):\n    from sklearn.model_selection import train_test_split\n\n    train, remainder = train_test_split(frame, test_size=0.4, random_state=SEED, stratify=frame[target])\n    holdout, test = train_test_split(remainder, test_size=0.5, random_state=SEED, stratify=remainder[target])\n    return train, holdout, test\n\n\ndef load_sample(P, source: str):\n    """(train, holdout, test, name, target) for one of the three public samples."""\n    if source == "Sample: Breast Cancer":\n        from sklearn.datasets import load_breast_cancer\n\n        dataset = load_breast_cancer(as_frame=True)\n        frame = dataset.frame.rename(columns={dataset.target.name: "target"})\n        frame["target"] = frame["target"].map({0: "malignant", 1: "benign"})\n        return (*stratified_60_20_20(frame, "target"), "sklearn-breast-cancer", "target")\n    if source == "Sample: Wine":\n        from sklearn.datasets import load_wine\n\n        dataset = load_wine(as_frame=True)\n        frame = dataset.frame.rename(columns={dataset.target.name: "target"})\n        frame["target"] = frame["target"].map(dict(enumerate(dataset.target_names)))\n        return (*stratified_60_20_20(frame, "target"), "sklearn-wine", "target")\n    if source == "Sample: Palmer Penguins":\n        import urllib.request\n        import zipfile\n\n        with urllib.request.urlopen(PENGUINS_URL, timeout=30) as response:\n            payload = response.read()\n        digest = hashlib.sha256(payload).hexdigest()\n        if digest != PENGUINS_SHA256:\n            raise ValueError(f"Palmer Penguins sample archive digest mismatch: {digest} != {PENGUINS_SHA256}")\n        with zipfile.ZipFile(io.BytesIO(payload)) as archive:\n            names = {Path(n).name: n for n in archive.namelist() if n.endswith(".csv")}\n            if not {"train.csv", "val.csv", "test.csv"} <= set(names):\n                raise ValueError("Palmer Penguins sample archive is missing train.csv, val.csv, or test.csv")\n            parts = [P.read_csv_payload(archive.read(names[n]), n) for n in ("train.csv", "val.csv", "test.csv")]\n        return (*parts, "palmer-penguins-cc0", "species")\n    if source == MAGIC_SOURCE:\n        raise ValueError("the MAGIC sample is loaded by the data stage (it needs the run\'s cache); choose another activity table")\n    raise ValueError(f"DATA_SOURCE must be one of {list(SAMPLE_SOURCES + UPLOAD_SOURCES)}, got {source!r}")\n\n\ndef fetch_magic(cache: Path) -> bytes:\n    """The pinned MAGIC archive: reused from ``cache`` when its digest matches, else fetched at the immutable commit."""\n    if cache.is_file() and hashlib.sha256(cache.read_bytes()).hexdigest() == MAGIC_SHA256:\n        return cache.read_bytes()\n    import urllib.request\n\n    with urllib.request.urlopen(MAGIC_URL, timeout=60) as response:\n        payload = response.read()\n    digest = hashlib.sha256(payload).hexdigest()\n    if digest != MAGIC_SHA256:\n        raise ValueError(f"MAGIC sample archive digest mismatch: {digest} != {MAGIC_SHA256}")\n    atomic_write_bytes(cache, payload)\n    return payload\n\n\ndef magic_member(payload: bytes, name: str) -> bytes:\n    """One CSV of the pinned archive, digest-checked. ``test.csv`` is read only by the fresh_test stage."""\n    import zipfile\n\n    with zipfile.ZipFile(io.BytesIO(payload)) as archive:\n        data = archive.read(name)\n    digest = hashlib.sha256(data).hexdigest()\n    if digest != MAGIC_MEMBERS[name]:\n        raise ValueError(f"MAGIC sample member {name} digest mismatch: {digest} != {MAGIC_MEMBERS[name]}")\n    return data\n\n\ndef check_byod_table(frame, name: str, target: str, text_columns: list[str]) -> None:\n    """Refuse, naming the file and the rule, a missing target column and numeric columns spoiled by a few strings\n    (they would otherwise be ordinal-encoded silently by sorted string value)."""\n    import pandas as pd\n\n    if target not in frame.columns:\n        raise ValueError(f"{name}: the target column {target!r} is not in the header {list(map(str, frame.columns))}. Set TARGET_COLUMN to the name of your label column.")\n    absent = [c for c in text_columns if c not in frame.columns]\n    if absent:\n        raise ValueError(f"{name}: TEXT_COLUMNS {absent} are not in the header {list(map(str, frame.columns))}.")\n    for column in frame.columns:\n        if column == target or column in text_columns or pd.api.types.is_numeric_dtype(frame[column]):\n            continue\n        values = frame[column].dropna()\n        if values.empty:\n            continue\n        numeric = pd.to_numeric(values.astype(str).str.strip(), errors="coerce")\n        share = float(numeric.notna().mean())\n        if share >= NUMERIC_LIKE_SHARE:\n            bad = values[numeric.isna()].astype(str).unique().tolist()[:5]\n            raise ValueError(\n                f"{name}: column {column!r} is numeric except for {int(numeric.isna().sum())} value(s) {bad}; it would be "\n                "encoded as categories and lose its numeric meaning. Fix those values (leave missing cells empty), or list "\n                "the column in TEXT_COLUMNS if it really is categorical."\n            )\n\n\ndef clear_outputs(run: Run) -> list[str]:\n    removed = []\n    for path in sorted(run.out.glob(f"{STEM}_*")):\n        if path.is_file():\n            path.unlink()\n            removed.append(path.name)\n    for name in ("artifact", "artifact-reload"):\n        if (run.out / name).is_dir():\n            shutil.rmtree(run.out / name)\n            removed.append(name + "/")\n    return removed\n\n\ndef stage_weights(run: Run) -> None:\n    P = package(run.root)\n    snapshot = run.weights / P.MODEL_KEY\n    snapshot.mkdir(parents=True, exist_ok=True)\n    carried = run.root / "weights" / P.MODEL_KEY / P.MANIFEST_NAME\n    manifest = json.loads(carried.read_text(encoding="utf-8"))\n    if (manifest["modelId"], manifest["revision"]) != (P.MODEL_ID, P.MODEL_REVISION):\n        raise RuntimeError("the carried manifest does not name the identity carried by the package; regenerate the notebook")\n    shutil.copyfile(carried, snapshot / P.MANIFEST_NAME)\n    print({"model_id": P.MODEL_ID, "revision": P.MODEL_REVISION, "license": P.MODEL_LICENSE, "files": len(manifest["files"]), "total_bytes": manifest["totalBytes"]})\n    fetched = P.stage_missing_files(snapshot, allow_download=True)\n    print({"weights_dir": str(snapshot), "fetched": fetched})\n    verified = P.verify_snapshot(snapshot)\n    print({"verified_files": len(verified["files"]), "revision": verified["revision"], "sha256": [f["sha256"] for f in verified["files"]]})\n    run.write_state("weights.json", {"snapshot": str(snapshot), "checkpoint": str(snapshot / P.BASE_CHECKPOINT_NAME)})\n\n\ndef stage_data(run: Run) -> None:\n    import pandas as pd\n\n    P = package(run.root)\n    opts = run.options\n    source = opts.get("data_source", DEFAULT_SOURCE)\n    use_byod = bool(opts.get("use_byod", False))\n    masked = fresh_test_masked()\n    print({"fresh_test_mask": f"{MASK_ENV}=1: the fresh test is not read in this run (the pinned archive holding it is still downloaded and digest-checked; test.csv is never extracted)" if masked else f"{MASK_ENV} unset: the fresh test is read only by the fresh_test stage, only under the pre-registered settings"})\n    if use_byod and source in SAMPLE_SOURCES:\n        raise ValueError(\'USE_BYOD=True requires DATA_SOURCE = "Upload CSV" or "Upload pre-split train/val/test".\')\n    if source in UPLOAD_SOURCES and not use_byod:\n        raise ValueError("Set USE_BYOD=True to use an upload DATA_SOURCE.")\n    removed = clear_outputs(run)\n    for name in ("train.parquet", "holdout.parquet", "test.parquet", "data.json", "encoded.json", "condition.json", "export_reference.json"):\n        (run.state / name).unlink(missing_ok=True)\n    if removed:\n        print({"removed_previous_outputs": removed})\n    text_columns = list(opts.get("text_columns") or [])\n    dropped: dict[str, int] = {}\n    extra: dict[str, Any] = {}\n    if source == MAGIC_SOURCE:\n        payload = fetch_magic(run.root / "inputs" / "magic-gamma.zip")\n        train = P.read_csv_payload(magic_member(payload, "train.csv"), "train.csv")\n        holdout = P.read_csv_payload(magic_member(payload, "val.csv"), "val.csv")\n        test, name, target, kind = None, "uci-159-magic-gamma-telescope", "target", "sample"\n        extra = {"archive_sha256": MAGIC_SHA256, "source_sha256": MAGIC_SOURCE_SHA256, "row_ids_sha256": MAGIC_ROW_IDS_SHA256, "holdout_role": "validation (1,000 rows): grid selection and early stopping", "test_role": "fresh test (1,500 rows): not loaded here; read only by the fresh_test stage under the pre-registered settings", "fresh_test_rows": MAGIC_FRESH_TEST_ROWS, "fresh_test_member_sha256": MAGIC_MEMBERS["test.csv"], "reserve_rows": 13520}\n    elif source in SAMPLE_SOURCES:\n        train, holdout, test, name, target = load_sample(P, source)\n        kind = "sample"\n        extra = {"test_role": "exposed development test (scored in the 2026-10-10 and 2026-10-11 runs; an earlier-run reference)" if source == REFERENCE_SOURCE else "independent test (exploratory sample)"}\n    else:\n        target = opts.get("target_column") or "target"\n        paths = opts.get("paths") or {}\n        def read(key: str):\n            path = Path(paths.get(key) or "")\n            if not str(path) or not path.is_file():\n                raise FileNotFoundError(f"BYOD {key} file {str(path)!r} does not exist in this runtime: upload it or correct the path field.")\n            if path.suffix.lower() != ".csv":\n                raise ValueError(f"{path.name}: BYOD files must be CSV.")\n            frame = P.read_csv_payload(path.read_bytes(), path.name)\n            check_byod_table(frame, path.name, target, text_columns)\n            return frame, path.name\n        if source == "Upload CSV":\n            frame, name = read("csv")\n            raw_manifest = P.validate_inputs(frame, target_column=target, names=[name])\n            dropped["csv"] = raw_manifest["inputs"][0]["dropped_missing_target_rows"]\n            frame, _ = P.prepare_classification_table(frame, target)\n            P.check_stratifiable(frame, target)\n            from sklearn.model_selection import train_test_split\n\n            train, holdout = train_test_split(frame, test_size=VALIDATION_SPLIT, random_state=SEED, stratify=frame[target])\n            test = None\n        else:\n            train, train_name = read("train")\n            holdout, _ = read("val")\n            test = read("test")[0] if paths.get("test") else None\n            name = "pre-split upload (" + train_name + ")"\n        kind = "BYOD"\n        extra = {"test_role": "your test partition (exploratory: BYOD never scores the fresh test)"}\n    parts = {"train": train, "holdout": holdout, "test": test}\n    for key, frame in parts.items():\n        if frame is not None:\n            run.write_frame(f"{key}.parquet", frame)\n    digest = hashlib.sha256(pd.concat([f for f in parts.values() if f is not None]).to_csv(index=False).encode("utf-8")).hexdigest()\n    record = {"sample_kind": kind, "name": name, "source": source, "target": target, "train_rows": len(train), "holdout_rows": len(holdout), "test_rows": 0 if test is None else len(test), "csv_sha256": digest, "dropped_missing_target_rows_before_split": dropped, "text_columns": text_columns, "use_byod": use_byod, **extra}\n    run.write_state("data.json", record)\n    print(record)\n    if dropped.get("csv"):\n        print(f"{dropped[\'csv\']} row(s) with a missing target were dropped before the split; Section 5\'s input manifest records them.")\n\n\ndef stage_validate(run: Run) -> None:\n    P = package(run.root)\n    data = run.read_state("data.json", "validate")\n    target, name = data["target"], data["name"]\n    train = run.read_frame("train.parquet", "validate")\n    holdout = run.read_frame("holdout.parquet", "validate")\n    test = run.read_frame("test.parquet", "validate") if (run.state / "test.parquet").is_file() else None\n    print({"ceilings": {"MIN_TRAIN_ROWS": P.MIN_TRAIN_ROWS, "MIN_EVAL_ROWS": P.MIN_EVAL_ROWS, "MAX_TRAIN_ROWS": P.MAX_TRAIN_ROWS, "MAX_FEATURES": P.MAX_FEATURES, "MIN_CLASSES": P.MIN_CLASSES, "MIN_ROWS_PER_CLASS": P.MIN_ROWS_PER_CLASS}, "decision_rule": P.DECISION_RULE})\n    manifest = P.validate_inputs(train, target_column=target, names=[name + ":train"])\n    if data["dropped_missing_target_rows_before_split"].get("csv"):\n        manifest["inputs"][0]["dropped_missing_target_rows_before_split"] = data["dropped_missing_target_rows_before_split"]["csv"]\n    manifest["inputs"].extend(P.validate_inputs(holdout, target_column=target, min_rows=P.MIN_EVAL_ROWS, names=[name + ":holdout"])["inputs"])\n    if test is not None:\n        manifest["inputs"].extend(P.validate_inputs(test, target_column=target, min_rows=P.MIN_EVAL_ROWS, names=[name + ":test"])["inputs"])\n    # Demonstrate rejection on a probe that breaks a ceiling; the finding is recorded, not swallowed.\n    try:\n        P.validate_inputs(train.head(P.MIN_TRAIN_ROWS - 1), target_column=target)\n    except ValueError as exc:\n        manifest["findings"].append({"input": "too-few-rows-probe", "verdict": "rejected", "message": str(exc)})\n    train_row_ids = list(map(int, train.index)) if train.index.dtype.kind in "iu" else list(range(len(train)))\n    holdout_row_ids = list(map(int, holdout.index)) if holdout.index.dtype.kind in "iu" else list(range(len(holdout)))\n    train, dropped_train = P.prepare_classification_table(train, target)\n    holdout, dropped_holdout = P.prepare_classification_table(holdout, target, min_rows=P.MIN_EVAL_ROWS, min_classes=1)\n    dropped_test = 0\n    if test is not None:\n        test, dropped_test = P.prepare_classification_table(test, target, min_rows=P.MIN_EVAL_ROWS, min_classes=1)\n    features = [c for c in train.columns if c != target]\n    classes = sorted(train[target].unique(), key=str)\n    train = train[features + [target]].reset_index(drop=True)\n    holdout = P.align_to_schema(holdout, features, target, classes)\n    test = P.align_to_schema(test, features, target, classes) if test is not None else None\n    encoders = P.fit_categorical_encoder(train, features)\n    train_enc, _ = P.apply_categorical_encoder(train, encoders)\n    holdout_enc, unseen_holdout = P.apply_categorical_encoder(holdout, encoders)\n    test_enc, unseen_test = P.apply_categorical_encoder(test, encoders) if test is not None else (None, {})\n    baseline = P.majority_class_baseline(train_enc[target], holdout_enc[target])\n    run.write_output(f"{STEM}_input_manifest.json", manifest)\n    print(json.dumps(manifest["inputs"][0], indent=2))\n    print("findings:", manifest["findings"])\n    for key, frame in (("train_raw", train), ("holdout_raw", holdout), ("train_enc", train_enc), ("holdout_enc", holdout_enc)):\n        run.write_frame(f"{key}.parquet", frame)\n    if test is not None:\n        run.write_frame("test_enc.parquet", test_enc)\n    shares = train[target].value_counts(normalize=True).round(3).to_dict()\n    dropped = {"train": dropped_train, "holdout": dropped_holdout, "test": dropped_test, **({"before_split": data["dropped_missing_target_rows_before_split"]["csv"]} if data["dropped_missing_target_rows_before_split"].get("csv") else {})}\n    run.write_state("encoded.json", {"features": features, "target": target, "classes": [str(c) for c in classes], "encoders": encoders, "baseline": baseline, "dropped": dropped, "holdout_row_ids": holdout_row_ids if len(holdout_row_ids) == len(holdout) else list(range(len(holdout))), "train_row_ids": train_row_ids, "class_shares": {str(k): v for k, v in shares.items()}})\n    print({"dropped_missing_target_rows": dropped, "unseen_categorical_values": {"holdout": unseen_holdout, "test": unseen_test}, "encoded_categoricals": sorted(encoders)})\n    print({"train": len(train_enc), "holdout": len(holdout_enc), "test": 0 if test_enc is None else len(test_enc), "features": len(features), "classes": [str(c) for c in classes], "training_class_shares": shares})\n    print("majority-class baseline on the holdout", rounded(baseline))\n\n\n# --------------------------------------------------------------------------------------------------\n# model stages\n# --------------------------------------------------------------------------------------------------\n\n\ndef encoded(run: Run, needed_by: str):\n    enc = run.read_state("encoded.json", needed_by)\n    train = run.read_frame("train_enc.parquet", needed_by)\n    holdout = run.read_frame("holdout_enc.parquet", needed_by)\n    test = run.read_frame("test_enc.parquet", needed_by) if (run.state / "test_enc.parquet").is_file() else None\n    return enc, train, holdout, test\n\n\ndef pretrained(run: Run, n_estimators: int = INFERENCE_ESTIMATORS, random_state: int = SEED):\n    P = package(run.root)\n    return P.TabICLClassificationPipeline.from_pretrained(weights_dir=run.weights / P.MODEL_KEY, n_estimators=n_estimators, random_state=random_state)\n\n\ndef from_checkpoint(run: Run, checkpoint: Path, like, source: str):\n    P = package(run.root)\n    return P.TabICLClassificationPipeline(P.create_classifier(model_path=checkpoint, allow_auto_download=False, n_estimators=like.n_estimators, random_state=SEED, device=like.device), model_path=checkpoint, n_estimators=like.n_estimators, random_state=SEED, device=like.device, source=source)\n\n\ndef active_model(run: Run, enc: dict, train):\n    """The model Section 6 selected, conditioned on the support rows."""\n    cond = run.read_state("condition.json", "active model")\n    model = pretrained(run)\n    if cond["active_mode"] == "fine-tuned":\n        model = from_checkpoint(run, Path(cond["active_checkpoint"]), model, "fine-tuned")\n    model.fit(train[enc["features"]], train[enc["target"]])\n    return model, cond\n\n\ndef score(P, model, frame, enc):\n    X = frame[enc["features"]]\n    return P.classification_metrics(frame[enc["target"]].to_numpy(), model.predict(X), model.predict_proba(X), model.classes_)\n\n\nPREREGISTERED: dict[str, Any] = {\n    "data_source": MAGIC_SOURCE,\n    "use_byod": False,\n    "archive_sha256": MAGIC_SHA256,\n    "row_ids_sha256": MAGIC_ROW_IDS_SHA256,\n    "run_fine_tuning": True,\n    "grid_epochs_cap": 0,\n    "grid_learning_rate": 0.0,\n    "random_state": SEED,\n    "cpu_grid": False,\n    "grid": [dict(c) for c in GRID],\n    "grid_common": dict(GRID_COMMON),\n    "selection_tolerance": SELECTION_TOLERANCE,\n    "inference_estimators": INFERENCE_ESTIMATORS,\n    "bootstrap": dict(BOOTSTRAP),\n    "determinism": dict(DETERMINISM),\n}\n\n\ndef current_settings(opts: dict[str, Any], data: dict[str, Any], determinism: dict[str, Any] | None, device: str) -> dict[str, Any]:\n    """Every setting the pre-registration fixes, as this run used it (SCORE_ONCE_RULE item 1)."""\n    return {\n        "data_source": data.get("source"),\n        "use_byod": bool(data.get("use_byod", False)),\n        "archive_sha256": data.get("archive_sha256"),\n        "row_ids_sha256": data.get("row_ids_sha256"),\n        "run_fine_tuning": bool(opts.get("run_fine_tuning", True)),\n        "grid_epochs_cap": int(opts.get("grid_epochs_cap", 0) or 0),\n        "grid_learning_rate": float(opts.get("grid_learning_rate", 0.0) or 0.0),\n        "random_state": int(opts.get("random_state", SEED)),\n        "cpu_grid": bool(opts.get("cpu_grid", False)),\n        "grid": [dict(c) for c in GRID],\n        "grid_common": dict(GRID_COMMON),\n        "selection_tolerance": SELECTION_TOLERANCE,\n        "inference_estimators": INFERENCE_ESTIMATORS,\n        "bootstrap": dict(BOOTSTRAP),\n        "determinism": determinism,\n        "device": device,\n    }\n\n\ndef settings_differing(settings: dict[str, Any]) -> list[str]:\n    """Names of the settings that differ from the pre-registration; empty means ``preregistered``. Computed from ALL\n    fixed settings, including the device (a Tesla T4 is pre-registered)."""\n    differing = [key for key, value in PREREGISTERED.items() if settings.get(key) != value]\n    if REQUIRED_DEVICE not in str(settings.get("device", "")):\n        differing.append("device")\n    return differing\n\n\ndef grid_hash(grid: list[dict[str, Any]], selection: str, sdpa_backend: str | None) -> str:\n    """Fingerprint of the grid\'s outcome, printed and reported for information. On a T4 the fp16 grid has given different hashes\n    run to run at an identical revision (issue #28), so it never decides a reproduction; within one run it still checks that the\n    stored grid is the one the condition stage wrote."""\n    def r(value):\n        return None if value is None else round(float(value), 6)\n\n    rows = [{"id": g.get("id"), "epochs_run": g.get("epochs_run"), "curve": [r(v) for v in g.get("val_log_loss_per_epoch") or []], "baseline": r(g.get("baseline_val_log_loss")), "candidate_val_log_loss": r((g.get("candidate_val") or {}).get("log_loss")), "delta_best": r(g.get("param_delta_l2_best")), "delta_last": r(g.get("param_delta_l2_last_epoch"))} for g in grid]\n    return hashlib.sha256(json.dumps({"grid": rows, "selection": selection, "sdpa_backend": sdpa_backend}, sort_keys=True).encode("utf-8")).hexdigest()\n\n\ndef model_state(pipeline) -> dict[str, Any]:\n    """The loaded network\'s weights of a conditioned pipeline (tabicl deserialises the verified checkpoint on ``fit``)."""\n    return {k: v.detach().cpu() for k, v in pipeline.estimator.model_.state_dict().items()}\n\n\ndef param_delta_l2(a: dict[str, Any], b: dict[str, Any]) -> float:\n    """L2 norm of the difference over the floating-point tensors both state dicts hold: 0.0 means the weights are identical."""\n    total = 0.0\n    for key, value in a.items():\n        other = b.get(key)\n        if other is None or not hasattr(value, "is_floating_point") or not value.is_floating_point():\n            continue\n        total += float(((other.float() - value.float()) ** 2).sum())\n    return math.sqrt(total)\n\n\nclass StepCounter:\n    """Counts optimizer steps instead of computing them (R96 round 2, F2; amendment 1).\n\n    ``attempted`` counts the AMP grad scaler\'s ``step`` calls (one per training batch); ``taken`` counts the optimizer\'s own\n    steps through a step-post hook, which the scaler does not reach when it skips an update whose fp16 gradients overflowed.\n    With AMP off (CPU) the scaler is disabled and every attempt is taken."""\n\n    def __init__(self) -> None:\n        self.attempted = 0\n        self.taken = 0\n\n    def on_optimizer_step(self, *args, **kwargs) -> None:\n        self.taken += 1\n\n    def instrument(self, finetuner) -> None:\n        make_amp = finetuner._make_amp\n\n        def counted_make_amp(*args, **kwargs):\n            use_amp, scaler, ctx = make_amp(*args, **kwargs)\n            step = scaler.step\n\n            def counted_step(optimizer, *a, **k):\n                self.attempted += 1\n                return step(optimizer, *a, **k)\n\n            scaler.step = counted_step\n            return use_amp, scaler, ctx\n\n        finetuner._make_amp = counted_make_amp\n\n    def record(self, planned: int) -> dict[str, int]:\n        return {"optimizer_steps_planned": planned, "optimizer_steps_attempted": self.attempted, "optimizer_steps_taken": self.taken, "amp_skipped_steps": self.attempted - self.taken}\n\n\ndef run_config(run: Run, P, cfg: dict[str, Any], pipe, enc: dict, train, holdout, *, device: str, seed: int, epochs_cap: int, lr_override: float, base_state: dict) -> dict[str, Any]:\n    """One grid config: fine-tune with early stopping on validation log loss, then re-score its best.ckpt at the inference\n    ensemble size. Records the validation curve, optimizer steps and whether the weights tabicl kept changed."""\n    import time\n\n    from tabicl._finetune.data import count_chunks\n\n    ft_dir = run.state / "finetune" / cfg["id"]\n    shutil.rmtree(ft_dir, ignore_errors=True)\n    ft_dir.mkdir(parents=True)\n    epochs = min(cfg["epochs"], epochs_cap) if epochs_cap else cfg["epochs"]\n    lr = lr_override or cfg["learning_rate"]\n    c = GRID_COMMON\n    finetuner = P.create_finetuned_classifier(epochs=epochs, learning_rate=lr, weight_decay=c["weight_decay"], grad_clip=c["grad_clip"], amp=device == "cuda", use_lr_scheduler=c["use_lr_scheduler"], n_estimators_finetune=cfg["n_estimators_finetune"], n_estimators_validation=cfg["n_estimators_validation"], n_estimators_inference=c["n_estimators_inference"], max_data_size=c["max_data_size"], finetune_ctx_query_ratio=c["finetune_ctx_query_ratio"], early_stopping=c["early_stopping"], patience=cfg["patience"], min_delta=c["min_delta"], time_limit=cfg["time_limit"], eval_metric=c["eval_metric"], model_path=str(pipe.model_path), allow_auto_download=False, device=device, random_state=seed, verbose=True, support_many_classes=True)\n    curve: list[float] = []\n    validate = finetuner._validate_current_model\n\n    def recorded(*args, **kwargs):  # baseline, then one validation per epoch (log loss is negated by tabicl)\n        result = validate(*args, **kwargs)\n        curve.append(-float(result.primary))\n        return result\n\n    finetuner._validate_current_model = recorded\n    import torch.optim.optimizer as optim_hooks\n\n    counter = StepCounter()\n    counter.instrument(finetuner)\n    hook = optim_hooks.register_optimizer_step_post_hook(counter.on_optimizer_step)\n    t0 = time.perf_counter()\n    try:\n        P.fine_tune_classifier(finetuner, train[enc["features"]], train[enc["target"]], X_val=holdout[enc["features"]], y_val=holdout[enc["target"]], output_dir=str(ft_dir))\n    finally:\n        hook.remove()\n    seconds = time.perf_counter() - t0\n    best = ft_dir / "best.ckpt"\n    if not best.exists():\n        raise RuntimeError(f"Fine-tuning config {cfg[\'id\']} did not produce best.ckpt")\n    steps_per_epoch = count_chunks(len(train), c["max_data_size"])\n    epochs_run = max(0, len(curve) - 1)\n    epoch_ckpts = sorted(ft_dir.glob("epoch*.ckpt"), key=lambda path: int("".join(ch for ch in path.stem if ch.isdigit()) or 0))\n    last = epoch_ckpts[-1] if epoch_ckpts else None\n    candidate = from_checkpoint(run, best, pipe, "fine-tuned")\n    candidate.fit(train[enc["features"]], train[enc["target"]])\n    candidate_val = score(P, candidate, holdout, enc)\n    delta_best = param_delta_l2(base_state, model_state(candidate))\n    last_val = delta_last = None\n    if last is not None:\n        final = from_checkpoint(run, last, pipe, "fine-tuned (last epoch, exploratory)")\n        final.fit(train[enc["features"]], train[enc["target"]])\n        last_val = score(P, final, holdout, enc)\n        delta_last = param_delta_l2(base_state, model_state(final))\n    for checkpoint in epoch_ckpts:\n        checkpoint.unlink()  # keep best.ckpt only: TabICL writes one 331 MB checkpoint per epoch\n    record = {"id": cfg["id"], "role": cfg["role"], "learning_rate": lr, "lr_schedule": "constant (no warmup, so no zero-LR step)", "epochs_requested": epochs, "epochs_run": epochs_run, "patience": cfg["patience"], "steps_per_epoch": steps_per_epoch, **counter.record(epochs_run * steps_per_epoch), "optimizer_steps": counter.taken, "nonzero_lr_steps": counter.taken, "baseline_val_log_loss": curve[0] if curve else None, "val_log_loss_per_epoch": curve[1:], "weights_changed": delta_best > 0.0, "param_delta_l2_best": delta_best, "param_delta_l2_last_epoch": delta_last, "candidate_val": candidate_val, "last_epoch_val_exploratory": last_val, "best_ckpt": str(best), "best_ckpt_sha256": sha256_file(best), "seconds": round(seconds, 1), "complete": True}\n    print({"config": cfg["id"], "role": cfg["role"], "lr": lr, "epochs_run": epochs_run, "optimizer_steps_attempted": record["optimizer_steps_attempted"], "optimizer_steps_taken": record["optimizer_steps_taken"], "amp_skipped_steps": record["amp_skipped_steps"], "baseline_val_log_loss": rounded(record["baseline_val_log_loss"]), "val_log_loss_per_epoch": rounded(curve[1:]), "weights_changed": record["weights_changed"], "param_delta_l2_best": rounded(delta_best, 6), "param_delta_l2_last_epoch": rounded(delta_last, 6), "candidate_val_log_loss": rounded(candidate_val["log_loss"]), "seconds": record["seconds"]})\n    if not record["weights_changed"]:\n        print(f"  {cfg[\'id\']}: no epoch beat tabicl\'s baseline by min_delta {c[\'min_delta\']}, so tabicl restored the pretrained weights and wrote them as best.ckpt (parameter delta 0). The last epoch\'s weights did move (delta {rounded(delta_last, 6)}); they are only read out on validation below.")\n    return record\n\n\ndef select_config(pretrained_val: dict[str, float], grid: list[dict[str, Any]]) -> tuple[dict[str, Any] | None, str]:\n    """Pre-registered selection on validation log loss: keep the pretrained model unless a config beats it by more than\n    SELECTION_TOLERANCE; ties within SELECTION_TIE go to the config with fewer optimizer steps."""\n    reference = pretrained_val["log_loss"]\n    # A restored config (kept weights == pretrained weights) is the pretrained model, never "fine-tuned" (amendment 1).\n    eligible = [g for g in grid if g.get("complete") and g.get("weights_changed") is True and g["candidate_val"]["log_loss"] < reference - SELECTION_TOLERANCE]\n    if not eligible:\n        return None, f"validation:log_loss; pretrained kept (no config beat {reference:.4f} by more than {SELECTION_TOLERANCE})"\n    lowest = min(g["candidate_val"]["log_loss"] for g in eligible)\n    tied = [g for g in eligible if g["candidate_val"]["log_loss"] <= lowest + SELECTION_TIE]\n    chosen = min(tied, key=lambda g: (g["optimizer_steps"], g["candidate_val"]["log_loss"]))\n    return chosen, f"validation:log_loss; {chosen[\'id\']} beat the pretrained model by {reference - chosen[\'candidate_val\'][\'log_loss\']:.4f} > {SELECTION_TOLERANCE}"\n\n\ndef stage_condition(run: Run) -> None:\n    import torch\n\n    P = package(run.root)\n    data = run.read_state("data.json", "condition")\n    enc, train, holdout, test = encoded(run, "condition")\n    opts = run.options\n    for retired in ("fine_tune_epochs", "eval_metric"):\n        if retired in opts:\n            raise ValueError(f"{retired} was replaced by the pre-registered grid (docs/magic-finetune-preregistration.md); use GRID_EPOCHS_CAP / GRID_LEARNING_RATE for an exploratory run")\n    run_ft = bool(opts.get("run_fine_tuning", True))\n    seed = int(opts.get("random_state", SEED))\n    epochs_cap = int(opts.get("grid_epochs_cap", 0) or 0)\n    lr_override = float(opts.get("grid_learning_rate", 0.0) or 0.0)\n    if epochs_cap < 0 or not 0.0 <= lr_override <= 1e-3:\n        raise ValueError("GRID_EPOCHS_CAP must be >= 0 and GRID_LEARNING_RATE from 0 (pre-registered) to 1e-3")\n    determinism = apply_determinism()\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    name = device_name()\n    settings = current_settings(opts, data, determinism, name)\n    differing = settings_differing(settings)\n    print({"device": name, "determinism": determinism, "preregistered": not differing, "settings_differing": differing})\n    X_train, y_train = train[enc["features"]], train[enc["target"]]\n    pipe = pretrained(run, random_state=seed)\n    pipe.fit(X_train, y_train)\n    pretrained_holdout = score(P, pipe, holdout, enc)\n    test_role = data.get("test_role", "independent test")\n    pretrained_test = score(P, pipe, test, enc) if test is not None else None\n    print("pretrained validation", rounded(pretrained_holdout))\n    if pretrained_test:\n        print(f"pretrained {test_role}", rounded(pretrained_test))\n    grid: list[dict[str, Any]] = []\n    fine_tune: dict[str, Any] = {"requested": run_ft, "ran": False}\n    cpu_grid = bool(opts.get("cpu_grid", False))\n    if run_ft and device != "cuda" and not cpu_grid:\n        fine_tune["skipped"] = "no CUDA device: the pre-registered grid runs on a GPU runtime (a CPU run would take tens of minutes); recorded here, and the run is exploratory"\n        print("Fine-tuning skipped:", fine_tune["skipped"], "- choose Runtime > Change runtime type > T4 GPU to run the grid.")\n    elif run_ft and len(holdout) < MIN_SELECTION_HOLDOUT_ROWS:\n        fine_tune["skipped"] = f"validation partition too small for selection: {len(holdout)} < {MIN_SELECTION_HOLDOUT_ROWS} rows"\n        print("Fine-tuning skipped:", fine_tune["skipped"])\n    elif run_ft:\n        base_state = model_state(pipe)\n        for cfg in GRID:\n            grid.append(run_config(run, P, cfg, pipe, enc, train, holdout, device=device, seed=seed, epochs_cap=epochs_cap, lr_override=lr_override, base_state=base_state))\n        for g in grid:  # the package\'s strict comparison, recorded beside the pre-registered tolerance rule\n            g["strictly_better_than_pretrained"] = bool(P.compare_metric(g["candidate_val"], pretrained_holdout, "log_loss"))\n        fine_tune.update(ran=True, configs=[g["id"] for g in grid])\n    chosen, selection = select_config(pretrained_holdout, grid) if grid else (None, "default:pretrained (no grid run)")\n    active_mode, active_checkpoint = ("fine-tuned", chosen["best_ckpt"]) if chosen else ("pretrained", str(pipe.model_path))\n    for g in grid:  # keep the selected checkpoint only; every config\'s digest stays in the record\n        if not chosen or g["id"] != chosen["id"]:\n            Path(g["best_ckpt"]).unlink(missing_ok=True)\n    readout = None\n    finals = [g for g in grid if g.get("last_epoch_val_exploratory")]\n    if finals:\n        best_final = min(finals, key=lambda g: g["last_epoch_val_exploratory"]["log_loss"])\n        readout = {"label": "exploratory readout, validation only (never the fresh test)", "config": best_final["id"], "last_epoch_val": best_final["last_epoch_val_exploratory"], "param_delta_l2_last_epoch": best_final["param_delta_l2_last_epoch"], "pretrained_val": pretrained_holdout, "val_log_loss_difference_last_minus_pretrained": best_final["last_epoch_val_exploratory"]["log_loss"] - pretrained_holdout["log_loss"]}\n    candidate_holdout = chosen["candidate_val"] if chosen else (min((g["candidate_val"] for g in grid), key=lambda m: m["log_loss"]) if grid else None)\n    candidate_test = None\n    if chosen and test is not None:\n        model = from_checkpoint(run, Path(active_checkpoint), pipe, "fine-tuned")\n        model.fit(X_train, y_train)\n        candidate_test = score(P, model, test, enc)\n    ghash = grid_hash(grid, selection, determinism.get("sdpa_backend"))\n    record = {"active_mode": active_mode, "active_checkpoint": active_checkpoint, "active_checkpoint_sha256": sha256_file(Path(active_checkpoint)), "selection": selection, "selection_tolerance": SELECTION_TOLERANCE, "eval_metric": "log_loss", "fine_tune": fine_tune, "grid": grid, "grid_hash": ghash, "exploratory_readout": readout, "pretrained_holdout": pretrained_holdout, "pretrained_test": pretrained_test, "candidate_holdout": candidate_holdout, "candidate_test": candidate_test, "test_role": test_role, "classes": [str(c) for c in pipe.classes_], "device": pipe.device, "device_name": name, "settings": settings, "settings_differing": differing, "preregistered": not differing, "determinism_warnings": determinism_warnings()}\n    run.write_state("condition.json", record)\n    print({"recommended_for_export": active_mode, "selection_basis": selection, "grid_hash": ghash, "device": name, "classes_": record["classes"]})\n    if readout:\n        print({"exploratory_readout": rounded(readout)})\n    print("grid hash (informational: on a T4 it can differ run to run at an identical revision, issue #28):", ghash)\n\n\n# --------------------------------------------------------------------------------------------------\n# fresh test (score-once, fail-closed; SCORE_ONCE_RULE.md items 1-8)\n# --------------------------------------------------------------------------------------------------\n\n\ndef wilson(successes: int, n: int, z: float = 1.959964) -> list[float]:\n    if n == 0:\n        return [float("nan"), float("nan")]\n    p = successes / n\n    centre = (p + z * z / (2 * n)) / (1 + z * z / n)\n    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)\n    return [centre - half, centre + half]\n\n\ndef fresh_test_reading(y, p_pretrained, p_selected, classes, *, same_model: bool, exclude=None) -> dict[str, Any]:\n    """The pre-registered reading rule on row-level arrays: paired bootstrap over rows of the log-loss difference\n    (selected - pretrained), the verdict, and the secondary accuracy readouts."""\n    import numpy as np\n    from sklearn.metrics import roc_auc_score\n\n    y = np.asarray(y).astype(str)\n    labels = [str(c) for c in classes]\n    index = np.array([labels.index(v) for v in y])\n    pa = np.asarray(p_pretrained, dtype=float)\n    pb = np.asarray(p_selected, dtype=float)\n    keep = np.ones(len(y), dtype=bool) if exclude is None else ~np.asarray(exclude, dtype=bool)\n    y, index, pa, pb = y[keep], index[keep], pa[keep], pb[keep]\n    n = len(y)\n    rows = np.arange(n)\n    ll_a = -np.log(np.clip(pa[rows, index], 1e-15, 1.0))\n    ll_b = -np.log(np.clip(pb[rows, index], 1e-15, 1.0))\n    ok_a = (pa.argmax(axis=1) == index).astype(float)\n    ok_b = (pb.argmax(axis=1) == index).astype(float)\n    d_ll, d_acc = ll_b - ll_a, ok_b - ok_a\n    rng = np.random.default_rng(BOOTSTRAP["seed"])\n    boot_ll, boot_acc = [], []\n    for start in range(0, BOOTSTRAP["n_boot"], 1000):\n        draws = rng.integers(0, n, size=(min(1000, BOOTSTRAP["n_boot"] - start), n))\n        boot_ll.append(d_ll[draws].mean(axis=1))\n        boot_acc.append(d_acc[draws].mean(axis=1))\n    boot_ll, boot_acc = np.concatenate(boot_ll), np.concatenate(boot_acc)\n    delta = float(d_ll.mean())\n    ci = [float(np.percentile(boot_ll, 2.5)), float(np.percentile(boot_ll, 97.5))]\n    if same_model:\n        verdict = "same model (Δ = 0)"\n    elif ci[1] < 0 and delta <= -PRACTICAL_DELTA:\n        verdict = "improved"\n    elif ci[0] > 0:\n        verdict = "worse"\n    else:\n        verdict = "no detectable difference"\n\n    def auc(p):\n        try:\n            return float(roc_auc_score(index, p[:, 1])) if len(labels) == 2 else float(roc_auc_score(index, p, multi_class="ovr"))\n        except ValueError:\n            return float("nan")\n\n    return {"n": n, "primary": {"endpoint": "fresh-test log loss difference, selected - pretrained", "delta": delta, "ci95": ci, "verdict": verdict, "practical_delta": PRACTICAL_DELTA, "bootstrap": dict(BOOTSTRAP)}, "log_loss": {"pretrained": float(ll_a.mean()), "selected": float(ll_b.mean())}, "accuracy": {"pretrained": float(ok_a.mean()), "pretrained_wilson95": wilson(int(ok_a.sum()), n), "selected": float(ok_b.mean()), "selected_wilson95": wilson(int(ok_b.sum()), n), "difference": float(d_acc.mean()), "difference_ci95": [float(np.percentile(boot_acc, 2.5)), float(np.percentile(boot_acc, 97.5))]}, "roc_auc": {"pretrained": auc(pa), "selected": auc(pb)}}\n\n\ndef fresh_test_probabilities(run: Run, P, cond: dict, test):\n    """(y, p_pretrained, p_selected, classes, max_row_sum_deviation) on the fresh test, both models conditioned on the\n    same support rows at the inference ensemble size."""\n    import numpy as np\n\n    enc, train, _holdout, _ = encoded(run, "fresh_test")\n    features, target = enc["features"], enc["target"]\n    test = P.align_to_schema(test, features, target, enc["classes"])\n    test, _ = P.apply_categorical_encoder(test, enc["encoders"])\n    pre = pretrained(run)\n    pre.fit(train[features], train[target])\n    raw = np.asarray(pre.estimator.predict_proba(test[features]), dtype=float)\n    p_pre, deviation = P.renormalise_probabilities(raw)\n    if cond["active_mode"] == "pretrained":\n        p_sel = p_pre\n    else:\n        sel = from_checkpoint(run, Path(cond["active_checkpoint"]), pre, "fine-tuned")\n        sel.fit(train[features], train[target])\n        p_sel = sel.predict_proba(test[features])\n    return test[target].to_numpy(), p_pre, p_sel, [str(c) for c in pre.classes_], deviation\n\n\ndef read_ledger(path: Path) -> tuple[list[dict[str, Any]], bool]:\n    """(entries, corrupt). A corrupt or malformed ledger is set aside, never trusted and never a crash."""\n    if not path.is_file():\n        return [], False\n    try:\n        value = json.loads(path.read_text(encoding="utf-8"))\n        if isinstance(value, dict) and isinstance(value.get("entries"), list):\n            return value["entries"], False\n    except (json.JSONDecodeError, UnicodeDecodeError):\n        pass\n    n = 1\n    while path.with_name(f"ledger.corrupt-{n}.json").exists():\n        n += 1\n    os.replace(path, path.with_name(f"ledger.corrupt-{n}.json"))\n    print(f"The fresh-test ledger was unreadable and was set aside as ledger.corrupt-{n}.json; this scoring is labelled a reproduction, never the first evaluation.")\n    return [], True\n\n\ndef look_marker(run: Run) -> Path:\n    """Marker of a pre-registered look, beside the per-run directories (outside outputs/ and outside any one run)."""\n    return run.root.parent / f"{STEM}_fresh_test_look_{MAGIC_SHA256[:16]}.json"\n\n\ndef next_index(folder: Path, kind: str) -> int:\n    return len(list(folder.glob(f"{STEM}_fresh_test_{kind}_*.json"))) + 1\n\n\ndef stage_fresh_test(run: Run) -> None:\n    import pandas as pd\n\n    masked = fresh_test_masked()\n    print({"fresh_test_mask": f"{MASK_ENV}=1" if masked else f"{MASK_ENV} unset"})\n    if masked:\n        print("Fresh test masked: it is not read, scored or recorded in this run (not a look).")\n        return\n    P = package(run.root)\n    data = run.read_state("data.json", "fresh_test")\n    if data["source"] != MAGIC_SOURCE:\n        print(f"No fresh test for {data[\'source\']!r}: the fresh test belongs to the pre-registered MAGIC sample. Section 7 showed this source\'s {data.get(\'test_role\', \'test\')}.")\n        return\n    cond = run.read_state("condition.json", "fresh_test")\n    fresh = run.out / FRESH_DIR\n    # Reset and recompute: nothing cached by an earlier stage is trusted.\n    differing = settings_differing(cond.get("settings") or {})\n    grid = cond.get("grid") or []\n    checks = {"grid_complete": [g.get("id") for g in grid] == [c["id"] for c in GRID] and all(g.get("complete") for g in grid)}\n    sdpa = ((cond.get("settings") or {}).get("determinism") or {}).get("sdpa_backend")\n    checks["grid_hash_matches"] = checks["grid_complete"] and grid_hash(grid, cond.get("selection", ""), sdpa) == cond.get("grid_hash")\n    active = Path(cond.get("active_checkpoint", ""))\n    checks["selected_model_matches"] = active.is_file() and sha256_file(active) == cond.get("active_checkpoint_sha256")\n    cache = run.root / "inputs" / "magic-gamma.zip"\n    payload = cache.read_bytes() if cache.is_file() else b""\n    checks["fresh_pool_digest"] = hashlib.sha256(payload).hexdigest() == MAGIC_SHA256\n    validation = {"pretrained": cond.get("pretrained_holdout"), "selected_config_candidate": cond.get("candidate_holdout"), "selection": cond.get("selection")}\n    if differing:\n        record = {"test_look": TEST_LOOKS[2], "settings_differing": differing, "preregistered": False, "validation_only": validation, "exploratory_readout": cond.get("exploratory_readout"), "note": "This run differs from the pre-registration, so the fresh test was not read. Only the pre-registered settings on a Tesla T4 score it; see docs/release-verification.md for the recorded evaluation and its reproductions."}\n        out = atomic_write_json(fresh / f"{STEM}_fresh_test_exploratory_{next_index(fresh, \'exploratory\')}.json", record)\n        print({"test_look": TEST_LOOKS[2], "settings_differing": differing, "written": f"outputs/{FRESH_DIR}/{out.name}"})\n        return\n    failed = [k for k, v in checks.items() if not v]\n    if failed:\n        raise RuntimeError(f"fresh test not scored: pre-scoring check(s) failed {failed}; this is not a look. Re-run the notebook from Section 4.")\n    test_bytes = magic_member(payload, "test.csv")  # digest-checked; parsed only below\n    ledger_path = fresh / "ledger.json"\n    entries, corrupt = read_ledger(ledger_path)\n    first_report = fresh / f"{STEM}_fresh_test_report.json"\n    marker = look_marker(run)\n    first = RECORDED_EVALUATION is None and not corrupt and not entries and not first_report.exists() and not marker.exists()\n    label = TEST_LOOKS[0] if first else TEST_LOOKS[1]\n    test = P.read_csv_payload(test_bytes, "test.csv")\n    keys = {}\n    for member in ("train.csv", "val.csv", "test.csv"):\n        frame = pd.read_csv(io.BytesIO(test_bytes if member == "test.csv" else magic_member(payload, member)), dtype=str, keep_default_na=False)\n        keys[member] = frame.drop(columns=["target"]).agg(",".join, axis=1)\n    duplicated = keys["test.csv"].isin(set(keys["train.csv"]) | set(keys["val.csv"])).to_numpy()\n    y, p_pre, p_sel, classes, deviation = fresh_test_probabilities(run, P, cond, test)\n    same = cond["active_mode"] == "pretrained"\n    reading = fresh_test_reading(y, p_pre, p_sel, classes, same_model=same)\n    sensitivity = fresh_test_reading(y, p_pre, p_sel, classes, same_model=same, exclude=duplicated)\n    sens = {"excluded_rows": int(duplicated.sum()), **{k: sensitivity[k] for k in ("n", "primary")}}\n    comparison = compare_with_record(cond["active_mode"], cond["grid_hash"], cond.get("device_name"), printed_reading(reading, sens)) if label == TEST_LOOKS[1] else None\n    report = {"test_look": label, "settings_differing": [], "preregistered": True, "recorded_evaluation": RECORDED_EVALUATION, "checks": checks, "selected": {"mode": cond["active_mode"], "selection": cond["selection"]}, "grid_hash": cond["grid_hash"], "device": cond.get("device_name"), "determinism": (cond.get("settings") or {}).get("determinism"), "fresh_test": {"rows": len(y), "member_sha256": MAGIC_MEMBERS["test.csv"], "row_ids_sha256": MAGIC_ROW_IDS_SHA256["test"], "max_probability_row_sum_deviation": deviation}, **reading, "sensitivity_without_duplicated_rows": sens, "comparison_with_record": comparison, "power": "planning estimates: a single accuracy on 1,500 rows about ±0.017 (Wilson); a paired accuracy difference about ±0.011 at 5 % discordance; a paired log-loss difference about ±0.017 (pessimistic proxy)", "validation": validation, "exploratory_readout": cond.get("exploratory_readout"), "notebook_source": json.loads((run.root / "source.json").read_text(encoding="utf-8")) if (run.root / "source.json").is_file() else {}}\n    out = first_report if first else fresh / f"{STEM}_fresh_test_reproduction_{next_index(fresh, \'reproduction\')}.json"\n    atomic_write_json(out, report)  # the look counts from here: the report is written\n    if not marker.exists():  # outside outputs/: deleting outputs/ cannot make a later look claim to be the first\n        atomic_write_json(marker, {"first_look": out.name, "archive_sha256": MAGIC_SHA256, "grid_hash": cond["grid_hash"]})\n    entries.append({"file": out.name, "test_look": label, "device": cond.get("device_name"), "grid_hash": cond["grid_hash"], "delta": reading["primary"]["delta"]})\n    atomic_write_json(ledger_path, {"entries": entries})\n    print({"test_look": label, "written": f"outputs/{FRESH_DIR}/{out.name}", "selected": cond["active_mode"]})\n    print(rounded({"primary": reading["primary"], "accuracy": reading["accuracy"], "roc_auc": reading["roc_auc"], "sensitivity_without_duplicated_rows": report["sensitivity_without_duplicated_rows"]}))\n    if comparison is not None:\n        print(comparison)\n    print("Scored only under the pre-registered settings; see docs/release-verification.md for the recorded evaluation and its reproductions.")\n\n\ndef stage_report(run: Run) -> None:\n    import time\n\n    import numpy as np\n    from lightgbm import LGBMClassifier\n    from sklearn.ensemble import RandomForestClassifier\n\n    P = package(run.root)\n    data = run.read_state("data.json", "report")\n    enc, train, holdout, test = encoded(run, "report")\n    model, cond = active_model(run, enc, train)\n    classes = list(model.classes_)\n    features, target = enc["features"], enc["target"]\n    X_train, y_train = train[features], train[target]\n    lgbm = LGBMClassifier(random_state=SEED, n_estimators=100, verbose=-1).fit(X_train, y_train)\n    rf = RandomForestClassifier(random_state=SEED, n_estimators=100).fit(X_train, y_train)\n\n    def timed(m, frame, repeats=5):\n        X = frame[features]\n        if repeats > 1:\n            m.predict_proba(X)  # discarded warm-up call\n        latencies = []\n        for _ in range(repeats):\n            t0 = time.perf_counter()\n            raw = np.asarray(m.predict_proba(X), dtype=float)\n            latencies.append((time.perf_counter() - t0) * 1000.0)\n        proba = P.align_probabilities(raw, list(m.classes_), classes)\n        return proba, np.asarray(classes, dtype=object)[np.argmax(proba, axis=1)], float(np.median(latencies))\n\n    y = holdout[target].to_numpy()\n    n = len(holdout)\n    p_tab, pred_tab, lat_tab = timed(model, holdout)\n    p_lgbm, pred_lgbm, lat_lgbm = timed(lgbm, holdout)\n    p_rf, pred_rf, lat_rf = timed(rf, holdout)\n    classical = {"tabicl_" + cond["active_mode"]: P.classification_metrics(y, pred_tab, p_tab, classes), "lightgbm": P.classification_metrics(y, pred_lgbm, p_lgbm, classes), "random_forest": P.classification_metrics(y, pred_rf, p_rf, classes)}\n    errors = {name: int((pred != y).sum()) for name, pred in (("tabicl_" + cond["active_mode"], pred_tab), ("lightgbm", pred_lgbm), ("random_forest", pred_rf))}\n    print({"holdout_rows": n, "one_row_resolution_pct": round(100.0 / n, 2)})\n    for name, values in classical.items():\n        print(f"{name:<20} accuracy={values[\'accuracy\']:.4f} errors={errors[name]} roc_auc={values[\'roc_auc\']:.4f} log_loss={values[\'log_loss\']:.4f}")\n    print({"latency_ms_median_of_5": {"tabicl": round(lat_tab, 2), "lightgbm": round(lat_lgbm, 2), "random_forest": round(lat_rf, 2)}, "rows": n, "device": model.device})\n    best_acc, tied = -1.0, []\n    for w in np.linspace(0.0, 1.0, 101):\n        acc = float(np.mean(np.asarray(classes, dtype=object)[np.argmax(w * p_tab + (1.0 - w) * p_lgbm, axis=1)] == y))\n        if acc > best_acc + 1e-9:\n            best_acc, tied = acc, [float(w)]\n        elif abs(acc - best_acc) <= 1e-9:\n            tied.append(float(w))\n    best_w = tied[len(tied) // 2]  # upper-median tied weight: even ties break toward the foundation model\n    p_blend = best_w * p_tab + (1.0 - best_w) * p_lgbm\n    blend = P.classification_metrics(y, np.asarray(classes, dtype=object)[np.argmax(p_blend, axis=1)], p_blend, classes)\n    blend_record = {"blend_objective": "maximise holdout accuracy", "weight_tabicl": best_w, "tied_weights": len(tied), "tied_weight_range": [min(tied), max(tied)], "blend_holdout": blend, "accuracy_delta_vs_tabicl": round(blend["accuracy"] - classical["tabicl_" + cond["active_mode"]]["accuracy"], 4)}\n    print(rounded(blend_record))\n    if len(tied) > 1:\n        print(f"Holdout accuracy is the same for every blend weight from {min(tied):.2f} to {max(tied):.2f} ({len(tied)} grid points); the selected weight {best_w:.2f} is not uniquely informative.")\n    if test is not None:\n        y_test = test[target].to_numpy()\n        pt, predt, _ = timed(model, test, repeats=1)\n        pl, predl, _ = timed(lgbm, test, repeats=1)\n        pb = best_w * pt + (1.0 - best_w) * pl\n        tab_test = P.classification_metrics(y_test, predt, pt, classes)\n        blend_test = P.classification_metrics(y_test, np.asarray(classes, dtype=object)[np.argmax(pb, axis=1)], pb, classes)\n        print({data.get("test_role", "independent test"): rounded({"tabicl": tab_test, "lightgbm": P.classification_metrics(y_test, predl, pl, classes), "blend": blend_test})})\n        if blend["accuracy"] > classical["tabicl_" + cond["active_mode"]]["accuracy"] and blend_test["accuracy"] <= tab_test["accuracy"]:\n            print("WARNING mixed evidence: the blend improved holdout accuracy but not independent-test accuracy; treat the holdout gain as selection-biased.")\n    else:\n        if data["source"] == MAGIC_SOURCE:\n            print("The MAGIC fresh test is not read here: the fresh_test stage (Section 7b) scores it, only under the pre-registered settings.")\n        print("The validation blend score is selection-biased demonstration evidence (the weight was chosen on validation).")\n    active_holdout = cond["candidate_holdout"] if cond["active_mode"] == "fine-tuned" else cond["pretrained_holdout"]\n    active_test = cond["candidate_test"] if cond["active_mode"] == "fine-tuned" else cond["pretrained_test"]\n    report = P.evaluation_report(active_holdout, baseline=enc["baseline"], independent_test=active_test, n_holdout=n, n_test=None if test is None else len(test), class_labels=classes, target_column=target, selection=cond["selection"], sample_kind=data["sample_kind"], estimation="single seeded stratified split (support/holdout/independent test); no dispersion estimate")\n    report["classical_baselines_holdout"] = {k: v for k, v in classical.items() if not k.startswith("tabicl_")}\n    report["holdout_errors"] = errors\n    report["blend_holdout"] = {"weight_tabicl": best_w, "tied_weight_range": [min(tied), max(tied)], **blend}\n    report["fine_tune"] = cond["fine_tune"]\n    report["fine_tune_grid"] = cond.get("grid")\n    report["grid_hash"] = cond.get("grid_hash")\n    report["selection_tolerance"] = cond.get("selection_tolerance")\n    report["exploratory_readout"] = cond.get("exploratory_readout")\n    report["test_role"] = data.get("test_role")\n    report["fresh_test"] = {"scored_by": "fresh_test stage", "preregistered": cond.get("preregistered"), "settings_differing": cond.get("settings_differing"), "files": f"outputs/{FRESH_DIR}/"}\n    report["interpretation"] = f"One validation row is {1.0 / n:.4f} of accuracy ({n} rows). Log loss is the pre-registered primary metric because it registers changes accuracy cannot; no dispersion over splits was measured."\n    run.write_output(f"{STEM}_evaluation_report.json", report)\n    print(json.dumps({k: report[k] for k in ("verdict", "reason", "decision_rule", "selection", "n_holdout", "n_test", "interpretation")}, indent=2))\n\n\ndef runtime_versions(device: str) -> dict[str, Any]:\n    import importlib.metadata\n    import platform\n\n    import numpy\n    import pandas\n    import sklearn\n    import torch\n\n    return {"python": platform.python_version(), "torch": torch.__version__, "tabicl": importlib.metadata.version("tabicl"), "numpy": numpy.__version__, "pandas": pandas.__version__, "sklearn": sklearn.__version__, "device": device}\n\n\ndef new_rows_frame(run: Run, enc: dict):\n    """The first eight holdout rows (raw features, unlabelled) with their original row id: the default new-data input\n    (TIC-S2)."""\n    raw = run.read_frame("holdout_raw.parquet", "new rows").reset_index(drop=True)\n    rows = raw[enc["features"]].head(NEW_ROWS).copy()\n    rows.insert(0, "row_id", enc["holdout_row_ids"][: len(rows)])\n    return rows\n\n\ndef with_predictions(P, model, rows, enc, classes):\n    X, unseen = P.apply_categorical_encoder(rows[enc["features"]], enc["encoders"])\n    out = rows.copy()\n    proba = model.predict_proba(X)\n    out["prediction"] = model.predict(X)\n    for index, label in enumerate(classes):\n        out[f"probability_{label}"] = proba[:, index]\n    return out, unseen\n\n\ndef stage_predict(run: Run) -> None:\n    P = package(run.root)\n    data = run.read_state("data.json", "predict")\n    enc, train, holdout, test = encoded(run, "predict")\n    model, cond = active_model(run, enc, train)\n    classes = list(model.classes_)\n    new_data = None\n    path = run.options.get("new_data_path") or ""\n    if path:\n        file = Path(path)\n        if not file.is_file():\n            raise FileNotFoundError(f"NEW_DATA_PATH {path!r} does not exist in this runtime.")\n        rows = P.read_inference_csv(file.read_bytes(), enc["features"])\n        manifest = P.validate_inputs(rows, None, feature_columns=enc["features"], names=[file.name])\n        out, unseen = with_predictions(P, model, rows, enc, classes)\n        new_data = {"input": file.name, "rows": len(out), "unseen_categorical_values": unseen, "input_manifest": manifest}\n    else:\n        rows = new_rows_frame(run, enc)\n        out, unseen = with_predictions(P, model, rows, enc, classes)\n        rows.to_csv(run.out / f"{STEM}_new_rows.csv", index=False)\n        print("No new-data file: the first eight holdout rows (already scored in Section 7) are scored instead; their unlabelled copy is written to", f"outputs/{STEM}_new_rows.csv")\n    out.to_csv(run.out / f"{STEM}_predictions.csv", index=False)\n    report = json.loads((run.out / f"{STEM}_evaluation_report.json").read_text(encoding="utf-8"))\n    input_manifest = json.loads((run.out / f"{STEM}_input_manifest.json").read_text(encoding="utf-8"))\n    source = json.loads((run.root / "source.json").read_text(encoding="utf-8")) if (run.root / "source.json").is_file() else {}\n    active_holdout = cond["candidate_holdout"] if cond["active_mode"] == "fine-tuned" else cond["pretrained_holdout"]\n    active_test = cond["candidate_test"] if cond["active_mode"] == "fine-tuned" else cond["pretrained_test"]\n    payload = {\n        "predictions": out.to_dict(orient="records"),\n        "new_data": new_data,\n        "metrics": {"active_mode": cond["active_mode"], "holdout": active_holdout, "independent_test": active_test, "pretrained_holdout": cond["pretrained_holdout"], "candidate_holdout": cond["candidate_holdout"]},\n        "fine_tune": cond["fine_tune"],\n        "majority_class_baseline": enc["baseline"],\n        "evaluation_report": report,\n        "input_manifest": input_manifest,\n        "inference": {"decisionRule": P.DECISION_RULE + " over uncalibrated class probabilities", "threshold": None, "classLabels": [str(c) for c in classes]},\n        "sample": {"kind": data["sample_kind"], "name": data["name"], "source": data["source"], "csv_sha256": data["csv_sha256"], "train_rows": len(train), "holdout_rows": len(holdout), "test_rows": 0 if test is None else len(test), "training_class_shares": enc["class_shares"]},\n        "notebook_source": source,\n        "repository_revision": source.get("revision"),\n        "model_id": P.MODEL_ID,\n        "model_revision": P.MODEL_REVISION,\n        "model_license": P.MODEL_LICENSE,\n        "runtime": runtime_versions(model.device),\n    }\n    run.write_output(f"{STEM}_result.json", payload)\n    if unseen:\n        print("unseen categorical values mapped to the unknown code:", unseen)\n    print(out.round(4).to_string(index=False))\n\n\ndef build_manifest(P, *, enc: dict, classes: list, mode: str, selection: str, metrics: dict | None, n_estimators: int, checkpoint: Path | None, context: Path, runtime: dict, source: dict, checkpoint_facts: tuple[str, int] | None = None) -> dict[str, Any]:\n    """The bundle manifest written by the export stage."""\n    return {\n        "artifactFormat": P.ARTIFACT_FORMAT,\n        "checkpoint": "checkpoints/best.ckpt",\n        "trainingContext": "training_context.parquet",\n        "targetColumn": enc["target"],\n        "featureColumns": enc["features"],\n        "classLabels": [str(c) for c in classes],\n        "baseCheckpoint": P.BASE_CHECKPOINT_NAME,\n        "baseModelRevision": P.MODEL_REVISION,\n        "baseModelSha256": P.BASE_MODEL_SHA256,\n        "tabiclVersion": "2.1.1",\n        "mode": mode,\n        "selectionBasis": selection,\n        "metrics": metrics,\n        "inference": {"class": "TabICLClassifier", "modelPath": "checkpoints/best.ckpt", "nEstimators": n_estimators, "randomState": SEED, "supportManyClasses": True, "allowAutoDownload": False, "decisionRule": P.DECISION_RULE, "categoricalEncoders": enc["encoders"]},\n        "digests": {"checkpointSha256": checkpoint_facts[0] if checkpoint_facts else sha256_file(checkpoint), "trainingContextSha256": sha256_file(context)},\n        "sizes": {"checkpoint": checkpoint_facts[1] if checkpoint_facts else checkpoint.stat().st_size, "trainingContext": context.stat().st_size},\n        "payloadFiles": ["checkpoints/best.ckpt", "training_context.parquet"],\n        "runtime": runtime,\n        "notebookSource": source,\n    }\n\n\nZIP_DATE_TIME = (1980, 1, 1, 0, 0, 0)  # the earliest ZIP timestamp; fixed so equal members give an equal ZIP digest\n\n\ndef deterministic_zip(root: Path, archive: Path) -> Path:\n    """Write every file under ``root`` to ``archive`` in sorted order with a fixed timestamp and mode, atomically.\n\n    ``shutil.make_archive`` stamps members with their file-system mtimes, so two runs that write identical members still\n    print different ZIP digests. Here the digest depends only on the member names and bytes.\n    """\n    import zipfile\n\n    files = sorted(p for p in root.rglob("*") if p.is_file())\n    tmp = archive.with_name(archive.name + ".tmp")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:\n        for path in files:\n            info = zipfile.ZipInfo(path.relative_to(root).as_posix(), date_time=ZIP_DATE_TIME)\n            info.compress_type = zipfile.ZIP_DEFLATED\n            info.external_attr = 0o644 << 16\n            with path.open("rb") as source, zf.open(info, "w", force_zip64=True) as target:\n                shutil.copyfileobj(source, target, 1024 * 1024)\n    os.replace(tmp, archive)\n    return archive\n\n\ndef stage_export(run: Run) -> None:\n    import importlib.metadata\n    import platform\n\n    import numpy\n    import pandas\n    import sklearn\n    import torch\n\n    P = package(run.root)\n    enc, train, holdout, _test = encoded(run, "export")\n    model, cond = active_model(run, enc, train)\n    classes = list(model.classes_)\n    art = run.out / "artifact"\n    shutil.rmtree(art, ignore_errors=True)\n    (art / "checkpoints").mkdir(parents=True)\n    ckpt = art / "checkpoints" / "best.ckpt"\n    shutil.copy2(cond["active_checkpoint"], ckpt)\n    context = art / "training_context.parquet"\n    train[enc["features"] + [enc["target"]]].to_parquet(context, index=False)\n    source = json.loads((run.root / "source.json").read_text(encoding="utf-8")) if (run.root / "source.json").is_file() else {}\n    runtime = {"pythonVersion": platform.python_version(), "torchVersion": torch.__version__, "pandasVersion": pandas.__version__, "pyarrowVersion": importlib.metadata.version("pyarrow"), "scikitLearnVersion": sklearn.__version__, "device": model.device}\n    metrics = {"selectionMetric": cond["eval_metric"], "pretrainedHoldout": cond["pretrained_holdout"], "fineTunedHoldout": cond["candidate_holdout"], "pretrainedIndependentTest": cond["pretrained_test"], "fineTunedIndependentTest": cond["candidate_test"]}\n    manifest = build_manifest(P, enc=enc, classes=classes, mode=cond["active_mode"], selection=cond["selection"], metrics=metrics, n_estimators=model.n_estimators, checkpoint=ckpt, context=context, runtime=runtime, source=source)\n    (art / "artifact.json").write_text(json.dumps(manifest, indent=2) + "\\n", encoding="utf-8")\n    archive = deterministic_zip(art, run.out / f"{STEM}_artifact.zip")\n    smoke = holdout[enc["features"]].iloc[: min(NEW_ROWS, len(holdout))]\n    run.write_state("export_reference.json", {"labels": [str(x) for x in model.predict(smoke)], "proba": numpy.asarray(model.predict_proba(smoke)).tolist(), "classes": [str(c) for c in classes], "device": model.device})\n    digest = sha256_file(archive)\n    result_path = run.out / f"{STEM}_result.json"\n    if result_path.is_file():\n        result = json.loads(result_path.read_text(encoding="utf-8"))\n        result["artifact"] = {"zip": archive.name, "zip_sha256": digest, "manifest": manifest}\n        run.write_output(result_path.name, result)\n    record = {"artifact_zip": str(archive), "zip_sha256": digest, "mode": cond["active_mode"], "checkpoint_bytes": ckpt.stat().st_size, "training_context_sha256": manifest["digests"]["trainingContextSha256"]}\n    print(record)\n    print(f"Trusted digest of this export: whoever loads {archive.name} elsewhere checks it against \'{digest}\' (EXPECTED_ZIP_SHA256 in Section 10). Section 11 itself never loads this run\'s export; it loads the pinned sample artifact.")\n\n\ndef stage_reload(run: Run) -> None:\n    import numpy as np\n    import pandas as pd\n\n    P = package(run.root)\n    enc, _train, holdout, _test = encoded(run, "reload")\n    reference = run.read_state("export_reference.json", "reload")\n    archive = run.out / f"{STEM}_artifact.zip"\n    reload_dir = run.out / "artifact-reload"\n    shutil.rmtree(reload_dir, ignore_errors=True)\n    root = P.safe_extract_zip(archive, reload_dir)\n    served = json.loads((root / "artifact.json").read_text(encoding="utf-8"))\n    members = P.verify_artifact_bundle(root, served)\n    context = pd.read_parquet(members["training_context"])\n    model = P.TabICLClassificationPipeline(P.create_classifier(model_path=members["checkpoint"], allow_auto_download=False, n_estimators=served["inference"]["nEstimators"], random_state=served["inference"]["randomState"], device=reference["device"]), model_path=members["checkpoint"], n_estimators=served["inference"]["nEstimators"], random_state=served["inference"]["randomState"], device=reference["device"], source="artifact")\n    model.fit(context[served["featureColumns"]], context[served["targetColumn"]])\n    if [str(c) for c in model.classes_] != reference["classes"]:\n        raise RuntimeError("Reloaded class labels differ from the exported class order")\n    smoke = holdout[enc["features"]].iloc[: min(NEW_ROWS, len(holdout))]\n    if [str(x) for x in model.predict(smoke)] != reference["labels"]:\n        raise RuntimeError("Prediction mismatch: the classifier rebuilt from the bundle predicts different labels")\n    proba = np.asarray(model.predict_proba(smoke))\n    np.testing.assert_allclose(np.asarray(reference["proba"]), proba, rtol=RELOAD_RTOL, atol=RELOAD_ATOL)\n    print({"reload_dir": str(reload_dir), "process": "fresh (separate from the exporting stage)", "labels_identical": True, "max_abs_probability_difference": float(np.max(np.abs(np.asarray(reference["proba"]) - proba)))})\n    print(f"PASS: bundle extracted safely, digests verified, classifier rebuilt from the bundle alone; labels identical and probabilities equivalent (rtol={RELOAD_RTOL}, atol={RELOAD_ATOL}).")\n\n\ndef stage_activity(run: Run) -> None:\n    """Optional activity: run the pretrained model on the three-class Wine sample and compare the one-row resolution,\n    accuracy and one-vs-rest ROC-AUC with the canonical run; writes only to outputs/activity/."""\n    P = package(run.root)\n    source = run.options.get("data_source", "Sample: Wine")\n    if source not in SAMPLE_SOURCES:\n        raise ValueError(f"ACTIVITY_DATA_SOURCE must be one of {list(SAMPLE_SOURCES)}")\n    before = {p.name: sha256_file(p) for p in sorted(run.out.glob(f"{STEM}_*")) if p.is_file()}\n    train, holdout, _test, name, target = load_sample(P, source)\n    train, _ = P.prepare_classification_table(train, target)\n    features = [c for c in train.columns if c != target]\n    classes = sorted(train[target].unique(), key=str)\n    holdout = P.align_to_schema(holdout, features, target, classes)\n    encoders = P.fit_categorical_encoder(train, features)\n    train_enc, _ = P.apply_categorical_encoder(train, encoders)\n    holdout_enc, _ = P.apply_categorical_encoder(holdout, encoders)\n    enc = {"features": features, "target": target}\n    model = pretrained(run)\n    model.fit(train_enc[features], train_enc[target])\n    metrics = score(P, model, holdout_enc, enc)\n    baseline = P.majority_class_baseline(train_enc[target], holdout_enc[target])\n    canonical = json.loads((run.out / f"{STEM}_evaluation_report.json").read_text(encoding="utf-8"))\n    record = {"activity_sample": name, "classes": [str(c) for c in classes], "holdout_rows": len(holdout_enc), "one_row_resolution": 1.0 / len(holdout_enc), "tabicl_pretrained_holdout": metrics, "majority_class_baseline": baseline, "canonical": {"n_holdout": canonical["n_holdout"], "one_row_resolution": 1.0 / canonical["n_holdout"], "metrics": {m["id"]: m["value"] for m in canonical["metrics"]}}}\n    run.write_output(f"activity/{STEM}_activity_{name}.json", record)\n    after = {p.name: sha256_file(p) for p in sorted(run.out.glob(f"{STEM}_*")) if p.is_file()}\n    if before != after:\n        raise RuntimeError("The activity changed a canonical output; it must write only to outputs/activity/.")\n    print(rounded(record))\n    print({"canonical_outputs_unchanged": True})\n\n\n# --- External-artifact section (§19.1): stages external_artifact → external_reconstruct → external_predict ---------------\n# These stages read only the carried files, the run directory\'s own `inputs/` cache and their own state; they never read\n# a file that the data, validate, condition, report, fresh_test, predict, export or reload stages wrote (XAS5).\n\n\ndef check_trusted_digest(path: Path, expected: str, field: str) -> str:\n    """Refuse a file whose SHA-256 is not ``expected``; nothing is extracted or deserialised before this passes."""\n    expected = str(expected).strip().lower()\n    if not SHA256_HEX.match(expected):\n        raise ValueError(f"{field} must be 64 hexadecimal characters (the digest the producer published with the artifact), got {expected!r}.")\n    observed = sha256_file(path)\n    if observed != expected:\n        raise ValueError(f"Trusted digest mismatch for {path.name}: {field} is {expected}, the file\'s SHA-256 is {observed}. This is not the artifact the digest was issued for; nothing was extracted. Obtain the artifact and its digest from the producer again.")\n    return observed\n\n\ndef fetch_pinned_artifact(cache: Path, url: str, expected: str) -> Path:\n    """The pinned sample artifact: reused from ``cache`` when its digest matches, else downloaded (no authentication),\n    size-capped, and accepted only at the pinned SHA-256. The digest is checked before anything opens the file."""\n    if cache.is_file() and sha256_file(cache) == expected:\n        return cache\n    import urllib.request\n\n    cache.parent.mkdir(parents=True, exist_ok=True)\n    part = cache.with_name(cache.name + ".part")\n    size = 0\n    with urllib.request.urlopen(url, timeout=120) as response, part.open("wb") as out:\n        while chunk := response.read(1024 * 1024):\n            size += len(chunk)\n            if size > SAMPLE_ARTIFACT_MAX_BYTES:\n                out.close()\n                part.unlink(missing_ok=True)\n                raise ValueError(f"The pinned sample artifact at {url} exceeds {SAMPLE_ARTIFACT_MAX_BYTES} bytes: refused before hashing or extraction.")\n            out.write(chunk)\n    try:\n        check_trusted_digest(part, expected, "the pinned SAMPLE_ARTIFACT_SHA256")\n    except ValueError:\n        part.unlink(missing_ok=True)\n        raise\n    os.replace(part, cache)\n    return cache\n\n\ndef check_bundle_identity(P, manifest: dict[str, Any]) -> dict[str, Any]:\n    """The manifest must declare the pinned base checkpoint; a ``pretrained`` bundle\'s checkpoint must BE that base\n    checkpoint (``digests.checkpointSha256 == BASE_MODEL_SHA256``), so a swapped, code-capable checkpoint behind a\n    re-hashed manifest is refused (TICA-m2). A ``fine-tuned`` checkpoint is a derivative bound only by the bundle\'s own\n    digests and the trusted ZIP digest."""\n    declared = (manifest.get("baseCheckpoint"), manifest.get("baseModelRevision"), manifest.get("baseModelSha256"))\n    if declared != (P.BASE_CHECKPOINT_NAME, P.MODEL_REVISION, P.BASE_MODEL_SHA256):\n        raise RuntimeError(f"Artifact was not produced on the pinned base checkpoint carried by this notebook: declares {declared}, expected {(P.BASE_CHECKPOINT_NAME, P.MODEL_REVISION, P.BASE_MODEL_SHA256)}.")\n    mode = manifest.get("mode")\n    checkpoint_sha = (manifest.get("digests") or {}).get("checkpointSha256")\n    if mode == "pretrained":\n        if checkpoint_sha != P.BASE_MODEL_SHA256:\n            raise RuntimeError(f"The artifact declares mode \'pretrained\', so its checkpoint must be the pinned base checkpoint with SHA-256 {P.BASE_MODEL_SHA256}; its manifest records {checkpoint_sha}. Refusing to load a different checkpoint.")\n        return {"mode": mode, "checkpoint_binding": "identical to the pinned base checkpoint (SHA-256 checked)"}\n    if mode == "fine-tuned":\n        return {"mode": mode, "checkpoint_binding": "fine-tuned derivative of the pinned base; bound by the manifest digest and the trusted ZIP digest"}\n    raise RuntimeError(f"Unsupported artifact mode {mode!r}: expected \'pretrained\' or \'fine-tuned\'.")\n\n\ndef check_numeric_features(name: str, rows, features: list[str], encoders: dict[str, Any]):\n    """Columns the producer treated as numeric (no fitted encoder) must hold numbers; refuse naming the column and the\n    first bad values (TICA-m3), instead of handing strings to the model."""\n    import pandas as pd\n\n    out = rows.copy()\n    for column in features:\n        if column in encoders:\n            continue\n        original = out[column]\n        converted = pd.to_numeric(original, errors="coerce")\n        bad = original.notna() & converted.isna()\n        if bad.any():\n            examples = original[bad].astype(str).unique().tolist()[:5]\n            raise ValueError(f"{name}: numeric feature {column!r} has {int(bad.sum())} non-numeric value(s), e.g. {examples}. Fix them (leave missing cells empty).")\n        out[column] = converted\n    return out\n\n\ndef magic_contract(run: Run, P) -> dict[str, Any]:\n    """The data contract of the current default path, read from the pinned MAGIC archive (never ``test.csv``)."""\n    payload = fetch_magic(run.root / "inputs" / "magic-gamma.zip")\n    train = P.read_csv_payload(magic_member(payload, "train.csv"), "train.csv")\n    features = [c for c in train.columns if c != "target"]\n    return {"task": "binary classification", "featureColumns": features, "targetColumn": "target", "classLabels": sorted(str(c) for c in train["target"].unique()), "archive_sha256": MAGIC_SHA256}\n\n\ndef check_data_contract(manifest: dict[str, Any], contract: dict[str, Any]) -> None:\n    """SART10: the artifact must follow the current default path\'s features, target and labels, or it is refused."""\n    for key in ("featureColumns", "targetColumn", "classLabels"):\n        if manifest.get(key) != contract[key]:\n            raise RuntimeError(f"The artifact does not follow this notebook\'s default data contract (SART10): its {key} is {manifest.get(key)!r}, the MAGIC default path\'s is {contract[key]!r}. A sample artifact built from another dataset or an earlier default is refused even when its digest verifies.")\n\n\ndef external_out(run: Run) -> Path:\n    return run.out / EXTERNAL_DIR\n\n\ndef stage_external_artifact(run: Run) -> None:\n    import importlib.metadata\n\n    import pandas as pd\n    import torch\n\n    P = package(run.root)\n    opts = run.options\n    source = opts.get("source", "pinned")\n    shutil.rmtree(external_out(run), ignore_errors=True)  # a failed check leaves no earlier success record behind (VER6)\n    for name in ("external_artifact.json", "external_rows.json", "external_rows.parquet"):\n        (run.state / name).unlink(missing_ok=True)\n    bundle = run.state / "external_bundle"\n    shutil.rmtree(bundle, ignore_errors=True)\n    if source == "pinned":\n        # SART3: provenance, format and integrity mechanism, stated before anything is downloaded or deserialised.\n        print({"artifact": "pinned sample artifact (created outside this run)", "url": SAMPLE_ARTIFACT_URL, "pinned_sha256": SAMPLE_ARTIFACT_SHA256, "producer": SAMPLE_ARTIFACT_PRODUCER, "format": P.ARTIFACT_FORMAT, "integrity": "SHA-256 of the whole ZIP, checked before extraction; then per-member sizes and digests from artifact.json"})\n        zip_path = fetch_pinned_artifact(run.root / "inputs" / "external_artifact.zip", SAMPLE_ARTIFACT_URL, SAMPLE_ARTIFACT_SHA256)\n        trust = {"trusted_digest": "verified (pinned SAMPLE_ARTIFACT_SHA256)", "zip_sha256": SAMPLE_ARTIFACT_SHA256, "url": SAMPLE_ARTIFACT_URL, "producer": SAMPLE_ARTIFACT_PRODUCER}\n    elif source == "own":\n        zip_path = Path(opts.get("zip_path") or "")\n        if not str(zip_path) or not zip_path.is_file():\n            raise FileNotFoundError(f"ARTIFACT_ZIP_PATH {str(zip_path)!r} is not a file in this runtime: point it at the artifact ZIP you were given.")\n        expected = str(opts.get("expected_zip_sha256") or "")\n        if expected:\n            trust = {"trusted_digest": "verified (EXPECTED_ZIP_SHA256)", "zip_sha256": check_trusted_digest(zip_path, expected, "EXPECTED_ZIP_SHA256")}\n        else:\n            trust = {"trusted_digest": "not supplied", "zip_sha256": sha256_file(zip_path)}\n            print("No EXPECTED_ZIP_SHA256 was supplied: the checks below establish internal consistency only, not that this is the artifact you were sent. Ask the producer for the digest it printed when it exported.")\n    else:\n        raise ValueError(f"ARTIFACT_SOURCE must be the pinned sample or your own ZIP, got {source!r}")\n    P.safe_extract_zip(zip_path, bundle)\n    matches = list(bundle.rglob("artifact.json"))\n    if len(matches) != 1:\n        raise ValueError(f"{zip_path.name}: expected exactly one artifact.json in the artifact, found {len(matches)}.")\n    root = matches[0].parent\n    manifest = json.loads((root / "artifact.json").read_text(encoding="utf-8"))\n    compatibility = P.validate_artifact_runtime(manifest, expected_tabicl_version=importlib.metadata.version("tabicl"), expected_torch_version=torch.__version__.split("+")[0])\n    binding = check_bundle_identity(P, manifest)\n    members = P.verify_artifact_bundle(root, manifest)\n    contract = None\n    if source == "pinned":\n        contract = magic_contract(run, P)\n        check_data_contract(manifest, contract)\n    context = pd.read_parquet(members["training_context"])\n    target = manifest["targetColumn"]\n    record = {"source": source, "zip": zip_path.name, **trust, **binding, "root": str(root), "manifest": manifest, "contextRows": len(context), "contextClassCounts": {str(k): int(v) for k, v in context[target].value_counts().sort_index().items()}, "compatibility": compatibility, "data_contract": "MAGIC default path (features, target and class labels match)" if contract else "not checked (your own artifact; your rows must follow its manifest)"}\n    run.write_state("external_artifact.json", record)\n    print({k: record[k] for k in ("source", "zip", "zip_sha256", "trusted_digest", "mode", "checkpoint_binding", "data_contract")})\n    print({"artifactFormat": manifest.get("artifactFormat"), "tabiclVersion": manifest.get("tabiclVersion"), "selection": manifest.get("selectionBasis"), "base": manifest.get("baseCheckpoint"), "base_revision": str(manifest.get("baseModelRevision"))[:12], "producer_runtime": manifest.get("runtime"), "producer_notebook_source": {k: (manifest.get("notebookSource") or {}).get(k) for k in ("repository", "revision", "generator")}})\n    print({"featureColumns": len(manifest["featureColumns"]), "targetColumn": target, "classLabels": manifest.get("classLabels"), "contextRows": len(context), "contextClassCounts": record["contextClassCounts"], "nEstimators": manifest["inference"]["nEstimators"], "categoricalEncoders": sorted(manifest["inference"].get("categoricalEncoders", {}))})\n\n\ndef external_serving(run: Run):\n    """Serving state rebuilt from the validated artifact\'s files alone: its checkpoint, its training context (the\n    fitted support rows) and its fitted categorical encoders. Nothing is refitted from inference data."""\n    import pandas as pd\n    import torch\n\n    P = package(run.root)\n    art = run.read_state("external_artifact.json", "external_reconstruct")\n    manifest = art["manifest"]\n    members = P.verify_artifact_bundle(Path(art["root"]), manifest)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    inference = manifest["inference"]\n    n_estimators, random_state = int(inference["nEstimators"]), inference["randomState"]\n    serving = P.TabICLClassificationPipeline(P.create_classifier(model_path=members["checkpoint"], allow_auto_download=False, n_estimators=n_estimators, random_state=random_state, device=device), model_path=members["checkpoint"], n_estimators=n_estimators, random_state=random_state, device=device, source="artifact")\n    context = pd.read_parquet(members["training_context"])\n    serving.fit(context[manifest["featureColumns"]], context[manifest["targetColumn"]])\n    classes = list(serving.classes_)\n    if manifest.get("classLabels") is not None and [str(c) for c in classes] != list(manifest["classLabels"]):\n        raise RuntimeError("Reconstructed class labels disagree with the validated manifest.")\n    return P, art, manifest, serving, classes\n\n\ndef stage_external_reconstruct(run: Run) -> None:\n    _P, art, manifest, serving, classes = external_serving(run)\n    print({"context_rows": art["contextRows"], "features": len(manifest["featureColumns"]), "classes_": [str(c) for c in classes], "device": serving.device, "source": serving.source, "checkpoint_binding": art["checkpoint_binding"], "categorical_encoders_restored": sorted(manifest["inference"].get("categoricalEncoders", {}))})\n\n\ndef external_sample_rows(run: Run, P, features: list[str]):\n    """The default input: the first eight rows of the pinned MAGIC validation member, label removed. They follow the\n    default data contract (SART10) and are not in the artifact\'s support rows; ``test.csv`` is never read."""\n    payload = fetch_magic(run.root / "inputs" / "magic-gamma.zip")\n    holdout = P.read_csv_payload(magic_member(payload, "val.csv"), "val.csv").reset_index(drop=True)\n    rows = holdout[features].head(EXTERNAL_SAMPLE_ROWS).copy()\n    rows.insert(0, "row_id", list(range(len(rows))))\n    return rows\n\n\ndef stage_external_predict(run: Run) -> None:\n    import io as _io\n    import platform\n\n    import numpy\n    import pandas\n    import sklearn\n\n    P, art, manifest, serving, classes = external_serving(run)\n    features = list(manifest["featureColumns"])\n    encoders = manifest["inference"].get("categoricalEncoders", {})\n    path = str(run.options.get("rows_path") or "")\n    if path:\n        file = Path(path)\n        if not file.is_file():\n            raise FileNotFoundError(f"EXTERNAL_ROWS_PATH {path!r} does not exist in this runtime: upload the file or correct the path.")\n        name, payload, kind = file.name, file.read_bytes(), "BYOD"\n    else:\n        if art["source"] != "pinned":\n            raise ValueError("The MAGIC sample rows follow the pinned sample artifact\'s contract only. With your own artifact, set EXTERNAL_ROWS_PATH to rows that follow its manifest.")\n        sample = external_sample_rows(run, P, features)\n        buffer = _io.StringIO()\n        sample.to_csv(buffer, index=False)\n        name, payload, kind = f"magic-val-first-{len(sample)}-rows.csv", buffer.getvalue().encode("utf-8"), "sample"\n    rows = P.read_inference_csv(payload, features)\n    manifest_in = P.validate_inputs(rows, None, feature_columns=features, names=[name])\n    try:  # a probe that breaks the artifact\'s schema is refused; the finding is recorded, not swallowed\n        P.validate_inputs(rows.drop(columns=[features[0]]), None, feature_columns=features)\n    except ValueError as exc:\n        manifest_in["findings"].append({"input": "missing-column-probe", "verdict": "rejected", "message": str(exc)})\n    else:\n        raise RuntimeError("The missing-column probe was accepted: input validation is not working, so no prediction is written.")\n    rows = check_numeric_features(name, rows, features, encoders)\n    X, unseen = P.apply_categorical_encoder(rows[features], encoders)\n    out = rows.copy()\n    proba = numpy.asarray(serving.predict_proba(X))\n    if not numpy.all(numpy.isfinite(proba)) or float(numpy.max(numpy.abs(proba.sum(axis=1) - 1.0))) > importlib.import_module(f"{PACKAGE}.api").MAX_PROBABILITY_ROW_SUM_DEVIATION:\n        raise RuntimeError("Class probabilities are not finite or do not sum to 1 per row: no prediction is written.")\n    out["prediction"] = serving.predict(X)\n    for index, label in enumerate(classes):\n        out[f"probability_{label}"] = proba[:, index]\n    target = external_out(run)\n    target.mkdir(parents=True, exist_ok=True)\n    out.to_csv(target / f"{STEM}_external_predictions.csv", index=False)\n    source = json.loads((run.root / "source.json").read_text(encoding="utf-8")) if (run.root / "source.json").is_file() else {}\n    payload_out = {\n        "section": "external artifact (NOTEBOOK_SPEC 3.0 §19.1): an artifact made outside this run, validated and served here",\n        "predictions": out.to_dict(orient="records"),\n        "input_manifest": manifest_in,\n        "input": {"filename": name, "kind": kind, "rows": len(out), "extra_columns": manifest_in["inputs"][0]["extra_columns"], "features": features},\n        "artifact": {k: art.get(k) for k in ("source", "zip", "zip_sha256", "url", "producer", "trusted_digest", "mode", "checkpoint_binding", "data_contract", "contextRows", "compatibility")} | {"manifest": manifest},\n        "inference": {"nEstimators": serving.n_estimators, "randomState": serving.random_state, "decisionRule": P.DECISION_RULE + " over uncalibrated class probabilities", "threshold": None, "classLabels": [str(c) for c in classes], "unseenCategoricalValues": unseen},\n        "notebook_source": source,\n        "model_id": P.MODEL_ID,\n        "model_revision": P.MODEL_REVISION,\n        "model_license": P.MODEL_LICENSE,\n        "runtime": {"python": platform.python_version(), "numpy": numpy.__version__, "pandas": pandas.__version__, "sklearn": sklearn.__version__, "device": serving.device},\n    }\n    atomic_write_json(target / f"{STEM}_external_result.json", payload_out)\n    if unseen:\n        print("unseen categorical values mapped to the unknown code:", unseen)\n    print(out.round(4).to_string(index=False))\n    print({"outputs": [f"outputs/{EXTERNAL_DIR}/{STEM}_external_predictions.csv", f"outputs/{EXTERNAL_DIR}/{STEM}_external_result.json"], "input_kind": kind, "artifact_source": art["source"], "zip_sha256": art["zip_sha256"]})\n\n\nSTAGES = {\n    "weights": stage_weights,\n    "data": stage_data,\n    "validate": stage_validate,\n    "condition": stage_condition,\n    "report": stage_report,\n    "predict": stage_predict,\n    "export": stage_export,\n    "reload": stage_reload,\n    "activity": stage_activity,\n    "fresh_test": stage_fresh_test,\n    "external_artifact": stage_external_artifact,\n    "external_reconstruct": stage_external_reconstruct,\n    "external_predict": stage_external_predict,\n}\nREFUSAL_STAGES = {"data", "validate", "predict"}  # input refusals there are shown as one message, not a traceback (DAT20)\n\n\ndef is_refusal(stage: str, options: dict, exc: BaseException) -> bool:\n    """A refused user input (DAT20), not a genuine failure: the kernel shows it as one line and never with a traceback.\n\n    Your own artifact ZIP (``external_artifact`` with ``source=\'own\'``) and your own rows (``external_predict`` with a\n    ``rows_path``, or your own artifact without rows) are user inputs too; the pinned artifact and sample rows are not.\n    """\n    import zipfile\n\n    if not isinstance(exc, ValueError | FileNotFoundError | zipfile.BadZipFile):\n        return False\n    if stage in REFUSAL_STAGES:\n        return True\n    if stage == "external_artifact":\n        return options.get("source") == "own"\n    if stage == "external_predict":\n        return bool(options.get("rows_path")) or "your own artifact" in str(exc)\n    return False\n\n\ndef refusal_name(run: Run, options: dict, message: str) -> str:\n    """The file a refusal names (DAT19): the input the message starts with, else every input the stage was given."""\n    paths = options.get("paths") or {}\n    given = [*paths.values(), options.get("new_data_path"), options.get("zip_path"), options.get("rows_path")]\n    names = [Path(str(value)).name for value in given if value]\n    for name in names:\n        if message.startswith(name + ":") or message.startswith(name + " "):\n            return name\n    if not names and (run.state / "data.json").is_file():\n        names = [str(json.loads((run.state / "data.json").read_text(encoding="utf-8")).get("name") or "")]\n    return ", ".join(name for name in names if name) or "(no file)"\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--outputs", type=Path, required=True)\n    parser.add_argument("--weights", type=Path, required=True)\n    parser.add_argument("--stage", choices=sorted(STAGES), required=True)\n    parser.add_argument("--options", default="{}")\n    args = parser.parse_args(argv)\n    os.environ["CUBLAS_WORKSPACE_CONFIG"] = DETERMINISM["CUBLAS_WORKSPACE_CONFIG"]  # before CUDA initialises\n    run = Run(args.root.resolve(), args.outputs.resolve(), args.weights.resolve(), json.loads(args.options))\n    error_file = run.state / f"{args.stage}.error.json"\n    error_file.unlink(missing_ok=True)\n    try:\n        if args.stage in MODEL_STAGES:\n            with deterministic_kernels():\n                STAGES[args.stage](run)\n        else:\n            STAGES[args.stage](run)\n    except BaseException as exc:  # noqa: BLE001 -- every failure is reported to the kernel with its own message\n        refusal = is_refusal(args.stage, run.options, exc)\n        record = {"type": type(exc).__name__, "message": str(exc), "refusal": refusal}\n        if refusal:  # DAT20: no traceback streams into the cell; the kernel prints the one Refused line\n            record["name"] = refusal_name(run, run.options, str(exc))\n        else:\n            traceback.print_exc()\n        atomic_write_json(error_file, record)\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'requirements.txt': '# This file was autogenerated by uv via the following command:\n#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt\nannotated-doc==0.0.5 \\\n    --hash=sha256:117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101 \\\n    --hash=sha256:c7e58ce09192557605d8bbd92836d7e1d520ac9580096042c0bfd197efacf1bb\n    # via typer\nannotated-types==0.8.0 \\\n    --hash=sha256:13b2beaad985e05e2d6407ee4c4f35590b11f8d693a258a561055cac8f64cab7 \\\n    --hash=sha256:f072f4d804ea359e4eaf198b1af7a8b0943881a87f31bb764f8bf219bb9419e0\n    # via pydantic\nanyio==4.15.1 \\\n    --hash=sha256:6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101 \\\n    --hash=sha256:9f28306018cbd6d329e64a36d58256edff76dd996fe423bc957326e578b82a94\n    # via httpx\ncertifi==2026.7.22 \\\n    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n    # via\n    #   httpcore\n    #   httpx\n    #   requests\ncharset-normalizer==3.5.2 \\\n    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \\\n    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \\\n    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \\\n    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \\\n    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \\\n    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \\\n    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \\\n    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \\\n    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \\\n    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \\\n    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \\\n    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \\\n    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \\\n    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \\\n    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \\\n    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \\\n    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \\\n    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \\\n    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \\\n    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \\\n    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \\\n    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \\\n    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \\\n    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \\\n    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \\\n    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \\\n    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \\\n    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \\\n    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \\\n    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \\\n    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \\\n    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \\\n    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \\\n    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \\\n    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \\\n    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \\\n    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \\\n    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \\\n    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \\\n    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \\\n    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \\\n    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \\\n    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \\\n    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \\\n    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \\\n    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \\\n    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \\\n    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \\\n    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \\\n    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \\\n    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \\\n    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \\\n    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \\\n    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \\\n    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \\\n    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \\\n    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \\\n    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \\\n    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \\\n    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \\\n    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \\\n    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \\\n    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \\\n    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \\\n    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \\\n    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \\\n    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \\\n    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \\\n    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \\\n    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \\\n    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \\\n    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \\\n    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \\\n    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \\\n    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \\\n    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \\\n    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \\\n    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \\\n    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \\\n    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \\\n    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \\\n    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \\\n    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \\\n    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \\\n    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \\\n    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \\\n    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \\\n    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \\\n    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \\\n    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \\\n    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \\\n    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \\\n    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \\\n    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \\\n    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \\\n    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \\\n    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \\\n    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \\\n    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \\\n    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \\\n    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \\\n    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \\\n    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \\\n    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \\\n    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \\\n    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \\\n    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \\\n    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \\\n    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \\\n    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \\\n    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \\\n    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \\\n    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \\\n    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \\\n    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \\\n    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \\\n    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \\\n    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \\\n    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \\\n    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \\\n    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \\\n    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \\\n    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \\\n    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \\\n    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \\\n    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \\\n    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \\\n    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \\\n    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \\\n    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \\\n    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \\\n    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \\\n    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \\\n    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \\\n    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \\\n    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \\\n    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \\\n    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \\\n    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \\\n    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \\\n    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \\\n    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \\\n    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \\\n    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \\\n    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \\\n    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \\\n    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \\\n    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \\\n    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \\\n    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \\\n    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \\\n    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \\\n    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \\\n    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \\\n    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \\\n    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \\\n    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \\\n    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \\\n    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \\\n    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \\\n    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \\\n    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \\\n    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \\\n    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \\\n    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \\\n    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \\\n    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \\\n    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \\\n    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \\\n    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \\\n    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \\\n    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f\n    # via requests\nclick==8.5.0 \\\n    --hash=sha256:255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360 \\\n    --hash=sha256:ba0d2089de75ea0310e2dde03160e6ca10009947fb95a182f9b54021bb272e34\n    # via\n    #   huggingface-hub\n    #   wandb\ncloudpickle==3.1.2 \\\n    --hash=sha256:7fda9eb655c9c230dab534f1983763de5835249750e85fbcef43aaa30a9a2414 \\\n    --hash=sha256:9acb47f6afd73f60dc1df93bb801b472f05ff42fa6c84167d25cb206be1fbf4a\n    # via joblib\ncuda-bindings==13.4.3 \\\n    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n    # via torch\ncuda-pathfinder==1.8.3 \\\n    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f\n    # via cuda-bindings\ncuda-toolkit==13.0.3.0 \\\n    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f\n    # via torch\neinops==0.8.2 \\\n    --hash=sha256:54058201ac7087911181bfec4af6091bb59380360f069276601256a76af08193 \\\n    --hash=sha256:609da665570e5e265e27283aab09e7f279ade90c4f01bcfca111f3d3e13f2827\n    # via tabicl\nfilelock==4.0.11 \\\n    --hash=sha256:1cbdc4ace7fb11d2f6d073f5c3d91537a23b3d58d2bf2fdffc12c6d717a589c6 \\\n    --hash=sha256:9dc453a1edd25626a6fc97c2cb1bd9f7e46345843d40b3181c071761fb9a5f7a\n    # via\n    #   huggingface-hub\n    #   torch\nfsspec==2026.9.0 \\\n    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n    # via\n    #   huggingface-hub\n    #   torch\ngoogleapis-common-protos==1.75.5 \\\n    --hash=sha256:c7a866fc34ed29a3b10af627a4b9b1dc2433313ca6e959f0ae4feb132047ed72 \\\n    --hash=sha256:d7285525c23039db98f2463e6d5a4f9b958b94d497f03a844ece3259c4e72d5d\n    # via opentelemetry-exporter-otlp-proto-http\nh11==0.16.0 \\\n    --hash=sha256:4e35b956cf45792e4caa5885e69fba00bdbc6ffafbfa020300e549b208ee5ff1 \\\n    --hash=sha256:63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86\n    # via httpcore\nhf-xet==1.6.0 \\\n    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n    # via huggingface-hub\nhttpcore==1.0.9 \\\n    --hash=sha256:2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55 \\\n    --hash=sha256:6e34463af53fd2ab5d807f399a9b45ea31c3dfa2276f15a2c3f00afff6e176e8\n    # via httpx\nhttpx==0.28.1 \\\n    --hash=sha256:75e98c5f16b0f35b567856f597f06ff2270a374470a5c2392242528e3e3e42fc \\\n    --hash=sha256:d909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad\n    # via huggingface-hub\nhuggingface-hub==1.30.0 \\\n    --hash=sha256:96ae0a8e99a234374a6fe43e989ebd21c04640b91ab2927e7e5773ba1131ca59 \\\n    --hash=sha256:e6a6120bc8c8e2723d03648434ee247088cceb55ba7067e7d34d692cad5fdb57\n    # via\n    #   tabicl-classifier-pipeline (pyproject.toml)\n    #   tabicl\n    #   tokenizers\n    #   transformers\nidna==3.20 \\\n    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n    # via\n    #   anyio\n    #   httpx\n    #   requests\njinja2==3.1.6 \\\n    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n    # via torch\njoblib==1.6.0 \\\n    --hash=sha256:2ccc96785b12046c08fd6d55839c12857831b54a3c1673ffadd2f04bfc4eda03 \\\n    --hash=sha256:3dbbf9f6e4b592a2357b854608e980fe6390d131d7a82f011a377ef2ebef7aba\n    # via scikit-learn\nlightgbm==4.7.0 \\\n    --hash=sha256:129535462686f274df179133643118c5c5c5667167fe6c3a28d955f0b3c8e868 \\\n    --hash=sha256:d23e922acd891e77212e4d0fbcee9ba973c96dee479491341d05ba595357ebb7 \\\n    --hash=sha256:d4529acec5c6fefe4768302a529707d0ead90f6a6f42df694b856212e09695b8 \\\n    --hash=sha256:dfc1cfe8e760387be1e7ba7a214688be21fdff96e4ed9749188f83e1877c2477 \\\n    --hash=sha256:f42d1e5b32b6f170e606d7c689c6165671da98d7bf37f1addec2623efc8740c9 \\\n    --hash=sha256:f8e20f682c9aabd000bcf4a7ed8aa6f473c1adfecccae34ec24e823d156f4af0\n    # via tabicl-classifier-pipeline (pyproject.toml)\nmarkdown-it-py==4.2.0 \\\n    --hash=sha256:04a21681d6fbb623de53f6f364d352309d4094dd4194040a10fd51833e418d49 \\\n    --hash=sha256:9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a\n    # via rich\nmarkupsafe==3.0.4 \\\n    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \\\n    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \\\n    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \\\n    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \\\n    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \\\n    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \\\n    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \\\n    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \\\n    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \\\n    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \\\n    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \\\n    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \\\n    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \\\n    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \\\n    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \\\n    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \\\n    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \\\n    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \\\n    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \\\n    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \\\n    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \\\n    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \\\n    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \\\n    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \\\n    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \\\n    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \\\n    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \\\n    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \\\n    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \\\n    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \\\n    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \\\n    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \\\n    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \\\n    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \\\n    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \\\n    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \\\n    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \\\n    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \\\n    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \\\n    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \\\n    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \\\n    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \\\n    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \\\n    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \\\n    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \\\n    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \\\n    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \\\n    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \\\n    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \\\n    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \\\n    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \\\n    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \\\n    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \\\n    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \\\n    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \\\n    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \\\n    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \\\n    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \\\n    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \\\n    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \\\n    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \\\n    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \\\n    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \\\n    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \\\n    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \\\n    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \\\n    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \\\n    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \\\n    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \\\n    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \\\n    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \\\n    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \\\n    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \\\n    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \\\n    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \\\n    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \\\n    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \\\n    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \\\n    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \\\n    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \\\n    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \\\n    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \\\n    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \\\n    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \\\n    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \\\n    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \\\n    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \\\n    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \\\n    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \\\n    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \\\n    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \\\n    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \\\n    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \\\n    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \\\n    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \\\n    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \\\n    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \\\n    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \\\n    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \\\n    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \\\n    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \\\n    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \\\n    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \\\n    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \\\n    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \\\n    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \\\n    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \\\n    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \\\n    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \\\n    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \\\n    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \\\n    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \\\n    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \\\n    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \\\n    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \\\n    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \\\n    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \\\n    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \\\n    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \\\n    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \\\n    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \\\n    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \\\n    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \\\n    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \\\n    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \\\n    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \\\n    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \\\n    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \\\n    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \\\n    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \\\n    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \\\n    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \\\n    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \\\n    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \\\n    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \\\n    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \\\n    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \\\n    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \\\n    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \\\n    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \\\n    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \\\n    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \\\n    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \\\n    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \\\n    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \\\n    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \\\n    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \\\n    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \\\n    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \\\n    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \\\n    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3\n    # via jinja2\nmdurl==0.1.2 \\\n    --hash=sha256:84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8 \\\n    --hash=sha256:bb413d29f5eea38f31dd4754dd7377d4465116fb207585f97bf925588687c1ba\n    # via markdown-it-py\nmpmath==1.3.0 \\\n    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n    # via sympy\nnarwhals==2.27.0 \\\n    --hash=sha256:27a1b68b94c36bbe60ecc63fb9b37674a85a98bee3f45d179cbdc0a872bc9b8d \\\n    --hash=sha256:9d5661f93a10cfb2b2cc2b3e6e866541dea1b05706a3bcd22f633d70cfae3067\n    # via\n    #   lightgbm\n    #   scikit-learn\nnetworkx==3.7 \\\n    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n    # via torch\nnumpy==2.5.2 \\\n    --hash=sha256:0090ccdd57ec2703e9b49d0bf554767370581c1dd0a6b2bb2b2d9def317d042a \\\n    --hash=sha256:078f9b027b478c9379b9677babbf0f8b8f1ecfada27636d7b9a93990c638739f \\\n    --hash=sha256:07d4e89f3a9ab0a9ba24264ccdb642b3dd951b2281e8883a5481a4aa79cc31a7 \\\n    --hash=sha256:0a4035ae1129ff8777f08bfbd44f1e5d8e9c049ce0c2dd78fc0d92c13e7251c0 \\\n    --hash=sha256:0aadf13b60048d501e05fa699efaf7734e2494f3498a4c2a5521d822640324f3 \\\n    --hash=sha256:14e373cfc6387177e8409dac3c7159be8eb05cd77096cd7c950268b86f62831c \\\n    --hash=sha256:1ab3d4a901f844ea836c3e80bf463c6a27d7f3c14e8e292fcf28d348b25b9bce \\\n    --hash=sha256:24b9dc2e3d84aa58523798805194e23e736f3f6ce2d1a5b92583ae734e6dbda8 \\\n    --hash=sha256:27650bb0e7140fa3d37b9923b4803645e0b125d190f326eecfd3f4dad8e8ade1 \\\n    --hash=sha256:28ac63476ec7651484215ee7fa15a1f78b57c14621f01e392afe17b9a1390ce4 \\\n    --hash=sha256:29b86ff8a6cc556b47ec6b64b194815cc80e6bf5eedcc6cddfd65318cb0b4eee \\\n    --hash=sha256:29d81e97f668489cba8ebfd796b9bdd453525d35dd9e162e2daec94bf3fc7740 \\\n    --hash=sha256:2cc779226e476d1e1f08c74068c419e60f41a9e0e069c92f6671d31d5c985e98 \\\n    --hash=sha256:2ffa7bacab3e2ee1b19ed31766bb60bb380b68c23f051e199c5cc598afd68710 \\\n    --hash=sha256:318b9a4c845dbea06708a29c84ee429cc3065048db34cdb799047643492050ee \\\n    --hash=sha256:34c319e2963be042673fb46570501b2f06c41924e17e3563d58646b4380dfb68 \\\n    --hash=sha256:3a2f061cebd9e3d23bdcfaaded5e2293a4c6a5b60fa42df85d410a725ce621bf \\\n    --hash=sha256:3cdec01fa790a186d430433fdd4d4ffb70eed6f0eeb4bf05c8dbe2dce0a9bcb8 \\\n    --hash=sha256:3e4c367352d3747784248a227fbec218e193b56f7e6692e3b64fc805478ecfdf \\\n    --hash=sha256:40f4d451aed46a8046a1aae41c4e55fb3612273df9c502480135e1501576a34b \\\n    --hash=sha256:44ef9675d908e65f9953063837c3277730f3f4437615a4cdab67b366cabaf884 \\\n    --hash=sha256:4bbd96c833ecc8cc069ce518078fc8c60cb9cbfb0fea5b7a803ad65035596d03 \\\n    --hash=sha256:4ec954036759bcee3aa484f8603bd9c14f3e776293b85578b8734c2d72777c69 \\\n    --hash=sha256:4f9744f9fbdcea0bc552e8f19e1f141f811a3f9bc2be2cc6e86d982cab23e3f4 \\\n    --hash=sha256:50a68f4bacd8a2b33d8da3d2269d0d78500f86ea582e4786dc10f5ef2c2c6842 \\\n    --hash=sha256:50e500dc868e9313530ce12ba470fe50ff3afe3d62993ed6eff652dacd555b65 \\\n    --hash=sha256:52c808f96484f5571a5cc863775ce50247c17dfb3b0361f8ed6b4b0456f80080 \\\n    --hash=sha256:5f8e00be2ec6f45f4e8a41a527f68d44a7d96fee92a650e4d8b1326f77f61e6e \\\n    --hash=sha256:60e902ac295855348a5ca2ea4c89108989a9f5fddfad3dfc0a8f36b10358567e \\\n    --hash=sha256:65f188481f1669e26f62b701e8205d19e460fa4a9b52a1414ba382330e4a3414 \\\n    --hash=sha256:6950c4b7dd562453090548ba7f5da7e59f57f85663f15d5dcc60e249192f7e59 \\\n    --hash=sha256:6a9bb119fb8dd21ba30b3f0e555b7e2b081bd9883af21ec9c1c633d161cda3a8 \\\n    --hash=sha256:6b588cc8f902d6bff201c19fd00c43ab8545671e3554d014e12e14139e5e8617 \\\n    --hash=sha256:6df895598c0edcb41030126c89e0f353b07d93238116143b7405e937359736c4 \\\n    --hash=sha256:6e8172ddfcf5cf74b811d372b570b83c60bd2de87a6fbfbebdadb4a9bd9c6cbb \\\n    --hash=sha256:7354826bc6f8f69402e9b7fe28d15fcd34feebd74f856f111585c5b0c9fb0251 \\\n    --hash=sha256:7587f53dfbd5edc0f7b87c6217b4c6d2d1f2ef9c3da70bc1315e7db5f8d7ec9d \\\n    --hash=sha256:77843ca236b777e67f8d6b3660ea116e499612703a0ecd7093f316201eb9d8e2 \\\n    --hash=sha256:7999d4ddb0c4025018373fd787510d46e04c769467af22869707b3c1cfd459ab \\\n    --hash=sha256:85aaccb24182c25df891ad0ec333585967e115269d5f1b17f2c9ae005bc96657 \\\n    --hash=sha256:8e4cb9a754c8a0c62eaa88273a5fba3391f4a610d1dee893c0755da31c083f15 \\\n    --hash=sha256:8ee9c4eeb8454b3660a8b53493563c3e121c2fc94fbd72b848ef814ed7b676a9 \\\n    --hash=sha256:9a0731745a72a184490a582fb4af2533512bd071ace67785b5fdffc0ae58dce8 \\\n    --hash=sha256:9e9413326d726c2545bfa65d2c0876871e8d8386e77f992c1d426e180bbd4323 \\\n    --hash=sha256:a610dc7e3c52edd39c2bc2375ff9c3fd59cb3ad00e4472d36f83bc1457145788 \\\n    --hash=sha256:a839318485284a6fb31be4f8f2c91c8f2cb22f4543c4a8903f12b0671ffe07cc \\\n    --hash=sha256:afb3f0632d6b2e3ba04dbce8d1e48d321b369138b73830b5ca371a0e8d479d56 \\\n    --hash=sha256:b879fb674276e331513fb136b78dbc6bd3c848309e0d841cfd63be3896c4cfc1 \\\n    --hash=sha256:b9727f472d2f3888053b8a75ab0cb94745a9de224bb5846dbadc0092101bc71d \\\n    --hash=sha256:ba0a474801b8dc67b66bf465548abc90e82b44d2611b5770f33008dcabffe8ec \\\n    --hash=sha256:bd68ece1553d2023c09a4226d9e41c586ad2d20594d1a456186c33513d2cb3f2 \\\n    --hash=sha256:c081cbe16ba1ab53078e5ff29013621e33c509eedab055775d956427712c236e \\\n    --hash=sha256:c1f017dc0875c9209d219f97feceb7d54c2661bb243deb4114478e1295808af7 \\\n    --hash=sha256:cebc2d6dbb605a7703d59751dea4bd6b0ab127a5a4338a6f432df1936fef8b26 \\\n    --hash=sha256:cf7de32f486e4ac9e2d93b810f9e9ac72a728dd46a32a0bb403222f27f653514 \\\n    --hash=sha256:d482d171c406ae88c5b19cad3b6a1c4c5209f886ab74bc44c2c865c23f52d860 \\\n    --hash=sha256:d6a48072864e3324e194a8fbb3c657bcc5b5c869dbc64c9537b1d5c862572c0a \\\n    --hash=sha256:d787cf769c3baeb5f6235e778edb52c08dfa923789b5958f28e6450f96107cb1 \\\n    --hash=sha256:dc649493697006bc90614a5f0bbc8cb3cb1866715c474e473694968d7e6b99ab \\\n    --hash=sha256:ddf47472af2e4280d79bac82304f5e80150211f1b9e614b760061d5fdfbb6eba \\\n    --hash=sha256:e5651f3f87add730ee6608d915009e19c911fba0cb000c7e3ea994b7d768eb12 \\\n    --hash=sha256:e79aba74ffaf5f78a050d777c184cddf8fdffabab38acf5f3ef1fecbc17895d6 \\\n    --hash=sha256:eaa088384c46f519dacb93b7ec483a6d6b19a4a2085ae4f25ab9b1c43d387d1e \\\n    --hash=sha256:eaca7ff36f0f52e2111ec71f169d8fd3e889e7ddc0d2592e0d703fd8d3ce8fac \\\n    --hash=sha256:f06571a052127dc1b4e8b83029b4d1b20daa2b64a31cdd181fc6bc774e9000eb \\\n    --hash=sha256:fd0d703772bba096843785bd38371e31bb4a0c1151497ad5739d182114a73f7f\n    # via\n    #   tabicl-classifier-pipeline (pyproject.toml)\n    #   lightgbm\n    #   pandas\n    #   scikit-learn\n    #   scipy\n    #   tabicl\n    #   torchvision\n    #   transformers\nnvidia-cublas==13.1.1.3 \\\n    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \\\n    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \\\n    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5\n    # via\n    #   cuda-toolkit\n    #   nvidia-cudnn-cu13\n    #   nvidia-cusolver\nnvidia-cuda-cupti==13.0.85 \\\n    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n    # via cuda-toolkit\nnvidia-cuda-nvrtc==13.0.88 \\\n    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n    # via\n    #   cuda-toolkit\n    #   nvidia-cublas\nnvidia-cuda-runtime==13.0.96 \\\n    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n    # via cuda-toolkit\nnvidia-cudnn-cu13==9.24.0.43 \\\n    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \\\n    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \\\n    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f\n    # via torch\nnvidia-cufft==12.0.0.61 \\\n    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n    # via cuda-toolkit\nnvidia-cufile==1.15.1.6 \\\n    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n    # via cuda-toolkit\nnvidia-curand==10.4.0.35 \\\n    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n    # via cuda-toolkit\nnvidia-cusolver==12.0.4.66 \\\n    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n    # via cuda-toolkit\nnvidia-cusparse==12.6.3.3 \\\n    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n    # via\n    #   cuda-toolkit\n    #   nvidia-cusolver\nnvidia-cusparselt-cu13==0.8.1 \\\n    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \\\n    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \\\n    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215\n    # via torch\nnvidia-nccl-cu13==2.30.7 \\\n    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \\\n    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50\n    # via torch\nnvidia-nvjitlink==13.4.92 \\\n    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \\\n    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \\\n    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \\\n    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323\n    # via\n    #   cuda-toolkit\n    #   nvidia-cufft\n    #   nvidia-cusolver\n    #   nvidia-cusparse\nnvidia-nvshmem-cu13==3.4.5 \\\n    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n    # via torch\nnvidia-nvtx==13.0.85 \\\n    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n    # via cuda-toolkit\nopentelemetry-api==1.45.0 \\\n    --hash=sha256:711ede81773c8025c2c03dac0450bc89f3d30aea6eabcc815c570d4e35a963f7 \\\n    --hash=sha256:80e068aba7cd56c8b58512d6a36f8d25cb1dfaa0c0a4cc1c938ccf9f362d9cb3\n    # via\n    #   opentelemetry-exporter-http-transport\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\n    #   wandb\nopentelemetry-exporter-http-transport==0.66b0 \\\n    --hash=sha256:2c229b6593eaa22c86d9b8a15843dc23b406dbda00fb138339189aab07923b4e \\\n    --hash=sha256:c82689928a11505a2a0a26a04afe0ce06d270d3db984bc16c1a80ffa902db00e\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-common==0.66b0 \\\n    --hash=sha256:35d24c867310f4713a9738b0202b6bade77955418fccb9e2f7eea01b76263916 \\\n    --hash=sha256:362268ec6aa705e183776ff938539df1e8ce45bc5509d242538b1d40c26fe6a6\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-common==1.45.0 \\\n    --hash=sha256:36495115a0c6a7aa946cfda9d59b6ed4e917b6ab0f75cdaf66bc1b176ec1be1f \\\n    --hash=sha256:7e1410ae6d3ed301f7a74bec96467d519d3f37e52f9d95b309ae125e5a1af863\n    # via opentelemetry-exporter-otlp-proto-http\nopentelemetry-exporter-otlp-proto-http==1.45.0 \\\n    --hash=sha256:2f35496d96809f946f41b8805e6b93aec6c9b71b5fd759b75b6af4c084d992ae \\\n    --hash=sha256:74385f99266dafdefff41f3da4b76f6dba79d237a9d4487e6ff1ee653c74a0a3\n    # via wandb\nopentelemetry-proto==1.45.0 \\\n    --hash=sha256:96ee414f24bc3f61ea8e17dc56b4348d4049d73db3eb17c6b3edf75b5b403300 \\\n    --hash=sha256:9731566359d7b8e1ee1e149d8e4ad6865bab965e657ad3387ec2784d16927666\n    # via\n    #   opentelemetry-exporter-otlp-proto-common\n    #   opentelemetry-exporter-otlp-proto-http\nopentelemetry-sdk==1.45.0 \\\n    --hash=sha256:20caa5130505e386c67c3da1c76e446c842698ced54c76c6148679539aa97972 \\\n    --hash=sha256:5dc634c946546f61b757c5b1781f9fd1a10e96ffaf7357c797e508fe9c57e75e\n    # via\n    #   opentelemetry-exporter-otlp-common\n    #   opentelemetry-exporter-otlp-proto-http\n    #   wandb\nopentelemetry-semantic-conventions==0.66b0 \\\n    --hash=sha256:175b19dd98c4473f4f43a2b1df59186fd7b4a48cd3f77cbe03438b6d6fda230a \\\n    --hash=sha256:97a77dce484c54861e7eeff7651fd8a806dd3c30e501dc316730215ec36890e6\n    # via opentelemetry-sdk\npackaging==26.3 \\\n    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n    # via\n    #   huggingface-hub\n    #   transformers\n    #   wandb\npandas==3.0.5 \\\n    --hash=sha256:08d24fe11a17dc33bd6e937dc9c665f9cba08fbdc9f657f405713515febe300d \\\n    --hash=sha256:0d298e951f23016ce4699951d044ae6418dbc91bf68cefca0f77666fcbb4e5c6 \\\n    --hash=sha256:0fac0010c75e4efb6b99e249c183a8993ce0dc95c240f9b120a5e67c727b7928 \\\n    --hash=sha256:1c10461f6eeb35d8f05b6184c65c8b9991663b66c46b1d559b682cb34ae7c6ea \\\n    --hash=sha256:25ff585b972a18ef1fe9ffa3ac6544d9950508aa76832e5147640b6022821e49 \\\n    --hash=sha256:2946e77e4a53cd248cbde631a12f0e51c8324ce354c3eba4d20147c1ad6f4282 \\\n    --hash=sha256:2a29c53d85ea98c5e792c59ef82ee9fbe6ca902c0d0adb6b23f45ef894cd7bf6 \\\n    --hash=sha256:2c0cf1dd9b55a22d105fc46c1b489af3bd42264fcba7c66297bf47a9a1d9c78a \\\n    --hash=sha256:2f264fc46911cc8131a7322a16199bbf8e353d27c10bb211f5bd0c814324dc36 \\\n    --hash=sha256:303da736987d481074ca720ada325f8bd80c64ebc2d45ed79b29df3aaa4a26ca \\\n    --hash=sha256:3b2801bbb049d0136f6c213eae02b5fca969384fc2064dd728d8620552aa49da \\\n    --hash=sha256:3c5015fd1730fbf883647e88068176c839c102cea883ba1769a6f4593bfc1f8c \\\n    --hash=sha256:3c5ed2e7c06e91d340dfd091d7934f9bc82e4a36b95f647f090b9d1c9ac649da \\\n    --hash=sha256:4b11c36e218331d0387cbe3a0a5f75162357a1d92d57b2b08a336ff94b19b2be \\\n    --hash=sha256:5183427f5a8156d480f30333777bc978be93650a49a7c01db26adffe95b31e85 \\\n    --hash=sha256:53730687fcd161883b24e10411c06d6a4c0f2275d2faf3bb2bc25deb4ba8007c \\\n    --hash=sha256:66266d3442a5e8b3c90274c2b8b230bee42dd1c286bc822cc2f9f2c7e12b883e \\\n    --hash=sha256:679f4e85b30ddb1515458ab1e788d3e260eae369b1f78da7a3aa4cac8ebf4a2a \\\n    --hash=sha256:71ecc8fb7ed1a7aa4392316b5309a6347e8e7f832f38fd897846b3a1457a9298 \\\n    --hash=sha256:73fa87b08a7ef706f8aafda39ddaccf2a99047bea62d8c88a0361bcafb2237bc \\\n    --hash=sha256:80a611068e8a3ac23f7398c6c14eb46dc974e5cc9997f653e2dcfd1da74edd41 \\\n    --hash=sha256:960d3ebcf249f75206899fcd2c6de53f736b7265759ced0d3e559df0b8b709b0 \\\n    --hash=sha256:9e94c2c5ca43bd3ca32bf64d32308887b65e5f9bfd8023ea52755107a999f93b \\\n    --hash=sha256:a5ad3b02ed6bc7d7ae9b70804b2c6aa31827489d150f8e623ce82491b82085d7 \\\n    --hash=sha256:b1261758dfb6cf12c3cff8300e21cefad30e7ec709abb4c24ac7318e6a52462a \\\n    --hash=sha256:b173f5951ff6b8b0ec7675e20dff3c97b7e7a57dfcce387c2d7c5afe87cb7899 \\\n    --hash=sha256:b2acb4650527eec6822c3dadb2b771277b65e7dae7a267d4bccf65fd1bb3fbce \\\n    --hash=sha256:b58b1b39d46a5862e3fb18f50d1a201398619d16a0f9f73f57eea5583cf0e63c \\\n    --hash=sha256:b86765f268b56f7e665b93bce9d5df69dee7f99e595cf8fb839483ab315942a3 \\\n    --hash=sha256:c1c05a767fe8e5b4fe9e1c29806829c582052eaedb9120a3da83ba3f69e24a5b \\\n    --hash=sha256:c2e26bb46934b8a2ca0c3de1d3d606fc5f6746584791b2db264d58cf370e08dc \\\n    --hash=sha256:c597ecf5616b5c420372c1d4d4c00dbbfba7398bea857dcc984347e1ea48417b \\\n    --hash=sha256:cce3a9d11d2b1f82c69a27ec1f4948a170e2c403c4bbfa8cca62e3fdebe2ef3a \\\n    --hash=sha256:cd8f7c6dc98527058ee6264219343f5392240a6f1bfa654fc5d79023020d0c92 \\\n    --hash=sha256:cf52e1f61d229496da17dc7ab54acdee627357e7008fd4fecba3d0ba2937fa58 \\\n    --hash=sha256:d373ce03ffd84010ed9839fa73672a9c8256990532e158440c0085db7d914b34 \\\n    --hash=sha256:db172144bb56422bd157812f3b021eacc255451470b31e2c633c349490a1cfee \\\n    --hash=sha256:dca3734d6ab7c906e6730f0788b0a1dbb9f2467731f9711f77995c8e9d62d712 \\\n    --hash=sha256:e2759e890db96dfcffdbd9b86c3c2cb6afaf58def482820317e06163ec1066cd \\\n    --hash=sha256:e819dd5f62966b481a8cb649d3299ebd886a1ea91ed5a99bf7ce77c98d18ab94 \\\n    --hash=sha256:ef01af4d8dc6cd2c8d6c7736f149574ef93fe043811eeb5e445f2647154b5040 \\\n    --hash=sha256:fa290c16964d4963fbfbc358928239cf3bd755b20e988ce944877def2f44471d\n    # via tabicl-classifier-pipeline (pyproject.toml)\npillow==12.3.0 \\\n    --hash=sha256:00808c5e14ef63ac5161091d242999076604ff74b883423a11e5d7bbb38bf756 \\\n    --hash=sha256:04f01d28a6aaff387bf842a13be313df23ba0597a44f1a976c9feb3c6ff4711a \\\n    --hash=sha256:06ff022112bc9cbf83b60f8e028d94ad87b60621706487e65f673de61610ab59 \\\n    --hash=sha256:0740a512dc522224c77d9aa5a8d70d8b7d73fb91f2c21125d8d025d3b8990e45 \\\n    --hash=sha256:0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3 \\\n    --hash=sha256:0dd2064cbc55aaec028ef5fbb60fa47bb6c3e7918e07ff17935284b227a9d2df \\\n    --hash=sha256:0feb2e9d6ad6c9e3c06effe9d00f3f1e618a6643273576b016f591e9315a7139 \\\n    --hash=sha256:10e41f0fbf1eec8cfd234b8fe17a4caac7c9d0db4c204d3c173a8f9f6ef3232b \\\n    --hash=sha256:1182d52bc2d5e5d7d0949503aa7e36d12f42205dc287e4883f407b1988820d39 \\\n    --hash=sha256:164b31cd1a0490ab6efae01aa5df49da7061be0af1b30e035b6e9a1bfe34ee6e \\\n    --hash=sha256:1657923d2d45afb66526e5b933e5b3052e6bdea196c90d3abb2424e18c77dae8 \\\n    --hash=sha256:186941b6aef820ad110fb01fb06eb925374dc3a21b17e37ec9a53b250c6fe2d1 \\\n    --hash=sha256:1cca606cd25738df4ed873d5ad46bbdb3d83b5cbca291f6b4ff13a4df6b0bbe8 \\\n    --hash=sha256:21900ce7ba264168cd50defae43cd75d25c833ad4ad6e73ffc5596d12e25ac89 \\\n    --hash=sha256:236ff70b9312fb68943c703aa842ca6a758abfa45ac187a5e7c1452e96ef72b5 \\\n    --hash=sha256:23aceaa007d6172b02c277f0cd359c79492bbb14f7072b4ede9fbcaf20648130 \\\n    --hash=sha256:23d27a3e0307ec2244cc51e7287b919aa68d097504ebe19df4e76a98a3eea5bd \\\n    --hash=sha256:24870b09b224f7ae3c39ed07d10e819d06f8720bc551847b1d623832b5b0e28d \\\n    --hash=sha256:251bf95b67017e27b13d82f5b326234ca62d70f9cf4c2b9032de2358a3b12c7b \\\n    --hash=sha256:25b9b82bb22e6e2b3cd07b39c68b7b862001226cb3dff7130d1cb914121b39ed \\\n    --hash=sha256:28ce87c5ab450a9dd970b52e5aca5fe63ed432d18a2eaddd1979a00a1ba24ace \\\n    --hash=sha256:300557495eb45ebb8aec96c2da9c4be642fbf7cd937278b4013ba894ea8eb0eb \\\n    --hash=sha256:30f2aa603c41533cc25c05acd0da21636e84a315768feb631c937177db558931 \\\n    --hash=sha256:331b624368d4f1d069149002f25f44bc61c8919ce8ddb3c45bdad8f6e2d89510 \\\n    --hash=sha256:37d6d0a00072fd2948eb22bce7e1475f34569d90c87c59f7a2ec59541b77f7a6 \\\n    --hash=sha256:37dc8f7bbb66efe481bb60defacef820c950c24713fb44962ed6aa2a50966de1 \\\n    --hash=sha256:3b8182a766685eaa002637e28b4ec8d6b18819a0c71f579bf0dbaa5830297cce \\\n    --hash=sha256:3edce1d53195db527e0191f84b71d02022de0540bf43a16ed734ed7537b07385 \\\n    --hash=sha256:446c34dcc4324b084a53b705127dc15717b22c5e140ae0a3c38349d4efec071e \\\n    --hash=sha256:4998562bf62a445225f22e07c896bb04b35b1b1f2eb6d760584c9c51d7a5f78c \\\n    --hash=sha256:4b0a7fe987b14c31ebda6083f74f22b561fd3739bc0ac51e019622e3d72668c7 \\\n    --hash=sha256:4e8c2a84d977f50b9daed6eeaf3baef67d00d5d74d932288f02cb94518ee3ace \\\n    --hash=sha256:4f883547d4b7f0495ebe7056b0cc2aea76094e7a4abc8e933540f3271df27d9c \\\n    --hash=sha256:514435a37670e3e5e08f3945b68718b6ed329bb84367777e16f9f4dfe1e61a0f \\\n    --hash=sha256:53aa02d20d10c3d814d536aa4e5ac9b84ca0ff5a88377963b085ad6822f93e64 \\\n    --hash=sha256:5594fc43d548a7ed94949d139aa1341b270f1863f11cfd37f5a6c8b778a6b67f \\\n    --hash=sha256:571b9fcb07b97ef3a492028fb3d2dc0993ca23a06138b0315286566d29ef718a \\\n    --hash=sha256:57b3d78c95ba9059768b10e28b813002261d3f3dfc55cc48b0c988f625175827 \\\n    --hash=sha256:5afb51d599ea772b8365ae807ae557f18bccfe46ab261fd1c2a9ed700fc6eb17 \\\n    --hash=sha256:6b02afb9b97f65fbca5f31db6a2a3ba21aa93030225f150fa3f249717e938fb4 \\\n    --hash=sha256:6c0016e7b354317c4e9e525b937ac8596c38d2d232b419529b9cd7a1cd46e39a \\\n    --hash=sha256:71d6097b330eea8fd15097780c8e89cb1a8ce7838669f48c5bacd6f663dd4701 \\\n    --hash=sha256:756c768d0c9c2955feb7a56c37ea24aea2e369f8d36a88da270b6a9f19e62b5e \\\n    --hash=sha256:78cb2c6865a35ab8ff8b75fd122f6033b92a62c82801110e48ddd6c936a45d91 \\\n    --hash=sha256:7a743ff716f746fc19a9557f60dab1600d4613255f8a7aeb3cdde4db7eb15a66 \\\n    --hash=sha256:85f998ea1848bc6757289e739cfbdda3a04adfd58b02fc018ce54d754a5ce468 \\\n    --hash=sha256:8728f216dcdb6e6d555cf971cb34076139ad74b31fc2c14da4fafc741c5f6217 \\\n    --hash=sha256:877c3f311ff35410f690861c4409e7ccbf0cd2f878e50628a28e5a0bb689e658 \\\n    --hash=sha256:8cd2f7bdda092d99c9fc2fb7391354f306d01443d22785d0cbfafa2e2c8bb418 \\\n    --hash=sha256:8e95e1385e4998ae9694eeaa4730ba5457ff61185b3a55e2e7bea0880aef452a \\\n    --hash=sha256:962864dc93511324d51ddbb5b9f8731bf71675b93ca612a07441896f4688fb8c \\\n    --hash=sha256:9cf95fe4d0f84c82d282745d9bb08ad9f926efa00be4697e767b814ce40d4330 \\\n    --hash=sha256:9e881fca225083806662a5c43d627d215f258ff43c890f831966c7d7ba9c7402 \\\n    --hash=sha256:a2b55dd6b2a4c4b7d87ffa56bdb33fdc5fdb9a462173861a7bc097f17d91cb09 \\\n    --hash=sha256:a45650e8ce7fafffd731db8550230db6b0d306d181a90b67d3e6bca2f1990930 \\\n    --hash=sha256:a876864214e136f0eb367788dbd7df045f4806801518e2cfe9e13229cfe06d8f \\\n    --hash=sha256:ae26d61dfa7a47befdc7572b521024e8745f3d809bd95ca9505a7bba9ef849ec \\\n    --hash=sha256:af8d94b0db561cf68b88a267c5c44b49e134f525d0dc2cb7ed413a66bc23559a \\\n    --hash=sha256:b343699e8308bdc51978310e1c959c584e7869cc8c40780058c87da7781a1e94 \\\n    --hash=sha256:b3c777e849237620b022f7f297dd67705f9f5cf1685f09f02e46f93e92725468 \\\n    --hash=sha256:b629de27fda84b42cde7edef0d85f13b958b47f6e9bbcbba9b673c562a89bd8b \\\n    --hash=sha256:ba09209fbe443b4acccebe845d8a138b89a8f4fbaeedd44953490b5315d5e965 \\\n    --hash=sha256:ba54cfebe86920a559a7c4d6b9050791c20513650a1952ebe3368c7dc70306f8 \\\n    --hash=sha256:bcb46e2f9feff8d06323983bd83ed00c201fdcab3d74973e7072a889b3979fcd \\\n    --hash=sha256:bcc33feacfaefce60c12fd500a277533bdc02b10a19f7f6d348763d8140bbba7 \\\n    --hash=sha256:bf16ba1b4d0b6b7c8e534936632270cf70eb00dbe09005bc345b2677b726855c \\\n    --hash=sha256:cf1845d02ad822a369a49f2bb9345b1614744267682e7a03527dc3bf6eea1777 \\\n    --hash=sha256:d69141514cc30b774ceea5e3ed3a6635c8d8a96edf664689b890f4089111fb35 \\\n    --hash=sha256:d9c7f76c0673154f044e9d78c8655fb4213f6ca31a836df48b40fe5d187717b9 \\\n    --hash=sha256:dbce0b29841537a2fa4a214c2bbf14de3587c9680caa9b4e217568472490b28f \\\n    --hash=sha256:dc624f6bc473dacdf7ef7eb8678d0d08edf15cd94fad6ae5c7d6cc67a4e4902f \\\n    --hash=sha256:e158cb00350dc278f3b91551101aa7d12415a66ebf2c91d8d5ac14e56ddd3ad0 \\\n    --hash=sha256:e491916b378fba47242221bb9ead245211b70d504f495d105d17b14a24b4907c \\\n    --hash=sha256:e795b7eb908249c4e43c7c99fac7c2c75dab0c43566e37db472a355f63693d71 \\\n    --hash=sha256:e7e480451b9fa137494bccd3a7d69adbe8ac65a87d97be61e11f1b1050a5bac3 \\\n    --hash=sha256:e91206ee562682b51b98ef4b26a6ef48fd84e15fd4c4bc5ec768eb641d206838 \\\n    --hash=sha256:e9871b1ffbfa9656b60aeee92ed5136a5742696006fa322b29ea3d8da0ecc9cf \\\n    --hash=sha256:e9aeb04d6aef139de265b29683e119b638208f88cf73cdd1658aa07221165321 \\\n    --hash=sha256:ebaea975e03d3141d9d3a507df75c9b3ec90fa9d2ffd07567b3a978d9d790b26 \\\n    --hash=sha256:f0606c8bf2cdefea14a43530f7657cbbb7ecf1c4222512492ef4a4434a9501ec \\\n    --hash=sha256:f13c32a3abd6079a66d9526e18dad9b6d280384d49d7c54040cd57b6424041d9 \\\n    --hash=sha256:f7401aebd7f581d7f83a439d87d474999317ee099218e5ad25d125290990ba65 \\\n    --hash=sha256:fa4ecea169a355be7a3ade2c783e2ed12f0e40d2c5621cda8b3297faf7fbb9f5 \\\n    --hash=sha256:fbd139c8447d25dd750ab79ee274cc5e1fe80fc56340ab10b18a195e1b6eca3e \\\n    --hash=sha256:fdafc9cce40277e0f7a0feabce0ee50dd2fa1800f3b38015e51296b5e814048d \\\n    --hash=sha256:fe3cca2e4e8a592be0f269a1ca4835c25199d9f3ce815c8491048f785b0a0198 \\\n    --hash=sha256:ffd0c5368496f41b0944be820fcb7a838aa6e623d250b01acf2643939c3f99d7\n    # via torchvision\nplatformdirs==4.12.3 \\\n    --hash=sha256:080f3b39423b5abfca9a23d84c4e9795f54d395cd8459867a8ded44084fcd5f8 \\\n    --hash=sha256:427fc0bb321ae0c5b037fa03238ca74820437be162e78b4848c4d4055b9b766c\n    # via wandb\nprotobuf==7.36.2 \\\n    --hash=sha256:497d0463ff3316681da6c0b9e8d06cb465d61abce00b613ab42226175644d1bb \\\n    --hash=sha256:89f23aa53c24553a2416fd4fd1ec06f74fa42b14b546d8883128813f775bbfd2 \\\n    --hash=sha256:912c1221170e16c08d1f086762f563dd61ff83c18b5fa6652952dfaded66f728 \\\n    --hash=sha256:a300819d441e078a5608c0d3c709796bb548136058fda017ae51d425b44fd353 \\\n    --hash=sha256:bdb3a345d48db958e6ce1f18e508beb0cc981d64f24088427549c866cd039f1e \\\n    --hash=sha256:cbc70b17ee27e28894c7fee8bb04be1abead49e936bc70eb60052531eee2079e \\\n    --hash=sha256:e11e1f0180583a2af89db6a2ecd9e8dc40aa6d2988ca175bfd0e6d12ea72d74e \\\n    --hash=sha256:f4fee11ec330d238b34a05c9b675f693c20415d1c5bd7d5320cc2f8a798eb9cf\n    # via\n    #   googleapis-common-protos\n    #   opentelemetry-proto\n    #   wandb\npsutil==7.2.2 \\\n    --hash=sha256:0746f5f8d406af344fd547f1c8daa5f5c33dbc293bb8d6a16d80b4bb88f59372 \\\n    --hash=sha256:076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9 \\\n    --hash=sha256:11fe5a4f613759764e79c65cf11ebdf26e33d6dd34336f8a337aa2996d71c841 \\\n    --hash=sha256:1a571f2330c966c62aeda00dd24620425d4b0cc86881c89861fbc04549e5dc63 \\\n    --hash=sha256:1a7b04c10f32cc88ab39cbf606e117fd74721c831c98a27dc04578deb0c16979 \\\n    --hash=sha256:1fa4ecf83bcdf6e6c8f4449aff98eefb5d0604bf88cb883d7da3d8d2d909546a \\\n    --hash=sha256:2edccc433cbfa046b980b0df0171cd25bcaeb3a68fe9022db0979e7aa74a826b \\\n    --hash=sha256:7b6d09433a10592ce39b13d7be5a54fbac1d1228ed29abc880fb23df7cb694c9 \\\n    --hash=sha256:8c233660f575a5a89e6d4cb65d9f938126312bca76d8fe087b947b3a1aaac9ee \\\n    --hash=sha256:917e891983ca3c1887b4ef36447b1e0873e70c933afc831c6b6da078ba474312 \\\n    --hash=sha256:ab486563df44c17f5173621c7b198955bd6b613fb87c71c161f827d3fb149a9b \\\n    --hash=sha256:ae0aefdd8796a7737eccea863f80f81e468a1e4cf14d926bd9b6f5f2d5f90ca9 \\\n    --hash=sha256:b0726cecd84f9474419d67252add4ac0cd9811b04d61123054b9fb6f57df6e9e \\\n    --hash=sha256:b58fabe35e80b264a4e3bb23e6b96f9e45a3df7fb7eed419ac0e5947c61e47cc \\\n    --hash=sha256:c7663d4e37f13e884d13994247449e9f8f574bc4655d509c3b95e9ec9e2b9dc1 \\\n    --hash=sha256:e452c464a02e7dc7822a05d25db4cde564444a67e58539a00f929c51eddda0cf \\\n    --hash=sha256:e78c8603dcd9a04c7364f1a3e670cea95d51ee865e4efb3556a3a63adef958ea \\\n    --hash=sha256:eb7e81434c8d223ec4a219b5fc1c47d0417b12be7ea866e24fb5ad6e84b3d988 \\\n    --hash=sha256:ed0cace939114f62738d808fdcecd4c869222507e266e574799e9c0faa17d486 \\\n    --hash=sha256:eed63d3b4d62449571547b60578c5b2c4bcccc5387148db46e0c2313dad0ee00 \\\n    --hash=sha256:fd04ef36b4a6d599bbdb225dd1d3f51e00105f6d48a28f006da7f9822f2606d8\n    # via tabicl\npyarrow==25.0.1 \\\n    --hash=sha256:0b1edbb2f385a6a65e9711b62ba86ac54a7816a3f8d17bb3e8a5929d65fb2485 \\\n    --hash=sha256:0b726ad7e7b669be982b0c71c07fe4b037d654354130da79a7902a669e93a66b \\\n    --hash=sha256:0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f \\\n    --hash=sha256:0fe7c8b6c03969b49c8c66182e4a18e3819ab92d07cfab5d8370c531b9369ef0 \\\n    --hash=sha256:119297a6dc197e45d9c6d4415f7814a67ffa36c180d26f68c154c58067ae782d \\\n    --hash=sha256:169d3429d5be7c752125890620f75a60776d38b0035eddae939651640822332e \\\n    --hash=sha256:25f8720bf6387d5dc2ebd2622112de630760419e4b66134405dd24110d15f37e \\\n    --hash=sha256:31e49a7888fcdf3a835da33ae777f6bb9a866334e5a789282fc26dcf426f7f15 \\\n    --hash=sha256:35935cd5de130aa5cf4dea052a63e6bf2e17006c35c3a468194242b9b2bf5956 \\\n    --hash=sha256:38a9a4b4b9613380e200641891495a56c3d5a98a092db4a870af9975e220471d \\\n    --hash=sha256:3f89685964f46e4216103c75483aac0c0692a5f72212d7ca835adba5ede56ce3 \\\n    --hash=sha256:4288f27577352d608ca08553b0865e4a9b3aa14820c5d95b53337218d609835b \\\n    --hash=sha256:4340f0ba6c1d2e13f21658de1d7c662ca2545018568d0030a1e9afca159d87e3 \\\n    --hash=sha256:44a9120ce5bd81936b8ab9a88076e3fd47c2c6838e0e43630fed83626aca81d9 \\\n    --hash=sha256:4facd65742a024a4a366328a1d2292062d72d6e023c1b7dda8d4c37544933a25 \\\n    --hash=sha256:51093dd9e10325fbdb3c10a2ae7c4806e5c822d94e74ae4938b26524a3323fee \\\n    --hash=sha256:514ddb60285631af068875550c90eddc181db3e8e63a032b1559be189e82f056 \\\n    --hash=sha256:5389cdf79447ed1515c9e31620e6e1e2302249564d603f2ad727d4f6d313e4c3 \\\n    --hash=sha256:59a2de54c0cbd954da861eee4d1d330f8e909c45b53455baef696380f2c55033 \\\n    --hash=sha256:60e89d8f13861a1f7f8d950fa54aebb8023b30734d0ac51ffa80beabe2df4bba \\\n    --hash=sha256:6109c94d8b9f3b17a041daca16cacb2f651ad8f1ef70a4232c2c0f37a23da2a8 \\\n    --hash=sha256:62cd0d785b8aa6675ee355f9fc02252a340f4441257c42674937826fd7594325 \\\n    --hash=sha256:6943e2fe7954d29d84de45d29d34c8dc36ce96570e67d89aa9976e650a4a9138 \\\n    --hash=sha256:6a1fdfc6659b6b19022f2e50627fb5cf7156a66c46bf4299379955cbe742382a \\\n    --hash=sha256:880523be3d29efcf83d3998835d206118ccf35e3871dbd2fb60408cf6b007a80 \\\n    --hash=sha256:8858d7bfc22e3f51529aeaa4077225029724623e4595dc9eff8c793935c34140 \\\n    --hash=sha256:9150a83248bfed9813ea3c3af74c3856c1984d444aa28e58bf7733b9750ddf6a \\\n    --hash=sha256:9171748cdf796972d85a4b60157c279913e242992e350c90c7450182a9838b2a \\\n    --hash=sha256:a4d6d5e9a3d1879a97c08ded0c797579b7965eafd0f0c26c30b45ccc06db939b \\\n    --hash=sha256:a4dd8bf99a8fac133efc0ed6a92f5fddbe2adba0d0f6dd720e39ba9855cea85c \\\n    --hash=sha256:aa0559502e1cd6254d6814614085dd9c5a3dd0419362978a936a3f68a9e5c3df \\\n    --hash=sha256:b7a296aac7a71fa0886c08e155ddb6c636a50013f801f6178daafa0f9e726188 \\\n    --hash=sha256:bddd0c4f7630c2a3ddf6347c1bdaa79d97bcf6bd445f9e60c816b7d77c85a5ae \\\n    --hash=sha256:bf0b672390cdcb640d7288f96b826d71ff4e9abb254a86c89890baf51a29cee6 \\\n    --hash=sha256:c7c534ec03c358a76ea3e505e74c1b6aef290af90c444dfd092dbfe23e755b85 \\\n    --hash=sha256:cab40b1edfef0262e0e5251aa2c58d75630f24d06dd7794480243acc001a1d7d \\\n    --hash=sha256:cc4aa407fde9fc660be3939e49ea31f50f3e9fec17c0ec63159f7711edd3efc9 \\\n    --hash=sha256:d51592cb7561e87877c506113e7adbf1342ab579e6c21f0ef44b8ba41cb74c80 \\\n    --hash=sha256:dda9470024204d7bbf2042b47c6e8a0e47a3eeb8e34405882dfaea6577e0c153 \\\n    --hash=sha256:df961f2e7ae9cf496459259d798652c70625f6c080650d6952f8c04053c58ee9 \\\n    --hash=sha256:eb6203482ff3746a5632303a7279ae0b5a304c46985b49ed1378cb350ea6728d \\\n    --hash=sha256:f3831aaa25c67a99f99dc8b05873cb9d64560390372e2aa197ce9dd4a3f06a44 \\\n    --hash=sha256:f729cfdbd36fd99d543b67a914d2de044c84ebe45be8b34902b299b608c15c8f\n    # via tabicl-classifier-pipeline (pyproject.toml)\npydantic==2.13.5 \\\n    --hash=sha256:346a034f080da3755d8e9cb5e00e8b07de1d39e4f6e2c87d8ab7cafa0b269a73 \\\n    --hash=sha256:51a9c5f7b2f8e636f04c6cada605d9b6a3bf1348fdf945a3d8869b19bba0ee08\n    # via wandb\npydantic-core==2.46.5 \\\n    --hash=sha256:013d6f3483d81e02e7c328831808f336c8596ee33b4bd4026b9ffb1e960b8942 \\\n    --hash=sha256:03b9666e41e35d8909852ba191a0607520f81b74eaf12ccf8737005dbb313821 \\\n    --hash=sha256:045ab3b6d308439e32b81cc173bba5b9018bc6ed896afd0c65b3b009b1699af5 \\\n    --hash=sha256:0bddb4020d8f04175865ccd17eff3040874fc11fb593f424edb452653b4b947c \\\n    --hash=sha256:0cdbada856a1c69a7624a64d3d9aefe79300bd6ef827b43a4f265010b9b55184 \\\n    --hash=sha256:0fc5be0abd4a407e200d844b404e33639a554e7bd0d448e7b9ae181be4789ac2 \\\n    --hash=sha256:10416c15b8839ecc4ef4d0885da76da6fd0f67333a0eb8aff6d93c4b8f2910fc \\\n    --hash=sha256:15f4a94963c95accac15b7b657bb177d3ad82bb90b0d0526d9a9b85079925db5 \\\n    --hash=sha256:18a09e1e1011b462f2e32774f25859ef1223d5c2b0546a633cf56654710721e0 \\\n    --hash=sha256:193375f3548919d3f0b60936ca113ada3e38f264f91b9b8e0508efaad57be931 \\\n    --hash=sha256:1a353f84de772f423b5ffb11d7ae352fbbef0f446f3c0b0af0f8236d7233606e \\\n    --hash=sha256:1e449def1945a462c464331254e5a44fca7c3b4f9aedf59ec2f50f8066dd8e25 \\\n    --hash=sha256:1e5aad1220a1192c42341c8fd4a8686657e73ab2a920c970bdc4de334fe3193d \\\n    --hash=sha256:200aa3dc9f8d54f0754f43247c0bad0999fdcfbfd2488384dd44f37279271fe6 \\\n    --hash=sha256:2471fd51c61c610e1dcf7de44d7299283661654d11264ab4802b303368d69c47 \\\n    --hash=sha256:24922243639cbdac66c75fcb6fd6495a9cb52b213d62f9a0d16f0310b1ff8038 \\\n    --hash=sha256:28a6a556cd3b6066bea827857f9d9cce027c96f776e512f544a581f9e42161f8 \\\n    --hash=sha256:2bc9419666990c06d7397831f2126a1ecc3594aaa3ff7de5bf2d066802f4e07b \\\n    --hash=sha256:2cbd9a5eff05e51c447c34dfa4632145b26b09120cf04bd0c871e44c1a5e1c9a \\\n    --hash=sha256:2d330aaba8621b1edcec8ae2c4050f63b84ccf6d98723a8f212e9684713abf0e \\\n    --hash=sha256:2d5d76654becf5efd62c9e51c3756c67b49498b0c9a40884934c40807adbd074 \\\n    --hash=sha256:337639ba62a11acde6ef3aeb08c8ea755f8ef1fe5e513356c0f36a2b0d7568b0 \\\n    --hash=sha256:347ec774390c87326a2e4929d58d3f7e8763a104d5d35f4cd595a4c952366433 \\\n    --hash=sha256:356c8368cbc321050b169595683a2e1d63413b1e0e2868b330af9fc14c616d3f \\\n    --hash=sha256:37ae34309d7bd8c0d61ab839668058f2a7962ea1fc51d105d2db228fe0618034 \\\n    --hash=sha256:37ea7b83c935e5b0d68c9449b82651accf78a10828b2c02b2f2d9e9496446c21 \\\n    --hash=sha256:3a3e26b6a8274211bddee2d0e4d0d42778f17a34510f49d2ec44b58abfc41736 \\\n    --hash=sha256:3aa166e99c4f2985407fb8714aebede877ecb5455cf321b606adca926d30d5a0 \\\n    --hash=sha256:3d2652072b2d774947ba5cf78a9e59644ac62ee572daf6dd2e1dfe905e15b2b7 \\\n    --hash=sha256:40375c2d05acec10323e45dfe2077ac44bc74659008614af5069034e2cfc781c \\\n    --hash=sha256:413a717a410d0c817ef5b786a059415550b3794e1d0c2abffd9efb93a3d9f7b4 \\\n    --hash=sha256:46c25dda9d092a06c08db76ffe0a197107904d0dfac653f7d5306bbcd6d6119c \\\n    --hash=sha256:49776eab08766a08dfff7012f8b422dcd7e25e43b316eedf0477c24fcfa84b7c \\\n    --hash=sha256:4d44cf99ddebf875f9b68cc267aa684c99b7b44fe63ee1cac4ec163807290069 \\\n    --hash=sha256:4dedce55295becb61921e386b99d4f2706045306e7fa52249a33004c837379fb \\\n    --hash=sha256:4f8507560a9284e1370bb048ed4282012fbef4e8d109875b95e884d228552061 \\\n    --hash=sha256:4fdc8b93a41521988916eeaa271173fcca7fa0803d62f87675aac8dcec1c8e29 \\\n    --hash=sha256:5086029a57366b8cf81b130a43908738095c270c21a8d7f0e8bdfdb89718e2f3 \\\n    --hash=sha256:52e24eacdb536cade636aa90fb851835222becff8484b7001fdc78cb0290f2aa \\\n    --hash=sha256:53feb344243bb9510a9dec7bf3cf1b64d88a98af5dc7872a5160465f8b198c8e \\\n    --hash=sha256:545f26c504b27c3758439a5e6d9349931f0a04f855668d5fe323c89e82300a38 \\\n    --hash=sha256:54d510bac3ee52247af28ed4bb18a1e799f040ac60fd2bf5ccd4c92f1fbe786f \\\n    --hash=sha256:5cb482e9e84c851f4e623fe4acc1ced89168cf1fe18f7089db4548c8f5bbb65b \\\n    --hash=sha256:5e81740c09e310f5aa5cbd3e434a01c154d4bef93241c7877b39f211d2b78ba8 \\\n    --hash=sha256:5ee239d575f80b08eca11f6e20f90c4c695de7825c67eefe6091fbf20dda648e \\\n    --hash=sha256:5f194189415698233dd1114a093a9b56e61e2c57e11b469be3b0506f46f0771c \\\n    --hash=sha256:5f93c5fe914d75fbec9a49209b00da5f08e9e467d69da2b1510c81940cfd10be \\\n    --hash=sha256:657b40d6240c0a7b6a64b30f22d1e3aa631c7e846c621b0c0f6d1d75e2e15ea6 \\\n    --hash=sha256:6d30e1a4f138b8951063e9a394752a9179b51da288ffa507b1e659222f4c1793 \\\n    --hash=sha256:6f7b393a8b3da82f5c1fc0751e6d01ac6c55b93c18226a60bdfba4a724efafd1 \\\n    --hash=sha256:701b2e04b560eeb4bddf7a25ab8ca476176e34fdbd9a0e18196f0d12d4685f0b \\\n    --hash=sha256:771cf63ae0b1b50dd22e5f3e3549fab5f3f4ff1635d352a9e1a97fe01c7b2e64 \\\n    --hash=sha256:79bdfa52f843137045b2d081cc05c120ba6665d29b7559c2c47690906f39279f \\\n    --hash=sha256:7ac031912d54f3d83ef3b3eb98dfabc1608802e2202263d25957eeed40b94761 \\\n    --hash=sha256:7b0fc826b16c55e561e5d2a0c5c77b051ba1d92808118c4e4b5390f5e0cf191d \\\n    --hash=sha256:7c6be839a5a8312626b32029a415644a0846b420bc8b52b95b28cd92da162168 \\\n    --hash=sha256:816ff0a6550ffc06c098ccd2e0698600f9aa7da192a79eaa6f9af504a35db869 \\\n    --hash=sha256:82a36973cf8a2ef5406f4fe2edbf8ed0c99629535d959e0b100c76a32535a111 \\\n    --hash=sha256:837b396ca3d7b74091ca623f6cbd8351bd42d670a79c2683e79fb089f06a2de5 \\\n    --hash=sha256:850a08d167dde16db8702c274f320c7be9d7da6f6dff2b58b18f9e815bd94f5b \\\n    --hash=sha256:8816f3d218beb4b787de5c9759c259b8fa61f9dec42dc7811f320a33771778b7 \\\n    --hash=sha256:892a881d5f68c2b9ea304b7a6c2c60d9343df578a311b0f86b94bc8f1ffe8129 \\\n    --hash=sha256:895395f8918627b04efb1ad2a4cf605387143300ba03304cd1dfa6d03f5e095e \\\n    --hash=sha256:8b10e3e8fd7ddc2bd915848a2768e44c15b22936f1cc54c462ad1164deb02655 \\\n    --hash=sha256:8e24d8f05fa2d28513d94e877e9c75ad66175376209b3977f916e240e623193c \\\n    --hash=sha256:8feeac04b5794e513e710af2f9c87d49f31a6dc47967bb264a1fed61a8989bec \\\n    --hash=sha256:9432f3598db432cb51c5b37fdbf29a60fcccc79e30d37a05022776a6bc4ab689 \\\n    --hash=sha256:976e1128455aa595ea04c79ccfedff1aaeab96ee013fcc916bed120c4f0ad94f \\\n    --hash=sha256:978e7b97d4824b5be09c69fb70507cbde3b0323fc147332ca40a94d9a6a0ebbf \\\n    --hash=sha256:97bf8de4d541598c94a59344eeb988a94c08ff76b5723c41f6567ec18c7892ea \\\n    --hash=sha256:97cf3eb53a8cccacf9d46686a0926186c9bfb5574f2ed66d3639d5fe117cd3a9 \\\n    --hash=sha256:9b68938dd5b0c783d88ff8e2dcc69451b5eb936fe212d516b21b9d5567f6d464 \\\n    --hash=sha256:9c4b71f10dd532fb7a5cbc8f58707779e64f03a258c2bf8bfbaecfcd9970b519 \\\n    --hash=sha256:9f47b8a949e60f027f0aa0a6f6c7b7e9c55cbf4380d10b344e282fa4e7ab1e1b \\\n    --hash=sha256:a1dee1b804ff4d11c663636cf15d2ea47e9f79cd56c033fb1cbf08924842a48f \\\n    --hash=sha256:a2468d93d181667a7abd66e1b64bb9f76f361b0fef8faddf687456453576f5ee \\\n    --hash=sha256:a2a5e1d0ff29adddc9f6d6821a66302e4493f8ca898b715b6b1182c2c201ea0a \\\n    --hash=sha256:a39ac25a9a2fa4072efdb429833c4a4c8009a51ff9eea3eeae131713cd27991e \\\n    --hash=sha256:a445486499897b88a7d6c310c88ed64dd37b1b59bfd7ae9107490bbb362f47d6 \\\n    --hash=sha256:a91c17edf6eea2402cb5457b4c89e99bc5ed1004aa34c4adf1d4258c1a5c22c2 \\\n    --hash=sha256:ab4b66edffb32d9e951efb3814bd104b8367a7501b81b955cacb5726d897389f \\\n    --hash=sha256:aca6c767f552b21b10f774aeac128e828eafb796adfa1b666a18bf6321453c3a \\\n    --hash=sha256:acf8a67ba51f4ca9ddbd0e6b3000a65ac51ab734661778b3e7ba64d99a710f2f \\\n    --hash=sha256:b10ec717381bdbfafef34607824db4c91de69ff085e4fca3b2af91b4fa17e68a \\\n    --hash=sha256:b49924c73a235e969511bf2aabdff3beebf9820931f646c80274d5d780010c47 \\\n    --hash=sha256:b6acfb46a814762367fb7ba0828b0a17d441b92ce249a0e007474c9072662dda \\\n    --hash=sha256:b7ca9034437b6022f941f4857459562ee00a560b97e7cce8a0ec5a74fc6766e0 \\\n    --hash=sha256:b98134087d9de723658d17a42c7d0da8d6e2ef08015dee7dc93889047315f5e4 \\\n    --hash=sha256:b9fe6fb92520e3fd61f2e49000b6911b188824f089b75973ea06d6267f0b476d \\\n    --hash=sha256:bce57638e08ac148e5778cce7feb968307a727d66f8e2274a543d0cf0c9ad6a3 \\\n    --hash=sha256:c14ad3bdc85ee7f318742c457ca3968a92126d144b15721c759033bfb06296c2 \\\n    --hash=sha256:c1c43ad4339643d70ebb8124e1305a7dab423001eff58bb41a0f731adbc98355 \\\n    --hash=sha256:c3471e5c4a949c26ec00a77f01df59096aa9495877de76fd60a980f8ee6be461 \\\n    --hash=sha256:c583b927a8838dab890706a6fa7573fbb8b70e24000ef9f7238e2d6f6435a5ed \\\n    --hash=sha256:c76fe65e607be28c7fd4d56fc3c42b1583aa058ce3408b7ad0fd540171d31f9f \\\n    --hash=sha256:c7ea57fc63aa7da93a1bd2d644e6577befae10c52c4e36377635eea1056a74f5 \\\n    --hash=sha256:cd5214352ae68f3b5e9af7768bdc5253695ee069675db3480518420b3be881f2 \\\n    --hash=sha256:cdbb78909f52b981d3b2d56b97328d71eb0b974c36bd77c920123a7ebb192829 \\\n    --hash=sha256:cdc8b74ecc48c0cb1e9607a05ec4e9e88db60a19ffcc9a1d5f9088ede40c8dc0 \\\n    --hash=sha256:d0a24b40877af2de4950252be9d21eaf7fb07660f3c2cae1f56c6b599ada5266 \\\n    --hash=sha256:d22a945598fb91236b4dd793a6e42e4f3dd7740bb5aace5ebd7d4c08d13bb575 \\\n    --hash=sha256:d2f9fc07a8042a8f95925b35c4f04f469707c981fc33245b6ca187cf5d2dd290 \\\n    --hash=sha256:d625a186a65201c23a9e3b8ed9c47e90a026e03256608cc91851c6709096844f \\\n    --hash=sha256:d925f3d9afd05a8c0fb3a1031463a8d59ebe5e2afad297e29c78be19e13b4e62 \\\n    --hash=sha256:e64e88d5585bea9ce95861079de72006c7fa6d3df4e3a3b65ba31eb979c15c9f \\\n    --hash=sha256:e652ab17569c94bff5475520f907b7148b8c24036a8ebbe5cf7cf7493d28579a \\\n    --hash=sha256:e7b891faeedeafba41b2983e5001a81b6a915b69544c7e7570d1989ce1c36ac7 \\\n    --hash=sha256:e80675d75ae2cd14372cb65cad5400d9347a3d3f6c13000183f22dfd027283ed \\\n    --hash=sha256:e9c134bb666dd54b778b9fc0d2b50cbb7f979b9e3716f26a88c9ab3b6fc1dd0f \\\n    --hash=sha256:eb7d8d0e5886a89a55d2eef490e272fa965a9d57c6b29a5b5088a7997ec2cad1 \\\n    --hash=sha256:ecb42011e12ee19cafbc312887cbf3546959fe02fbad44f272d4be5baa997615 \\\n    --hash=sha256:ef3fbbf161dc9351a2fe0422e51b129f9e97e42385bd0320b309c15f7d287dd8 \\\n    --hash=sha256:efd62a42486f1bda5d24cb4f63d15a3c7768375fe83d36f9417b4ad7a2fb20b3 \\\n    --hash=sha256:f077d0b97ab11fa7dcc633fca53515f290bca8a8a633e966d5b6d1879d9ed01a \\\n    --hash=sha256:f332f0e72a5a0400141f830744e141bf9f97917878dbe968669e8a7fefea78ff \\\n    --hash=sha256:f7b0ec93a2893de856652154d73b7ba622f26fa97726487dcac373de5f4c6084 \\\n    --hash=sha256:fa10ef4112775900e7a0661068635eb67b2ab824fbde764de6e0e21982a93db0 \\\n    --hash=sha256:fc5d783bd4a2387e97b8a2d5ec781cfb92b3d893bf82370548e99db5915935d3 \\\n    --hash=sha256:fc8515076c11f3cfdf4fb142dcca0fe384b1230a3b5415458ac84f3e0903ec13 \\\n    --hash=sha256:ff218293c9c806138dca139765e3b067621be52bcd93cdc14c7711be7ddc90a9\n    # via pydantic\npygments==2.21.0 \\\n    --hash=sha256:2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9 \\\n    --hash=sha256:610ca751c9bc2492b38eb9a38a7fbc93edbbb2d7182edaf34e66ae493dee5c8c\n    # via rich\npython-dateutil==2.9.0.post0 \\\n    --hash=sha256:37dd54208da7e1cd875388217d5e00ebd4179249f90fb72437e91a35459a0ad3 \\\n    --hash=sha256:a8b2bc7bffae282281c8140a97d3aa9c14da0b136dfe83f850eea9a5f7470427\n    # via pandas\npyyaml==6.0.3 \\\n    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n    # via\n    #   huggingface-hub\n    #   transformers\n    #   wandb\nregex==2026.9.29 \\\n    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \\\n    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \\\n    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \\\n    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \\\n    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \\\n    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \\\n    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \\\n    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \\\n    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \\\n    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \\\n    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \\\n    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \\\n    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \\\n    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \\\n    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \\\n    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \\\n    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \\\n    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \\\n    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \\\n    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \\\n    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \\\n    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \\\n    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \\\n    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \\\n    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \\\n    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \\\n    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \\\n    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \\\n    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \\\n    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \\\n    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \\\n    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \\\n    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \\\n    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \\\n    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \\\n    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \\\n    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \\\n    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \\\n    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \\\n    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \\\n    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \\\n    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \\\n    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \\\n    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \\\n    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \\\n    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \\\n    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \\\n    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \\\n    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \\\n    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \\\n    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \\\n    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \\\n    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \\\n    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \\\n    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \\\n    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \\\n    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \\\n    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \\\n    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \\\n    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \\\n    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \\\n    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \\\n    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \\\n    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \\\n    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \\\n    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \\\n    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \\\n    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \\\n    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \\\n    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \\\n    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \\\n    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \\\n    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \\\n    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \\\n    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \\\n    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \\\n    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \\\n    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \\\n    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \\\n    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \\\n    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \\\n    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \\\n    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \\\n    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \\\n    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \\\n    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \\\n    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \\\n    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \\\n    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \\\n    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \\\n    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \\\n    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \\\n    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \\\n    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \\\n    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \\\n    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \\\n    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \\\n    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \\\n    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \\\n    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \\\n    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \\\n    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \\\n    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \\\n    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \\\n    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \\\n    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \\\n    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \\\n    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \\\n    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \\\n    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \\\n    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \\\n    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \\\n    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \\\n    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \\\n    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \\\n    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \\\n    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \\\n    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \\\n    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \\\n    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \\\n    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \\\n    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \\\n    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \\\n    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \\\n    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \\\n    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \\\n    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \\\n    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \\\n    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \\\n    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52\n    # via transformers\nrequests==2.34.2 \\\n    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n    # via wandb\nrich==15.0.0 \\\n    --hash=sha256:33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb \\\n    --hash=sha256:edd07a4824c6b40189fb7ac9bc4c52536e9780fbbfbddf6f1e2502c31b068c36\n    # via typer\nsafetensors==0.8.0 \\\n    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n    # via transformers\nscikit-learn==1.9.0 \\\n    --hash=sha256:051075bda8b7aab87b1906ab3d4740a1e1224a19d7b3781a576736edc94e76aa \\\n    --hash=sha256:056c92bb67ad4c28463c2f2653d9701449201e7e7a9e94e321be0f71c4fef2b8 \\\n    --hash=sha256:147e9329ef0e39f75d4cffa02b2aa48d827832684926cd5210d9a2cb5c57246b \\\n    --hash=sha256:1b944b6db288f6b926e3650026ddafb988929de95d11fc2cc5fa117773c9ba42 \\\n    --hash=sha256:1fea2cc5677ab49d6f5bade978c866da44957b712d92e9635e8b4f723013c3cb \\\n    --hash=sha256:24360002ae845e7866522b0a5bbf690802e7bc388cac8663502e78aa98598aa2 \\\n    --hash=sha256:26e22435f63bcdcf396b574273f29f13dd531f5ea035801f5be10ba1540a4e60 \\\n    --hash=sha256:2bd41b0d201bc81575531b96b713d3eb5e5f50fb0b82101ff0f92294fdc236ac \\\n    --hash=sha256:366652351f092b219c248f1e72821e841960a63d8f358f1dcfd54dc1cbdbbc28 \\\n    --hash=sha256:38c3dcb9a1ffb85505ec53d54c7b4aea0cff70050425a7760c2af661ac85df05 \\\n    --hash=sha256:4306775fad04cc4b472a1b15af1ae9cede1540fbfcc17fbce3767cd8dc7ae283 \\\n    --hash=sha256:4ccacf04ca5f4b492158a5f28afe0ace43f81b2571e4b9a66d34848b46128949 \\\n    --hash=sha256:5162ad10a418c8a282dde04c9aa06965de3e9a65f33c1440c0ae69bb1a09d913 \\\n    --hash=sha256:5808d98f15c6bf6d9d96d2348c1997392a5888ce7097e664105f930c4bca1277 \\\n    --hash=sha256:5b934c45c252844a91d69fda3a34cff5e7307e1db10d77cb10a3980312c74713 \\\n    --hash=sha256:5bad8f8b9950321b54c965fdcbac6c6c55e79e16646b49977bcf3668d3870a1a \\\n    --hash=sha256:5be45aa4a42a68a533913a6ed736cf309de2226411c79ef8d609a5456f1939b1 \\\n    --hash=sha256:5dc1818c77575d149e25fce9ef82dd7b7263ae372f03494158668ad632a69759 \\\n    --hash=sha256:5e50ed4da51974e86e940690e9a3d82e729b62b5a49f7c9bac534d515d39d86f \\\n    --hash=sha256:64fa347efc1c839c487433e40c5144d38c336e8a2b59c81aa8660373945c2673 \\\n    --hash=sha256:78fc56eafd4edb9575d2d8950d1dd152061abb573341a1cb7e099fc40f6c6666 \\\n    --hash=sha256:80746d63bd4b6eaca54d36fe5feaf4d28bb38dc6f9470f81c7cad7c40155f119 \\\n    --hash=sha256:8833266989d3a5110178a9fae30783675460724d0e1efb13b14901d2c660c557 \\\n    --hash=sha256:9656acd4e93f74e0b66c8a36c88830a99252dfa900044d36bc2212ae89a47162 \\\n    --hash=sha256:9db6f4d34e68c8899e4cab27fdf8eafe6ed21f2ba52ceb25ea250cd237f8e47b \\\n    --hash=sha256:d77f54c017633791bc0225a43e2f8d03745fdcfe4880268fcc4df15f505dec2e \\\n    --hash=sha256:da76d09304a4706db7cc1e3ebaa3b6b98a67365cc11d2996c4f1e58ba47df714 \\\n    --hash=sha256:ee1a8db2c18c08e34c7412d4b10be1cac214cd4ea7dc9715a6a327eb49a37c96 \\\n    --hash=sha256:f401448645a3e7bc115aa3c094097865155b34bff1cba8101857d9104e99074c \\\n    --hash=sha256:f7e254636164090da847715a27f8e5478feb98c40a9e0ee90cbd277de9e5ceb8 \\\n    --hash=sha256:fd3a8ef0c758555a3b23c03adaa858af32f7736785ded50ad5991f59c4ed03fa\n    # via\n    #   tabicl-classifier-pipeline (pyproject.toml)\n    #   tabicl\nscipy==1.18.1 \\\n    --hash=sha256:011413b7426b75012840e35649e00fe0a2c3bae89fed433876e3a99251572efc \\\n    --hash=sha256:0ac49ea97594532dd44b7136094d35f5440fa06e6d9c6384a74c01764df388c5 \\\n    --hash=sha256:0e82073ecc7acc6436fac4b31674109c7e1d3e596789767eda01258a8c9e8123 \\\n    --hash=sha256:0fcb3c93519f27bb4f0c4b0f7802cdcaca7fcf93267b75edda2e9f4e8a55cbd7 \\\n    --hash=sha256:10ac20c69d880f77f375db44c22e3e6a644f9fefa291d4cd2fb9790a89fc99fd \\\n    --hash=sha256:11c423f1049c5755ad4409af52a9ada1cff96fe9b50795d4af3619f292901239 \\\n    --hash=sha256:179ce34a8d0fe273d8883ba59e17e052247d08973dfcb743ca52bb1cce2d60b0 \\\n    --hash=sha256:1bca3b943fc2567ea49cd02c99abde49da4d5178ec46f624bd8255cda8755beb \\\n    --hash=sha256:1d73131e358976663dd969e1fb4ed1404b815cd977eaaedc3b3a133ba2d81c35 \\\n    --hash=sha256:2a0b02f9fc46f8520330c23d45e6560db7e3a0d927232139427637f98943e11d \\\n    --hash=sha256:2d3ab0e8c69a17dd3559eab8cbb88f258e285c94d572c2719033f90f83290c89 \\\n    --hash=sha256:30f464bee641fa8e282577c7dce027308403213c6ca8270bba73285c91024bc5 \\\n    --hash=sha256:33a834464fdabc0f26a45508df31b3cc5d028e04dbf6c5ed398541418e0a12fe \\\n    --hash=sha256:3ab3523da44749156e1f68b464dc56af11ae4cbc5c739a49d05f32b982eca9f3 \\\n    --hash=sha256:3c085faa2cfa879c5141df483f836f4d691045a078224a670fa570fa01612d89 \\\n    --hash=sha256:457fd7a2a8edeb044ab6ffbc0aa03ff6cd18491356e5e0c834d76ce621b916d1 \\\n    --hash=sha256:49023963c193dacee096301452f223ee24d86ec5807f8df93c0f7221d119e305 \\\n    --hash=sha256:52c4b7422442aba924d03ad4019852b08a92e64ea187b933135687bfe2747307 \\\n    --hash=sha256:559ed65f60c1af5a03f3912605a1b5114f522c7c32fb23c3376ae8f03219fe28 \\\n    --hash=sha256:5632e3ae3d09197c446310cd5187de63e28448ce22f0f67b2b93d97503c0c230 \\\n    --hash=sha256:5e4d44984abc0020154ea81b247adeddcc3ac5527b975ff798bd1ba0adc513c2 \\\n    --hash=sha256:75b00eb8fb802090aa903f4ea1c7f5a584779f967361e68b7e98e531cc2d7174 \\\n    --hash=sha256:78a0d7c918e74a232394117160e7e3db503377572a45bcef8826e4ab8a35feba \\\n    --hash=sha256:78c0665edead396b1abb4897c41a5c1d9bf090c8a637a4c20a61678e0a264e66 \\\n    --hash=sha256:7bbf207c4453ce1ad2e00b17313852b33310b83090c2311bdaf97f93c0380d12 \\\n    --hash=sha256:7f4b8bc363b6d65ee2152bec57568e3c52639bb34c46057b09857a307ed5e21d \\\n    --hash=sha256:82f201b4c878551d48558337aab270d3c6cca5507b8737c8d8a608d234cccde0 \\\n    --hash=sha256:83de5453a7799afc9048b4616bd085cef126e36412f0ea2f6370c36a2a3a51e7 \\\n    --hash=sha256:88f0e784020649f88ea48c9f5ddfa403bf9205820667c0914740b392035afb82 \\\n    --hash=sha256:8bcf3c1ba5d6456e2effd30fcbd3459b044d683fcdac79a2e6830f0bdf7de487 \\\n    --hash=sha256:911de823097db8b63f034299d12662db93344e6ffa0b881cbb57748974b70168 \\\n    --hash=sha256:92c14f5bdbfb6216315ce33e78080474082de8b3830122ba97809bfbe65f75c0 \\\n    --hash=sha256:95298364e251be3e60249facbeeca03631d3bb7584f85879516ec55ac717b81f \\\n    --hash=sha256:9554bcc6d715ee87a633a3cc8e7703c6628b100dd29cb8a2efc4c0533c7ff729 \\\n    --hash=sha256:9f2897bf7737392ad0d5213ea7b6add72a4edf5679b3153106aeb88b6507b3b9 \\\n    --hash=sha256:a1d33a7836f7ddc1993427966a0823468ec41bcbdb1a9f9942d1d7e57f803ba3 \\\n    --hash=sha256:ac0333bdf38309aa3dcbe7e3fa7ea29e7a2c37c6ea306a757b700ded8e4596ad \\\n    --hash=sha256:bff0b729edd992766136b34e39cc76bc2fad905aa58897ee72a9cd000a6d8443 \\\n    --hash=sha256:c24acac1e18912761c4700239bbc1fd32f615af690f1584d49b35859be51324d \\\n    --hash=sha256:c35d74ce0e193ff740c2f2be2ac913ddc232fe6c1ff40b26cfecb9c670c63314 \\\n    --hash=sha256:c825cef2f49e46753726a7181a8e199804a912b29519ada542c6ebc654951899 \\\n    --hash=sha256:c9d18a33309122074ea483dd92dd444189166b8b2ec429fe9ed5ac73c7a0aa23 \\\n    --hash=sha256:cbf38d043c1aa4ab306e1ada6ab6eddacc3322a20b7af1b30bc93254b366fe09 \\\n    --hash=sha256:cd479fc04dd9401e3b4f49e76518768ef99c4f517a98c284eb091fd725719adf \\\n    --hash=sha256:ceb30a00ce7c92d459819443d29ca486d882b83fb6738bdcbb2a1cce94ac5daa \\\n    --hash=sha256:cfbf154f2ba187f2ed6cce2639efff7d105f1140573642c0161615b6d91d6a87 \\\n    --hash=sha256:d2924a03db38dc2e848bca2fe9f077dafb891480b91a00a0963a8cf86dfc31c1 \\\n    --hash=sha256:d416b16cccfd70fbf62400e84d0bb2f4e6af519a45557f1692c749b37f14b315 \\\n    --hash=sha256:d65d448389b8436493abcf629cc94ad0cf32aecaf06e1acca1de53cc795f2f12 \\\n    --hash=sha256:d84a09d0dad90ba6525d8ac1c2334b33e64bf3ccfe9e841f02feb867a22681e4 \\\n    --hash=sha256:ddef79fb382df40104a19bb7151b3b23e57c1778fcf857c71ceecd9bd264513f \\\n    --hash=sha256:e3b417bf8c2c7c16e8f58ad91db17783ec911ac16e7b50eb6eab6e809b4f5b07 \\\n    --hash=sha256:e402cf31eb68f453dbb2d36fc6d722b33f24a55d68b2ae1d92fa6305ca71c298 \\\n    --hash=sha256:e6fb6a55cc0ba97b59a1f288fb86dc6fce8bdfc0fffcbfd015e3a954bf2a2d93 \\\n    --hash=sha256:e708533e8b2ae2497d65346538a7dcc92814410b25b81432eac66de0f2af8265 \\\n    --hash=sha256:ea324d9dd34c38bfb9bec8ca4d1b407db97dbb74029f566b8e322b1b6fe56fe6 \\\n    --hash=sha256:eb0dfcf4e28a99c12c999744a2ff67c9b06200e20401c7c88186e33552a46331 \\\n    --hash=sha256:eda632a7981f69730d6281f451db9c1c370993a2c0d7ddb43e2a809a2862b83a \\\n    --hash=sha256:f29633129f9fa7e88a3f0fca835de2d030bfc9643f7799e1a0c46cee24d38fc7 \\\n    --hash=sha256:f55fa87b6c612ecd6b058f167c53231b1d14e412efe361d3d6e38b3631c73218 \\\n    --hash=sha256:fdaf5ea890a6183d0565f51a61799d67081bd5b1cf03c5f4b3fd3732108625c9\n    # via\n    #   lightgbm\n    #   scikit-learn\n    #   tabicl\nsetuptools==84.0.0 \\\n    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n    # via torch\nshellingham==1.5.4 \\\n    --hash=sha256:7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686 \\\n    --hash=sha256:8dbca0739d487e5bd35ab3ca4b36e11c4078f3a234bfce294b0a0291363404de\n    # via typer\nsix==1.17.0 \\\n    --hash=sha256:4721f391ed90541fddacab5acf947aa0d3dc7d27b2e1e8eda2be8970586c3274 \\\n    --hash=sha256:ff70335d468e7eb6ec65b95b99d3a2836546063f63acc5171de367e834932a81\n    # via python-dateutil\nsympy==1.14.0 \\\n    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n    # via torch\ntabicl==2.1.1 \\\n    --hash=sha256:7abdb1fa878e7a1edfa1c6606bf4189e9cccc20935e8011d7e690f4693c9c5c5 \\\n    --hash=sha256:cb4405cc93335c688bc9bcb703c7944032fcf542b43ebb66820f1a5acb5651b1\n    # via tabicl-classifier-pipeline (pyproject.toml)\nthreadpoolctl==3.7.0 \\\n    --hash=sha256:61348cfb77d53b9242e0017029244b559b810c142ced65b4e21eeca1843959a7 \\\n    --hash=sha256:cd8b60b5641b45c67bbf73c64c843235fc2d8a480c87389f52f5dbee893b86be\n    # via scikit-learn\ntokenizers==0.23.2 \\\n    --hash=sha256:12f0835dc2ee694746a76adf7b1567d4346a4a502ebe93fb1f5f80ea49799b78 \\\n    --hash=sha256:2e96f5699d5249c9c64aa8412e044f727aae3a4098cf830f9901ec1afc361cde \\\n    --hash=sha256:325fee2e0418a9dc6c9ecf736a5f5f0db7875183ace9549ae339da76f7a1fbb7 \\\n    --hash=sha256:41c2f84d172449b4dadb9cdc508e3e364076613c35b16e76ecfe47a60d1e3305 \\\n    --hash=sha256:43e4f2071e3cc8d5d86421c874aebc82659bb51a68bcdef5a0da75ee89511ccb \\\n    --hash=sha256:5c56bda1511921587789163e524d196ed8284174ac23abd7685d5ea8da6c4718 \\\n    --hash=sha256:7b7e37ba198f24150f523e1242e83c4970de4a525480586be5dcc24d9add32c5 \\\n    --hash=sha256:7f0f085686b9de0d0079e6f874ae053600db64c5d13049e0bbc0119926d25aac \\\n    --hash=sha256:85a9a357a3764aecc904ee76bdaf8cf1ad8e5a67a1b929a487c4a39b49ed0e90 \\\n    --hash=sha256:950d7c9426fa72406a0ffeacdbc0bb9985f5db20eb8b263f29c79aaf83105703 \\\n    --hash=sha256:986670e43691469dcee610ea0f846f91a8f84e91fc6f7a48d4c064414c0ec2bf \\\n    --hash=sha256:a37039b5dfc4af84eb3ef0a92f4307e28936c8f9adccba2629d36f652e9bf7a2 \\\n    --hash=sha256:bef235815a067b2648caf6dcc7a71091b0b0fff9ee8057f6451eb9335fae52ef \\\n    --hash=sha256:debf978920d93ba9c219bd67cc4bbfaf912c9039e41e7a28b91ec15e3728c95a \\\n    --hash=sha256:e49c394456dd9985787fec76132438ba3fb8911f857b1bf3d40119f9292d41aa \\\n    --hash=sha256:eb2f9c8a24da020ea8c11a01a19c1c2547912d92121ae4a01cfbca46125dee40 \\\n    --hash=sha256:f486f402f6f9abee5bb032553736813af0c710a86b2e0ca592634c55cea1f835\n    # via transformers\ntorch==2.14.0 \\\n    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \\\n    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \\\n    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \\\n    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \\\n    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \\\n    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \\\n    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \\\n    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \\\n    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \\\n    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \\\n    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \\\n    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \\\n    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \\\n    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \\\n    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \\\n    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \\\n    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \\\n    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \\\n    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \\\n    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \\\n    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \\\n    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \\\n    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \\\n    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963\n    # via\n    #   tabicl-classifier-pipeline (pyproject.toml)\n    #   tabicl\n    #   torchvision\ntorchaudio==2.11.0 \\\n    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \\\n    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \\\n    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \\\n    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \\\n    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \\\n    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \\\n    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \\\n    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \\\n    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \\\n    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \\\n    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \\\n    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \\\n    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \\\n    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \\\n    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \\\n    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \\\n    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \\\n    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \\\n    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \\\n    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \\\n    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \\\n    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \\\n    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \\\n    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \\\n    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \\\n    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \\\n    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \\\n    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84\n    # via tabicl-classifier-pipeline (pyproject.toml)\ntorchvision==0.29.0 \\\n    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \\\n    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \\\n    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \\\n    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \\\n    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \\\n    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \\\n    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \\\n    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \\\n    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \\\n    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \\\n    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \\\n    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \\\n    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \\\n    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \\\n    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \\\n    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \\\n    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \\\n    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \\\n    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \\\n    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \\\n    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \\\n    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \\\n    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \\\n    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e\n    # via tabicl-classifier-pipeline (pyproject.toml)\ntqdm==4.70.1 \\\n    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n    # via\n    #   huggingface-hub\n    #   tabicl\n    #   transformers\ntransformers==5.17.0 \\\n    --hash=sha256:78ec1ce21579b38dfb83950a0658cd119f87212a2fcfdff478096ce9d6c03801 \\\n    --hash=sha256:a153be279169b55b92d8000bf4af294aed684503d091cca7804da2dd8a9de000\n    # via tabicl-classifier-pipeline (pyproject.toml)\ntriton==3.8.0 \\\n    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \\\n    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \\\n    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \\\n    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \\\n    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \\\n    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \\\n    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \\\n    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \\\n    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \\\n    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \\\n    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \\\n    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3\n    # via torch\ntyper==0.27.2 \\\n    --hash=sha256:269b7eb9d3c202ca84b4bc9618cb04ebb43d3d4d1e567e4c768607232c05f945 \\\n    --hash=sha256:b3a5fc4342d5fc8fda8fc3010b1cf117e9249aab7fae800c2eff62fd3842d97d\n    # via transformers\ntyping-extensions==4.16.0 \\\n    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n    # via\n    #   anyio\n    #   huggingface-hub\n    #   opentelemetry-api\n    #   opentelemetry-exporter-otlp-proto-http\n    #   opentelemetry-sdk\n    #   opentelemetry-semantic-conventions\n    #   pydantic\n    #   pydantic-core\n    #   torch\n    #   typing-inspection\n    #   wandb\ntyping-inspection==0.4.4 \\\n    --hash=sha256:547274fa6b0a561ccf549cc9524b999a578e737d015d8709d021f9d0d13bea47 \\\n    --hash=sha256:65b8397ba37ccbce054456aaccddfc91e6e3083c92824df348d96ca832f3f147\n    # via pydantic\nurllib3==2.8.0 \\\n    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n    # via\n    #   opentelemetry-exporter-http-transport\n    #   requests\nwandb==0.30.0 \\\n    --hash=sha256:1f2cc20c033f1650d387ca073177e629f1e2a1b5c4be393698e3db4752b9cf3a \\\n    --hash=sha256:217bde0a7c49482f26166a1a7bee760bb9e2658135b2d45a33aacabffec68817 \\\n    --hash=sha256:36d6d8a5b88b36ad82b0f7ffab79033a8bbba047b5a90c0b7e26d0cf3a09f0b6 \\\n    --hash=sha256:72b564712e85e10d78d37ccdaef4a908d28cbb39240c9c00fa4dde567150c764 \\\n    --hash=sha256:9cf2492acd5c66ba7217667386e2a90535ccaec08079b97c389c3da5d15cde48 \\\n    --hash=sha256:bad9412f137d02303bf178513e0109f4fed37a6c817604085a1a4151047d8790 \\\n    --hash=sha256:e7dc3b0b1567f6a121d8f7a58131b5d085861139a653ef8bed1569c9be796721 \\\n    --hash=sha256:f2d7de3c2019204f493f3f9e9166071c2be36f74140af2e1409b4de0fd84c454 \\\n    --hash=sha256:f4f908ace761db42d239d100230df0b77eb9684924d962a762bba176d829a236\n    # via tabicl-classifier-pipeline (pyproject.toml)\nxxhash==4.0.1 \\\n    --hash=sha256:0163b5d259de23ae9e07b7eabf435ce4704f6f205589a2b154e6af4be985ce1b \\\n    --hash=sha256:03600a8987849b2bef7be795a60a6052b635c63fa98b718b08ca5ee823691cfc \\\n    --hash=sha256:04f9a24de11a6647666d5302fd73d6a5224ce50ddc965fb0bb44cee736e6bd7c \\\n    --hash=sha256:06713a5aaf1d0905c5579416c020c02e42b3ceb931e86c7d3b7fb85403dee3f3 \\\n    --hash=sha256:06d7fbd609503c3be5e65cdb6bb2f040d6a98574404e2e1d5c60815c97fff4aa \\\n    --hash=sha256:0718ad66f4ded2411f8e62bdba549ee71e313a2d26ef5060ca3fdbf29897dd3c \\\n    --hash=sha256:08ed8da18cd4fd0a6a5d6a444852d8fbd0e565388a74a4937085451b5f1a312a \\\n    --hash=sha256:09f9feb118966cc6650e1806205d577eae7ca394aa6acf349a0b62a94bbeb329 \\\n    --hash=sha256:0ab851b45c70d4992be7cdeeee16f97a0b677408c758c4b1efb1cfe8030bfd37 \\\n    --hash=sha256:0b1082fd0f089ce9098ed77aad8b777b5d156f8ac601c69cab73811822b8ef07 \\\n    --hash=sha256:0b20a06454b34f1531fc677c54efe2ecdec691ef9224f7fa919bf2c1363f7ff1 \\\n    --hash=sha256:0b42a5a26607e4b2409fea174773a66f2dff9dfdbf2c1a851bb7b804e2c97535 \\\n    --hash=sha256:101aa300de6ceef3d9c77569706330d8921fc45dd82bceed2084f1e9f2557a24 \\\n    --hash=sha256:1216f7ba5683f17a89eb7dcb4bc50a0b743dfe1902278d7b3d0786f538118433 \\\n    --hash=sha256:1642907941ee4b75aacc3db688af52ea02ca2305ab22af7ee686ed726b332684 \\\n    --hash=sha256:168dd6b51725a222abc722832e56624d15a63fc2e8249021509c93f1063913f6 \\\n    --hash=sha256:1749f0688020209fe0d357ce1e1cd9ec9c6161ed0405ea949d24581c4c43fa91 \\\n    --hash=sha256:1b3cccf75eeb5b01639b2feadb042a8e07889293b7ca72fa2985e7dcb64763cf \\\n    --hash=sha256:1b50223d92df94d54e1a31469335a2c74b16692e6c1cb726f1e6949514458706 \\\n    --hash=sha256:1bc591533fc975614f7e13594daee76af96b8e1fbcf8de76c8773858fa9e7cea \\\n    --hash=sha256:1c2200b98a805351cb3142ae4e1fdcc9e91b5e20f5d30d4862b0b96f92558f4e \\\n    --hash=sha256:1c7c642a0f79c3e3cf2965475507574d3d1a50ec71060039d60cb87358667cb2 \\\n    --hash=sha256:1ee523f51718e41753f04f7102bb4dc55a18d2ea5cbaceef8ec7ca08571bd428 \\\n    --hash=sha256:1f3346c5c287ac3c7f38b20380f55e8768230e7252af59fabcf3b87ab21e4256 \\\n    --hash=sha256:2194bf96d5f3d4e0cb65deba370ec83dda3edfba42155f9384190ed5e51ea5e2 \\\n    --hash=sha256:237b8f63a2a0fcfb1ffc06e21dad23add44e6d354b2b014364a1d41e419a4dee \\\n    --hash=sha256:23a4376b4a3183cb50d4d2a3179f887a7773cc695eb2c908e551bec3221b8c60 \\\n    --hash=sha256:247ece770647c0aef080561fa996f9774b4dadce2d0c42eeb98229db7dcf820d \\\n    --hash=sha256:2696bbac613f6880fed60316c298bf3091d4f8eee3ae2e9466f70bb76204fb0c \\\n    --hash=sha256:26fe6238c2d5b11ed5063b9bf4eb290624b004fd074688da6bb079bd564f10d7 \\\n    --hash=sha256:2d52dc7c33c1b83082b707f6b7814dc76d2faaa2ea62bd9c5fab4b36f83c087f \\\n    --hash=sha256:2df3ca8757dc381e75e90a4d7995a6324f58a923c7145220a7b2c0231f66fddc \\\n    --hash=sha256:303121aab4b7f898058582d7962ea79d9e26e2379d7b6d8743f70f2671674481 \\\n    --hash=sha256:3088dadbffa33c29e0518578430a7dff2e901a212e487aefa5faaa0dc06dad34 \\\n    --hash=sha256:31d86f9e81f3e84e00131ac7c54caf5119ae4ddd82c09c31cff597c813ce1ee2 \\\n    --hash=sha256:3358097d333d40657569ec1121e21043dd7d0efa10aead1b50e8b4fa83077d7b \\\n    --hash=sha256:33e270d302c95ec426dfa0f5a4e16bff2ab8d7b8a46faa4746affb05e684ac77 \\\n    --hash=sha256:33fd538191f47071deef6b1f676535e2aa770f1fd150ae4cc75a34c9e930be3d \\\n    --hash=sha256:348c8f288dc961d6bbd1985c8152a3ed7a85c95df00e82320f0c5215d922a399 \\\n    --hash=sha256:349775ac30372b344d2338b2a168c0a1312a644194da25b8bec476d55761a128 \\\n    --hash=sha256:34ed93e20bfd98d722b902121643791eeb4b1641871e2dc63d0d4c2d93f187df \\\n    --hash=sha256:37f667dee0f867c42894b34e2a6fe26bf195c0ea4683d9d2b713db023f242c3a \\\n    --hash=sha256:3891efe3d7a531ce6da0a4a50a99dd41c75b8fd4ca19d73c86431b4db5c305f0 \\\n    --hash=sha256:38c3d22129a6958846a3098d68bc8e661704461c0be4793ae28836e4690c8478 \\\n    --hash=sha256:3c2445edafc300cc40feb6a25a8356a971c30cd0bf47b5349c2ad74c508343b1 \\\n    --hash=sha256:3f68fe400ceec235f3e4a4b02a28c2fd2d283584a193223c921dd4c48f1d0754 \\\n    --hash=sha256:3fb1d30d4b6d6e2c4a08e5ac6fffdb2b572d2cfcca15a5509cf4e7a1350f955c \\\n    --hash=sha256:41e579025a6e13a99e6d71e39c9cfc621a0dcdbbf19106325e145fa858f2d794 \\\n    --hash=sha256:421b94f3ba7067958d02e38960d987756347aa150df06df11aa68ae1af78c619 \\\n    --hash=sha256:427b62d62d4f967fbb10b82a3813e4875c2a6e7e7634739f17265b650c7f65a6 \\\n    --hash=sha256:436e11b4dd966afe5f7f665e4cc4c5485ffe3ceb42f25a22e1701d236abf1853 \\\n    --hash=sha256:43bcf2a871f28f16135545415cab3ec43904d4c80425a64598a9e6cebfb2b5ba \\\n    --hash=sha256:43e5f9169e73d0f0db33b5f6b8554bcce69ac278c966daf83d5eb4eb2f13829f \\\n    --hash=sha256:440c401e146ce64bdb3beb8ff0c84677b6f21307c28a34779071cecee5d4d70c \\\n    --hash=sha256:44ab12e8cd17d4f001769f00ad465208b4bcb897ed29e65f058f74466b57a98f \\\n    --hash=sha256:4528cf80ebbbf57d40edfb31521ae265daa6dd636d615b1cf0ac86209579e59d \\\n    --hash=sha256:45e88111ebe331de478ef8d4293efbe88f3cf8b863386c9a2357136b838e1af0 \\\n    --hash=sha256:4741d42d59e4e5fa1a86c17ab9c27dc8ea459c700d91b6742fdb9138d9a516cb \\\n    --hash=sha256:4751f1d7eecae6b2d2a773630f1a7248f125c9a92a456694d03c15bceffc9d68 \\\n    --hash=sha256:488ca5c5e28ef56ec4bbb12f835b3f1cbecc5f3510062e70117bc6594851932a \\\n    --hash=sha256:4972332c079d6aad69c4620a68d015a4ecb33141583f70d642cf9edf6a713763 \\\n    --hash=sha256:4a252fb862b0ae2590587e625f47a0e03da05cf0205e8830b67b6596c06038b1 \\\n    --hash=sha256:4a76345f5aceb4ec404918edf9c7f2b5507db864dc0d7455982009ac0890b57b \\\n    --hash=sha256:4af350bc3f329970c0e3a59af84a8a30998bf8a9167eb50cd48e59baaa1d7bec \\\n    --hash=sha256:4bbf3ff651e0f1a19beb5d0f48e0874a9bad2482a588c9d214c96ef1fff1cd9c \\\n    --hash=sha256:4e5141543c7f7fe3087500bbb4ac2845cb528a980aa91f8f1e661e2292ff4a5d \\\n    --hash=sha256:4f5e5c6df4b703afcbe9352d238a51efd97c3b91fdc3a2052e40fdacb1e7505f \\\n    --hash=sha256:515a822c73abbf6a0b7c70976d9662be342835c9d78b8dc7c023411f39c35dbc \\\n    --hash=sha256:554f87034635bcec47c5d72447bf3db7e02da1bf493a0ada010db28a76f891c6 \\\n    --hash=sha256:567cbc630302a46a8ecfd943b309ccf5372bb3718f1f3762d452df30f033bcf0 \\\n    --hash=sha256:57d7fa8f23908d173001c21a9e82bfc6ad997d1b6c270fb121812b7ed158891c \\\n    --hash=sha256:5adf927dca8c47fde7e683fe69efdd81bc865c4db1fb6bb00b391e2b6185207b \\\n    --hash=sha256:5b7875ac1a2edcb691f27642b8b94b904baa6bcecb7d79c72df2228ba8cb5c51 \\\n    --hash=sha256:5b7979f71d06ae45a769de0699900a246d8cb632db1e8bfdc79ec019063a503c \\\n    --hash=sha256:5c2d525a3afabcd8e3549d85fc7e111fde6bc302d06a1893fe73adb79823415e \\\n    --hash=sha256:5dc434c946012e6d8a72b10f970ea30755b718251dd7591dbfdabafd3bcb21bc \\\n    --hash=sha256:5f1ea31d61bcd2cd2f3ec4ca80a64187bbd7948f490b63cf0dcbc6e717b4c1e9 \\\n    --hash=sha256:62198213fc3e0c56e567894b318ba45834e007d065f84ba6dc9165d21546fc56 \\\n    --hash=sha256:63aa52659bc32bb9bd7cb5caf523b4d14429a477762cfac886132d687c1f80fc \\\n    --hash=sha256:649f2682c090cca1ac4037866381f3652eaacbd56e5178030f4ce1325b8f945b \\\n    --hash=sha256:67e57b834e07ed973cee7b6da1548ff28a56458d77696fd2a5f397f340694848 \\\n    --hash=sha256:684160b3c0a9b62c6f0de90f44e11dc5d8643dcfa18a5856b45fb1c47478bb71 \\\n    --hash=sha256:6a8c5ce76b94ba49f3be8a8f2611abc6564210702c72ac9e237ca2bebfd17794 \\\n    --hash=sha256:6a9f98af872355e0c02439e48583958eee00e60b928bb20476460d9d40cb7b4e \\\n    --hash=sha256:6c45258a37fc22721395c09927cb982d3e7a83607cab15be7e2416501bd3a330 \\\n    --hash=sha256:6cbf4e21ef0890804b5bb9ad25c48f9c127758d7f6c66bef374efcacc63c738a \\\n    --hash=sha256:6cf633df84d80a1668fcf61e330791dae46825e395549e7d34f376411e75088a \\\n    --hash=sha256:6efb8f21cc136c79b3e5bb747c8682d37916fb202cdbbc32182de5c4e47f821f \\\n    --hash=sha256:70129ebb8f20e1ac1da58b78ed381624bd689a43a9a7366560bd8fabea145105 \\\n    --hash=sha256:704381264b36a18b9c62ecbabe2e71d0fc58c77c129c15355c989b10bf05b6b0 \\\n    --hash=sha256:7236be540d6be9ce448d98b940dd26ddf70ca41012e8a14a53fd9354cefe4e8d \\\n    --hash=sha256:72f34834518157a75e7090f328ee7a16c70c804cfc7c694fa069cc888e9fc03e \\\n    --hash=sha256:74379a577a9f3b6afbdedf1b90e5c7764467051977f18a326d7d607336d743bd \\\n    --hash=sha256:74a164e8b63f1e9cf35c9a7809d082b033d1a00e7375d5d814415436e7867e57 \\\n    --hash=sha256:760de77279e9cf9c81d012ce0705cba13afccee9b09c480f17d778c8c5cefae8 \\\n    --hash=sha256:764b32d52d15b8b95ac8160e540772fa1adeb611fe40bffaeb42e7bf98279e44 \\\n    --hash=sha256:79a3203aadf39637869dfea1185227d8452844d78b837e54fb1117b4d34ba5c3 \\\n    --hash=sha256:7c343ee174d417a44d0c3355602c0cbbfa52a04d1bbbf1723378c7d2c8f60626 \\\n    --hash=sha256:7e27dbed5c4ba033919e4b4ed8dc14e029e91d14a93cd9f920d25277c7df6781 \\\n    --hash=sha256:81507a68ba84c55241fb61cce1469f473a5da4205fc8ef6f698e5948eea8dd88 \\\n    --hash=sha256:81664268dba92e037b740ecf37fa02f1cab4a391f93f28e35792b3341c60648f \\\n    --hash=sha256:839f58c5bd9989875be0fd28446dbf32cace2c2cd8bf2f6762acdc38a95cd1aa \\\n    --hash=sha256:83b8c2013edb5dc1f9e7268b6496130705bc48d79c86bb8817b3d210b81a5513 \\\n    --hash=sha256:84df5f8da574caadbc0cb1b8866ecc2368cc941f0cd05f677756c802f370dafa \\\n    --hash=sha256:8580aab306888224074c7edeec734de0c3c5ccde65b2da4e6c9a5e28f7c0a1bd \\\n    --hash=sha256:85bdd40cb505a11e0ca04191711266c5fd696ed786ae83849955e457774edc96 \\\n    --hash=sha256:85e402dab0f9acd3604539747c6fcc57dc188a18af6ab07eb8189351cd32466c \\\n    --hash=sha256:863f3d3b44110f7243e86cf994aa5c5d88f2348b6e84ab4402fadadfbf9f7da7 \\\n    --hash=sha256:86b2b12bec60c678ed8f5cca0258ad93a8928ebddb6ca7732f0875afe1451d1a \\\n    --hash=sha256:87aa309a93bd5ec13f14309a305ff4e9bf74c5363fc46c264c0a22edfd5b0670 \\\n    --hash=sha256:87cbdec1a7dd930079671a60b249f3ca4e773e6fbd0676e21e36fdc9dd0f3b00 \\\n    --hash=sha256:87da13df72c5612771cd905a8b121e0bfea62d7659b1c92198736eb722220e83 \\\n    --hash=sha256:88d87719fe6bddf117238b341c5db851f8e96ba68ad9832b450e4a43dc60b37f \\\n    --hash=sha256:8b4477edc03091f51f5309406d230851c23cf4822029e3bf40b8df53093fff1c \\\n    --hash=sha256:8b99ebaf9e816ac5069423b1367ee7e8078fbcebcf62545506bb0608d2f4f468 \\\n    --hash=sha256:8ba782ca3bf1e81492611152b9a0d5264971339e95e34d69de0ac2c926be496d \\\n    --hash=sha256:8bcba9456242ebf180a04d9443812fd85ffe6bd12bda464dd116fcece8886ff3 \\\n    --hash=sha256:8c9fe122444e129881afd1d4d1c7ac0d3ce2d91b68c2b40173b6025ff1c31f9a \\\n    --hash=sha256:8ec4777d92fd61a5c8fdeddab894fd65bea301a8092fb5419ec6472aa4d458d7 \\\n    --hash=sha256:90cb2a1c9cc503a054a19612b48ff6e8e47805f618bdb3224a07568aad03a37e \\\n    --hash=sha256:9283d9dd6b44acad35118e2976fc763a065509e4118debdb61916ec322ed17b9 \\\n    --hash=sha256:94ac8a6b8c47951173f0b67bf862bcb971bf24e493b9fbbdb0e010cbbc7d9f54 \\\n    --hash=sha256:96d8de55029d42251945531f6aa7590c32b48163c66a43bf29d8657d7446a377 \\\n    --hash=sha256:96dedccfb09a73a25751053a183159b88f4ee75f388df8166040c152ac0531c6 \\\n    --hash=sha256:9761ff4a0ffa583fe850731ad24fe82c88cccb7a2294727db0955f3279a4cb3f \\\n    --hash=sha256:97b455de3e8b1b0b1e4594cb61a468992563f03ca264062fbb0a66b393c01d90 \\\n    --hash=sha256:97b94fb29abf21f5f0bde15f7dbdd3a4aa2dc59f37026adc7b4bee8563b84375 \\\n    --hash=sha256:99054b838b74d8d3995ea0d410976ae967c46207ae22d6ddfc535e809197dab9 \\\n    --hash=sha256:99166cc98637e8bf550cda2aab07f4f1d5f899c45fbd721801aeabcc9d404824 \\\n    --hash=sha256:9a51b061d54cda8b83e62c44458bfbf0dabbef9b975dd9649952ba5076b9f349 \\\n    --hash=sha256:9b1dddc257279417d93c9e59420d49ef90aece90d7a01996db3aade74b0281b1 \\\n    --hash=sha256:9c3c4b9aa9a27196b921197f7daf9e6c1412739df06a99cfa6e923879362eff6 \\\n    --hash=sha256:a14578102a6081465aec9cf73c76c3cd3f79f0709bdb3b8ae7ab0b54c9d8b089 \\\n    --hash=sha256:a16a3fa6936e36bb1414d16a6bd012c9033e5161b68b426805b61d895392437d \\\n    --hash=sha256:a33de7633c948ab2dc144af370a66e7e7af29b425dcd0f7e4f59689fb9391b53 \\\n    --hash=sha256:a43418e1a90b4809a9caf64aeb8b0696e3e1f300a323acc1e6ee2f93ae319fcf \\\n    --hash=sha256:a4553d36cc0b7fce1f35ba8a94dfd775aa3ed12f5eab2dc3b46ac75a0706b0bb \\\n    --hash=sha256:a5b21b42a01a343096a1c018d35e9b7aec9c7065dda53ae8da071e37478b2cea \\\n    --hash=sha256:a65785e653573fcd1e33062760ab4c3c3440e8e910765018e4b6ed4ad07b54a0 \\\n    --hash=sha256:a6671a8f6ea4f2101ce11fab5023a2e59391cff249fc3928cecb69d971525fd5 \\\n    --hash=sha256:a69e8946e4902ea11fc1c557740cdbfe7d75c78fcc5e4324ff89a696a634357d \\\n    --hash=sha256:a6e3653df1a70b8ac4191216324242e4be2bca18c9a7c10934e1bd56dc7ca15e \\\n    --hash=sha256:a865d2d470220e659220fdb59d5b6c4422802d8d6098e1324bc4d12444798914 \\\n    --hash=sha256:a949b072ea59c6eca0811ccd9e95133cc50d2afda8d464b5b077c78f78efa269 \\\n    --hash=sha256:aa6ccc7f31018484d652cf52db020003433f3c9fa83189c028bd807d2adde503 \\\n    --hash=sha256:ac0f291ab6485bd71f33941f9b92771318332a05d505460b41e893a549caadc0 \\\n    --hash=sha256:acb31ecdd1a97fab5cd39a84ee9f515e727d319f796fec48703b8339b9998360 \\\n    --hash=sha256:acf52474b2494ef66dc7e0fb6d5e2b50c18313039ad4d275fbf9f9907c804bc5 \\\n    --hash=sha256:ad889d58361a26ba75f5d6a1a0da08ed4950ec4ac8a6da86e1c5ce1b95ccb43f \\\n    --hash=sha256:adbd48b30e3f82c89fb2b3e6a87cdd28d113b190a5ed0ee2dee286323ee9a621 \\\n    --hash=sha256:af05a3f650220a6c59fa0ad2410249f2d2470a05225807c378fb67458693f8df \\\n    --hash=sha256:b3662719007e059abde7eddacf8517142ba076ddc7b30c807260e57d28c3c191 \\\n    --hash=sha256:b3bece52127ac20044311ee73567f9f0893b5de64f9028aecc90cc740cfd525a \\\n    --hash=sha256:b4c8842fb19d78b5e8c2a52baf4c8357658cc56c62bc822b86ce0f942f28e286 \\\n    --hash=sha256:b659fad79c99b0238c7ad7e9d7dbf4eebfea9097c2dba65fa0a4d18a25b29a2f \\\n    --hash=sha256:b6c1f9c59bbe593f88a0aad30be4150f15bd57bd64efb95feeabcb8e563f1ecd \\\n    --hash=sha256:bdd16718b63aa3ebd68aabb79021a40e47c81374852d41a306b9453141bbcbee \\\n    --hash=sha256:bf430c587f447a554c53768ad76b9846fe7c5632180ef6f69c4fce8b0552fbd0 \\\n    --hash=sha256:bfed61996d618eb90d6eaae0178002e3466a28b06bfc557a7a3a7266378d8c5a \\\n    --hash=sha256:c09ada495567c9c9a8156c5ebcfb93be7fece0755062d738c972dcbecd0d84b5 \\\n    --hash=sha256:c0e6ccc2b19ec8a726b2e26062ac71ea63e15500d6bf85910e42481844fdffc1 \\\n    --hash=sha256:c101180495cb4ba3617b279a944345c53a5e73b0c150053d1fa8d8af32de9579 \\\n    --hash=sha256:c10b9206753b64aa791b35b201485477525b26fdec5bf86e8364c388a03e2592 \\\n    --hash=sha256:c3074db513c81f764053e3da079312ecf85a50d8350c71f4cc0105d9662a9e6c \\\n    --hash=sha256:c30dd1af66a820820398b26e0d74e7a9aa43cae705924f23ed828cd8e5c26c3d \\\n    --hash=sha256:c57963970d359a72262f7fe6be88f945e2334d4bc41462b7f08c37b0abf35ca6 \\\n    --hash=sha256:c6301d92545c591ad31c3e050aa40a5f8a4c16413f1f9e6f9322c6f0f9d2b736 \\\n    --hash=sha256:c6370189e8e66b7e608f533b939a9de092ddca6cce084ca0d3d414d2ed5b5d59 \\\n    --hash=sha256:c6fc415b5568bd9accc7187f1729a99707330c0a67a8b9f93c1149ed573ed75d \\\n    --hash=sha256:c7484fea54964edd417cc3a104d5180562514aa7c4e2a2bc26d776ef0c4cb4a1 \\\n    --hash=sha256:cba763d84b06bda2c38d5185dee76f1b9dfdc0789e96e476d9e10005526d0788 \\\n    --hash=sha256:cd878d32f5c6cbce9783f8d6897561fb772211edba9dde49d85672b88ed45276 \\\n    --hash=sha256:ce6d5cc94a50291d080259a126cbf1e9ba4ac861e6429d2f3cdbb1474f51945d \\\n    --hash=sha256:d0d24a4f3fb63852cd09af46ae4b7a4d00cc8b8615a046dca543786e728d1056 \\\n    --hash=sha256:d1e0d1ea6e44f51808a9e8469c8afdebcdf6fa23d1ea524a0303d57d23919712 \\\n    --hash=sha256:d54b8ae068af532c8cdf56abb9e09a60fbe7b10792444c9c27987bb6d3b450fa \\\n    --hash=sha256:d55bf4ef10eb09b8b6866790e083d26d087d84caa3cc0946ba87c3ca7ecaf7b7 \\\n    --hash=sha256:d9f3848ffaf010bdbabdbf4c25641fa258b6227ff27bc74a4d06edef521a4873 \\\n    --hash=sha256:da0264844a09b538c894e5eff25313d941deb4dedec2131b98418a71a3c9944e \\\n    --hash=sha256:da544672efd9ad76077928a3e6c5d894e52ce82d3bf14002db4a1bf17d1a36a2 \\\n    --hash=sha256:daade8936c4deaaf7b01561324ce438ba4f885d717e9adc62b4d67212ad7d7bd \\\n    --hash=sha256:dd649663ddeafbfd4734eb8abae921dd5baa1242f20bda54e8bc927369ccded4 \\\n    --hash=sha256:deca2a30d983d240b8375ec2ee0a4288e72042827fc61df2f7671f8467e4cb2f \\\n    --hash=sha256:e259bb7e1e2d8de6b35f430f5c7220b1c0ebf3962d1ba7ec7545980d5931edb8 \\\n    --hash=sha256:e3996ff9b6f99180357024336bf5749a8ad6476a9a2523e535c5212b995b12a2 \\\n    --hash=sha256:e3eba72f9bb84fe696516f4cbca68d3d74a376157e68bacddbb7f2516af61523 \\\n    --hash=sha256:e4296fcc790876a8b0f297edc83d3b088457b774d8f67b4636807f8a2ec69a79 \\\n    --hash=sha256:e53926e76131a74e79cc0b39fa712c227875f180afc68646bd1e1d8a17e60313 \\\n    --hash=sha256:e681a6fc7e4f715252b9b5acfb30536ec7dd1f75033a32dc617e6fa95af1a3fd \\\n    --hash=sha256:e71b34978e77868cbf2d18c5206a4603f9c644dd7181bec5643bd40141d3b8c5 \\\n    --hash=sha256:e8cda075b10bb3917b002c74a04f9e02b7d13b5bf732571404d51c52b11c7329 \\\n    --hash=sha256:e90b4bcf1d9eb1010fdaee7c9209fb667e74c0684f3ba17f9032bd7319da90c9 \\\n    --hash=sha256:e961093277ff9d42addb9dad5614dfb7800ccba07c245c39c8e9b4daa35d160c \\\n    --hash=sha256:e9701c073bd062fb6bf6be51b47186ad15f1e87feedf4ea07198e0333ec068dc \\\n    --hash=sha256:e998cb3685b92101ec5de0fb4d9485cf01e50bc418211955c55d98064664cf4c \\\n    --hash=sha256:ea5ecf800b45bdb34afe05a1d0dae1f8ea02a290e50636dccd399063f6b180f8 \\\n    --hash=sha256:ec1a470c6db94ac4589c203921e89ac1bc13e796a8b1784d8135e1893559cd3b \\\n    --hash=sha256:edccc2ec58435a580f96a48a3ccae8cd0a480824119165dd90108718ad81ae6e \\\n    --hash=sha256:f00330ac7e24769e2032203f2b01794d670916b0c1799fd261340f1af9499875 \\\n    --hash=sha256:f09ee747e2a5f876cc5ad56947734811828335e13b403dd8ea1e06d77a9dd48d \\\n    --hash=sha256:f18732adcc271741bd651c3e56fa519d8a237d2cccda01fe3afb226bf87f783b \\\n    --hash=sha256:f1b603d0686c99fa0879f104a74e7db58367634c6e50ba827bee9aa095e23205 \\\n    --hash=sha256:f33cf0baa91eccd2cb7b62bf00f10c2264ef578b71dd33a12962e71a36eb4d32 \\\n    --hash=sha256:f3e1a44af01b6692de0ec6caba5f0bf93ceb36896e02b7fc00952c6ea7ef39e1 \\\n    --hash=sha256:f484ed57bb3e4142f9d6439568658c38be5f94b702ba00a1ff32c69783b6c66d \\\n    --hash=sha256:f5d031f35962e5483a613214e61f09fe24ab523062c3646d592dc16c4a217451 \\\n    --hash=sha256:f6247f5e23ee94f2557ac9dab738a336f607c6ff476fcf66ca70c3aef5eee15a \\\n    --hash=sha256:f7db035447a0ac8959aa230c5d36545ecf9f547413eb1711c0ca6f0ba1418925 \\\n    --hash=sha256:f83295394d34e1287e5b30fcc496c13b92cf886a131f3dae5444e38da8757efb \\\n    --hash=sha256:fac4832b638000106207bc44e44b9616a6a416aaee56c62b01d61f3705e49f58 \\\n    --hash=sha256:fb59a0dd61fb2ad481c03fda399d78ce57dab6bb62c2c8fdb446a7ba4754b89a \\\n    --hash=sha256:fc737c05ca2d48e5dcdbbb249314df3fc6c2a0be6da8b0aa28e13d72afaad7cd \\\n    --hash=sha256:ff48915bf1871a1f19f74c11834c6329443d306cedc0c05fe7fe617810422a80 \\\n    --hash=sha256:ffa44b4c7c5d0ffa31356b4428659516c0e47647825c74079a296b3857b6d99d\n    # via wandb\n', 'weights/tabicl-classifier-v2/dimer-base-manifest.json': '{\n  "format": "dimer_hf_snapshot",\n  "formatVersion": 1,\n  "modelKey": "tabicl-classifier-v2",\n  "modelId": "jingang/TabICL",\n  "revision": "4dcd344ece2c00be9e831fdd35bed57b5ad83e19",\n  "files": [\n    {\n      "path": "tabicl-classifier-v2-20260212.ckpt",\n      "bytes": 110368038,\n      "sha256": "bdc7dbd5e4ff21f8f0456fcf90c6b7cdf72dbea960f2d05b19bec19f9b3d4ed0"\n    }\n  ],\n  "totalBytes": 110368038\n}\n', 'LICENSE': 'BSD 3-Clause License\n\nCopyright (c) 2026, Kurt Valcorza\n\nRedistribution and use in source and binary forms, with or without\nmodification, are permitted provided that the following conditions are met:\n\n1. Redistributions of source code must retain the above copyright notice, this\n   list of conditions and the following disclaimer.\n\n2. Redistributions in binary form must reproduce the above copyright notice,\n   this list of conditions and the following disclaimer in the documentation\n   and/or other materials provided with the distribution.\n\n3. Neither the name of the copyright holder nor the names of its\n   contributors may be used to endorse or promote products derived from\n   this software without specific prior written permission.\n\nTHIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"\nAND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE\nIMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE ARE\nDISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE LIABLE\nFOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR CONSEQUENTIAL\nDAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF SUBSTITUTE GOODS OR\nSERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS INTERRUPTION) HOWEVER\nCAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN CONTRACT, STRICT LIABILITY,\nOR TORT (INCLUDING NEGLIGENCE OR OTHERWISE) ARISING IN ANY WAY OUT OF THE USE\nOF THIS SOFTWARE, EVEN IF ADVISED OF THE POSSIBILITY OF SUCH DAMAGE.\n', 'source.json': '{\n  "repository": "kurtvalcorza/tabicl-classifier-pipeline",\n  "revision": "ade4356ae2642841e4236ef3d8aaacc54f6561e4",\n  "generator": "build_notebook.py/3.0-tabular",\n  "notebook_spec": "3.0",\n  "files": {\n    "src/tabicl_classifier_pipeline/__init__.py": "c2b8cd9a177ac0c762db0d38417bf238f99160e0bf301c9a7c7c740e5694bdbb",\n    "src/tabicl_classifier_pipeline/api.py": "c1232f7c60b577eb2addbbd4656cef17446e002434412330416421b315431555",\n    "tutorial_stages.py": "79d87d18294fba598ecb8e8d999bb4fce229ac544f055639b9a834f7c251ce60",\n    "requirements.txt": "c364e6a779ced5aa08987476ab86a4d7d1910f2426b8b94d826735c8c8d32e13",\n    "weights/tabicl-classifier-v2/dimer-base-manifest.json": "9770b17efe4c06f1c667c74ce720c6eeb80d5417717b1afd96b1dafd934b03b7",\n    "LICENSE": "f35c084e579356a91e0c417d2946a9fbc075c1fd0908d5fadebee72f4d482d0c"\n  },\n  "sources": {\n    "src/tabicl_classifier_pipeline/__init__.py": "src/tabicl_classifier_pipeline/__init__.py",\n    "src/tabicl_classifier_pipeline/api.py": "src/tabicl_classifier_pipeline/api.py",\n    "tutorial_stages.py": "tools/tutorial_stages.py",\n    "requirements.txt": "tutorials/requirements-colab.lock.txt",\n    "weights/tabicl-classifier-v2/dimer-base-manifest.json": "weights/tabicl-classifier-v2/dimer-base-manifest.json",\n    "LICENSE": "LICENSE"\n  }\n}\n'}
CARRIED_BINARY = {}
CARRIED_HASHES = {'src/tabicl_classifier_pipeline/__init__.py': 'c2b8cd9a177ac0c762db0d38417bf238f99160e0bf301c9a7c7c740e5694bdbb', 'src/tabicl_classifier_pipeline/api.py': 'c1232f7c60b577eb2addbbd4656cef17446e002434412330416421b315431555', 'tutorial_stages.py': '79d87d18294fba598ecb8e8d999bb4fce229ac544f055639b9a834f7c251ce60', 'requirements.txt': 'c364e6a779ced5aa08987476ab86a4d7d1910f2426b8b94d826735c8c8d32e13', 'weights/tabicl-classifier-v2/dimer-base-manifest.json': '9770b17efe4c06f1c667c74ce720c6eeb80d5417717b1afd96b1dafd934b03b7', 'LICENSE': 'f35c084e579356a91e0c417d2946a9fbc075c1fd0908d5fadebee72f4d482d0c', 'source.json': '88b063faab89c3856ade2d199d8c7bced3b36a4bbe2481da81620908eafe3825'}
for name, payload in [*((n, t.encode('utf-8')) for n, t in CARRIED_FILES.items()), *((n, base64.b64decode(b)) for n, b in CARRIED_BINARY.items())]:
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(payload)
    if hashlib.sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried file integrity failure: ' + name + '. Do not edit this cell; regenerate the notebook from the repository.')
NOTEBOOK_SOURCE = json.loads((ROOT / 'source.json').read_text(encoding='utf-8'))
print({'carried_files': len(CARRIED_HASHES), 'verified': True, 'repository': NOTEBOOK_SOURCE['repository'], 'revision': NOTEBOOK_SOURCE['revision'], 'generator': NOTEBOOK_SOURCE['generator']})

**Infrastructure: the isolated environment.** This cell installs **nothing into the notebook kernel**. It downloads one pinned file — the `uv` 0.12.15 installer wheel, refused unless its size and SHA-256 match — creates a separate virtual environment with its own CPython 3.12.12, and installs `requirements.txt` into it with `--require-hashes`: every package must be the locked version and match a locked digest. The hosted runtime's own packages (Colab preloads its own NumPy, pandas and PyTorch) are never replaced, which is why no restart is needed. The environment is keyed on the lock's SHA-256: a second **Run all** in the same runtime reuses it and prints `environment_reused: True`. The first build downloads PyTorch with its CUDA libraries (a few GB) and typically takes a few minutes. The cell also defines `run_stage` and `load_record`, the two helpers the learner cells use: each learner cell runs one stage of the carried runner in its own process, streams what it prints, and stops with the stage's own error message if it fails.

**Expected result:** one dictionary with the generating revision, the isolated environment's Python, the `torch`, `tabicl`, `numpy`, `pandas`, `scikit-learn`, `lightgbm` versions, whether CUDA is visible, the number of locked packages, whether an existing environment was reused, and the setup time. A failed download or a hash mismatch stops the cell; never remove a pin or a hash to get past one.

In [ ]:
# @title Infrastructure: build (or reuse) the isolated locked environment and define the stage runner
import io
import urllib.error
import urllib.request
import zipfile

LOCK_SHA256 = 'c364e6a779ced5aa08987476ab86a4d7d1910f2426b8b94d826735c8c8d32e13'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
ENV_ROOT.mkdir(parents=True, exist_ok=True)
UV = ENV_ROOT / 'uv-0.12.15'
UV_MARK = ENV_ROOT / 'uv-0.12.15.sha256'  # digest of the binary extracted from the verified wheel
if not (UV.is_file() and UV_MARK.is_file() and hashlib.sha256(UV.read_bytes()).hexdigest() == UV_MARK.read_text().strip()):
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2 ** (attempt + 1))
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError('uv 0.12.15 wheel size/hash mismatch: refusing to run it')
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
        UV.write_bytes(archive.read(member))
    UV.chmod(0o700)
    UV_MARK.write_text(hashlib.sha256(UV.read_bytes()).hexdigest())
# Stage processes inherit no kernel Python path or start-up file and no credentials (every download is public). The
# kernel may export an inline matplotlib backend that the isolated environment cannot import; stages write files.
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1', UV_CACHE_DIR=str(ENV_ROOT / 'cache'), MPLBACKEND='Agg', PYTHONNOUSERSITE='1')
for name in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP', 'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'GOOGLE_APPLICATION_CREDENTIALS'):
    ENV.pop(name, None)
# One environment per lock digest: a second Run all (or a re-run of this cell) reuses a complete matching environment.
VENV = ENV_ROOT / ('venv-' + LOCK_SHA256[:16])
READY = VENV / '.dimer-ready'
PYTHON = VENV / 'bin' / 'python'
environment_reused = READY.is_file() and READY.read_text().strip() == LOCK_SHA256 and PYTHON.is_file()
if not environment_reused:
    shutil.rmtree(VENV, ignore_errors=True)
    subprocess.run([str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(VENV)], env=ENV, check=True)
    subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes', *['--only-binary', ':all:'], '--index-url', 'https://pypi.org/simple', '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
    READY.write_text(LOCK_SHA256)
probe = subprocess.run([str(PYTHON), '-c', "import importlib.metadata, json, platform, torch; print(json.dumps({'python': platform.python_version(), **{d: importlib.metadata.version(d) for d in ['torch', 'tabicl', 'numpy', 'pandas', 'scikit-learn', 'lightgbm']}, 'cuda': torch.cuda.is_available()}))"], env=ENV, check=True, capture_output=True, text=True)
RUNTIME = json.loads(probe.stdout.strip().splitlines()[-1])
print({'notebook_source': NOTEBOOK_SOURCE['revision'], **RUNTIME, 'locked_packages': 90, 'environment': str(VENV), 'environment_reused': environment_reused, 'setup_seconds': round(time.perf_counter() - SESSION_START)})
if not RUNTIME['cuda']:
    print('The isolated environment sees no CUDA GPU: every stage runs on the CPU (slower; see Prerequisites and Troubleshooting).')


class Refused(RuntimeError):
    """An input refusal: one line, `Refused: <file name>: <reason>`, without a traceback (NOTEBOOK_SPEC 3.0 DAT20)."""

    def __init__(self, name, reason):
        super().__init__(f'{name}: {reason}')
        self.name, self.reason = name, reason

    def _render_traceback_(self):
        return ['Refused: ' + str(self)]


def run_stage(stage, **options):
    """Run one stage of the carried runner in its own process with the isolated interpreter; stream its output."""
    if not (ROOT / 'tutorial_stages.py').is_file() or not PYTHON.is_file():
        raise RuntimeError(f'The run directory {ROOT} has no carried files, or the isolated environment is gone: run the Infrastructure cells again in order (Sections 1, 2 and 3), or choose Runtime > Run all.')
    log = ROOT / 'logs' / (stage + '.log')
    log.parent.mkdir(exist_ok=True)
    error_file = ROOT / 'state' / (stage + '.error.json')
    error_file.unlink(missing_ok=True)
    env = dict(ENV, DIMER_ISOLATED_ENV=str(VENV))
    # Re-read the fresh-test mask on every call: a copy taken once at install would fail open.
    mask = os.environ.get('DIMER_MASK_FRESH_TEST', '')
    env['DIMER_MASK_FRESH_TEST'] = mask
    print('DIMER_MASK_FRESH_TEST =', repr(mask) if mask else 'unset', flush=True)
    command = [str(PYTHON), '-u', str(ROOT / 'tutorial_stages.py'), '--root', str(ROOT), '--outputs', str(OUTPUTS), '--weights', str(WEIGHTS), '--stage', stage, '--options', json.dumps(options)]
    print('Running stage', repr(stage), 'in the isolated environment; log:', log, flush=True)
    with log.open('w', encoding='utf-8') as output:
        process = subprocess.Popen(command, env=env, cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            output.write(line)
            output.flush()
            if line.strip() and len(line) < 4000:
                print(line.rstrip(), flush=True)
        process.wait()
    if process.returncode:
        detail, error = 'see the log above', {}
        if error_file.is_file():
            try:
                error = json.loads(error_file.read_text(encoding='utf-8'))
                detail = error['type'] + ': ' + error['message']
            except (ValueError, KeyError):
                error = {}
        if error.get('refusal'):
            name, reason = error.get('name') or '(no file)', error['message']
            if reason.startswith(name + ': '):
                reason = reason[len(name) + 2:]
            again = 're-run the Section 4 cell' if stage in ('data', 'validate') else 'run this cell again'
            raise Refused(name, f'{reason} Nothing from the refused input was kept: fix it and {again}.') from None
        raise RuntimeError(f'Stage {stage!r} failed (exit {process.returncode}): {detail}')


def load_record(name):
    """A JSON record a stage wrote to OUTPUTS."""
    return json.loads((OUTPUTS / name).read_text(encoding='utf-8'))

## 3. Pin, stage and verify the model · [Engineering]

> **Infrastructure.** The next code cell is collapsed. It downloads one file and checks it; you may run it without studying its implementation.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the carried package and the 1-file manifest (path, byte size, SHA-256) — and the `weights` stage first checks that they agree. `stage_missing_files(..., allow_download=True)` then fetches exactly the absent entries from the Hugging Face Hub **at revision `4dcd344ece2c…`** (never `main`, ~110 MB), and `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch. There is no fallback to a different download and no remote model code is executed.

**What to notice:** the model id, revision, licence and file count; `fetched` lists the checkpoint on a first run and is empty on a rerun (staging only fetches a missing file); then `verified_files`. A mismatch stops the cell with an error naming the file — see **Troubleshooting**, and never edit a manifest to get past one.

In [ ]:
# @title Infrastructure: stage and digest-verify the pinned checkpoint
run_stage('weights')

## 4. Load the sample or your own data · [Concept]

From here on, every code cell runs one stage of the carried runner with `run_stage`. The default, `Sample: MAGIC Gamma Telescope`, is the UCI MAGIC table [3] (19,020 simulated telescope events, 10 numeric features, two classes: `g` gamma, `h` hadron, CC BY 4.0). The repository commits it as a pinned split archive (`examples/sample-data/magic-gamma.zip`, fetched by immutable commit and digest-verified): **3,000 support rows, 1,000 validation rows, and a 1,500-row fresh test** that was carved first and has never been fitted, predicted or scored by anyone. The 13,520 rows left over are an untouched reserve. This stage reads only the support and validation CSVs; the fresh test is read later by one stage only, under the pre-registered settings (Section 7b).

The other samples are exploratory: `Sample: Breast Cancer` is the **earlier default**, kept as a labelled earlier-run reference (its 114-row test was scored in recorded runs, so here it is an *exposed development test*); `Sample: Wine` is a small three-class table; `Sample: Palmer Penguins` has mixed categorical/numeric features. With `USE_BYOD = True`: `Upload CSV` takes one CSV and carves a stratified 80/20 validation split; `Upload pre-split train/val/test` takes your own partitions (`test` optional). Give files by path, or leave the path fields empty in Colab to choose them in upload dialogs. Any non-default source makes the run **exploratory**: it never scores the fresh test.

The stage refuses, naming the file and the rule, duplicate header names, a `TARGET_COLUMN` absent from the header, and a column that is numeric except for a few stray strings (list it in `TEXT_COLUMNS` if it really is categorical). A refusal is one message, not a traceback, and the stage first removes this notebook's earlier exports from `outputs/`, so a refused input never leaves an older result looking current. Rows whose label is missing are **dropped and counted** in Section 5's input manifest.

**BYOD privacy boundary.** Uploaded CSV bytes are read inside the current notebook runtime and are not sent by this notebook to an external inference or training service; the network requests on the default path are the pinned checkpoint download of Section 3 and the pinned MAGIC archive.

In [ ]:
DATA_SOURCE = 'Sample: MAGIC Gamma Telescope'  # @param ["Sample: MAGIC Gamma Telescope", "Sample: Breast Cancer", "Sample: Wine", "Sample: Palmer Penguins", "Upload CSV", "Upload pre-split train/val/test"]
USE_BYOD = False  # @param {type:"boolean"}
TARGET_COLUMN = 'target'  # @param {type:"string"}
BYOD_CSV_PATH = ''  # @param {type:"string"}
BYOD_TRAIN_PATH = ''  # @param {type:"string"}
BYOD_VAL_PATH = ''  # @param {type:"string"}
BYOD_TEST_PATH = ''  # @param {type:"string"}
TEXT_COLUMNS = []  # @param {type:"raw"}

def refuse(name, message):
    # DAT20: one message, no traceback; the earlier run is invalidated so nothing stale reads as current.
    (ROOT / 'state' / 'data.json').unlink(missing_ok=True)
    for stale in OUTPUTS.glob('tabiclv2_classifier_*'):
        if stale.is_file():
            stale.unlink()
    raise Refused(name, message)

def upload_files(what):
    try:
        from google.colab import files
    except ImportError:
        refuse('(no file)', f'No path was given for {what}, and the upload dialog exists only in Google Colab: set the BYOD path field(s), then run this cell again.')
    uploaded = files.upload()
    if not uploaded:
        refuse('(no file)', f'The upload was cancelled or empty: no file was received for {what}. Run this cell again, or set the path field(s).')
    saved = {}
    for upload_name, payload in uploaded.items():
        path = ROOT / 'inputs' / Path(upload_name).name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(payload)
        saved[path.name.lower()] = str(path)
    return saved

paths = {}
if USE_BYOD and DATA_SOURCE == 'Upload CSV':
    if BYOD_CSV_PATH:
        paths = {'csv': BYOD_CSV_PATH}
    else:
        saved = upload_files('one labelled CSV')
        if len(saved) != 1:
            refuse(', '.join(sorted(saved)), f'Upload exactly one CSV; got {len(saved)} files. Run this cell again.')
        paths = {'csv': next(iter(saved.values()))}
elif USE_BYOD and DATA_SOURCE == 'Upload pre-split train/val/test':
    if BYOD_TRAIN_PATH or BYOD_VAL_PATH:
        paths = {'train': BYOD_TRAIN_PATH, 'val': BYOD_VAL_PATH, 'test': BYOD_TEST_PATH}
    else:
        saved = upload_files('train.csv and val.csv (test.csv optional)')
        if not {'train.csv', 'val.csv'} <= set(saved):
            refuse(', '.join(sorted(saved)), 'Upload train.csv and val.csv (test.csv optional). Run this cell again.')
        paths = {'train': saved['train.csv'], 'val': saved['val.csv'], 'test': saved.get('test.csv', '')}
run_stage('data', data_source=DATA_SOURCE, use_byod=USE_BYOD, target_column=TARGET_COLUMN, paths=paths, text_columns=TEXT_COLUMNS)

**What to notice:** the `fresh_test_mask` line (printed before anything is fetched); `sample_kind: 'sample'`, 3,000 support and 1,000 validation (`holdout`) rows, `test_rows: 0` because the fresh test is not loaded here, and the archive and row-id digests. Every later number is measured on these exact partitions.

## 5. Validate the tables → input manifest, then encode · [Evaluation practice]

`validate_inputs` is the package's public validation stage: it applies exactly the checks `prepare_classification_table` applies — unique column names, the target present, missing-target rows dropped with the count recorded, at least `MIN_TRAIN_ROWS` support rows (`MIN_EVAL_ROWS` for a validation partition), at most `MAX_TRAIN_ROWS` rows and `MAX_FEATURES` columns, at least `MIN_CLASSES` classes — and returns an **input manifest** naming the schema, the observed structure and the verdict. It is written to `outputs/tabiclv2_classifier_input_manifest.json`. To show what rejection looks like, the stage also validates a probe with too few rows and records the package's own error message as a finding. The ceilings and the decision rule are printed before any model runs.

The validation partition is then aligned to the support schema **and to the support class set** (`align_to_schema` refuses a class the model never saw), the categorical encoder is fitted on the support split only and applied everywhere (unseen values counted), and the trivial baseline is computed with `majority_class_baseline`.

In [ ]:
run_stage('validate')

**What to notice:** the ceilings; the manifest entry for the support partition (10 numeric features, no categorical columns, two classes, a majority share near 0.65); the finding `too-few-rows-probe` rejected with the package's message; `dropped_missing_target_rows` all zero; and the majority-class baseline on validation.

**Checkpoint:** a BYOD CSV has 7 rows with no label. Where does that number appear, and why must it?

<details>
<summary>Check your reasoning (open after answering)</summary>

In the input manifest, as `dropped_missing_target_rows_before_split` on the support entry (and in the printed `dropped_missing_target_rows`). Those rows cannot be used, but dropping them silently would change the data you think you evaluated. Recording the count makes the change visible and auditable.

</details>

## 6. Condition TabICLv2, then run the pre-registered fine-tuning grid (GPU) · [Concept]

`fit` does not train anything; it registers the 3,000 support rows as the model's context. `classification_metrics` scores validation: **accuracy**, **balanced accuracy** and **weighted F1** are discrete correctness under the implicit `argmax` rule; **log loss** scores the class probabilities and is the **primary metric** here, because it registers changes that accuracy cannot; **ROC-AUC** scores ranking quality. Probabilities are renormalised row-wise (TabICL's reduced-precision ensemble average misses 1 by rounding only).

**Fine-tuning a strong in-context model often does not help.** TabICLv2 already reads the support rows at prediction time; gradient fine-tuning can only add something the context does not already give it, and with a few thousand rows **gradient fine-tuning** easily overfits. This section measures that honestly instead of assuming a gain. Three configs were **pre-registered** before anything was run (`docs/magic-finetune-preregistration.md`): **C1** production-like (learning rate 1e-5, up to 30 epochs, patience 8, the `dimer-pipeline.json` defaults), **C2** larger steps (5e-5) and **C3** smaller steps (1e-6), each up to 10 epochs with patience 3. All of them take **real optimizer steps**: `max_data_size=500` splits the 3,000 support rows into 6 chunks, so each epoch is 6 AdamW steps, and a constant learning rate with no warmup means no step runs at learning rate 0. (The earlier recipe of this notebook fitted the whole support set in one chunk, so an epoch was one step, and its first step ran at learning rate 0.)

**How tabicl keeps weights.** It measures the validation log loss of the pretrained weights first (the *baseline*), then after every epoch. An epoch's weights are kept only if they beat the best value so far by `min_delta` (1e-4); after `patience` epochs without improvement it stops and **restores the best state**, which is the pretrained weights if no epoch improved, and writes it as `best.ckpt`. The stage prints each config's validation curve, its optimizer steps, and a **parameter delta** (the L2 norm of the weight change): 0 means tabicl gave back the pretrained weights. It also reads out the **last epoch's** weights on validation, as a disclosed exploratory readout, so you can see what the gradient steps did.

TabICL writes one checkpoint per epoch; the stage keeps `best.ckpt` only (of the selected config). A fine-tuned `best.ckpt` may remain larger than the base checkpoint because upstream training state is embedded; the format is kept for checkpoint compatibility.

**Selection (pre-registered, validation only).** The pretrained model is kept unless a config beats its validation log loss by more than **0.005**; the fresh test plays no part.

**Reproducibility boundary.** The stages set `CUBLAS_WORKSPACE_CONFIG=:4096:8`, `cudnn.deterministic`, no `cudnn.benchmark` and `torch.use_deterministic_algorithms(True)` with fixed seeds; fp16 autocast runs with float32 weights. An operation without a deterministic CUDA kernel stops the stage and names itself. Attention runs on PyTorch's **MATH** kernel with tabicl's FlashAttention-3 path off, in every stage that runs the model (pre-registration amendment 2: two masked T4 runs that let PyTorch choose its memory-efficient kernel diverged at the first backward step), and any warning about a non-deterministic kernel is printed as `DETERMINISM WARNING`. The printed **grid hash** is informational: on a T4 the fp16 grid has given at least two different hashes for the same notebook revision (issue #28; hosted runs at commits `d4172f5` and `edbe94d`, same notebook and stage runner), while the selection and every fresh-test value matched exactly. So the grid hash is printed and reported, never compared to decide a pass.

**Form fields.** Leave them at their defaults for the pre-registered run. Changing any of them (a smaller `GRID_EPOCHS_CAP` to save time, a `GRID_LEARNING_RATE` override, another `RANDOM_STATE`), unticking `RUN_FINE_TUNING`, BYOD, another data source or a non-T4 runtime makes the run **exploratory**: validation only, and the fresh test is never read. On a CPU runtime the grid is skipped with a printed note.

**Predict before running:** will any config beat the pretrained model on validation log loss by more than 0.005? What will the parameter delta of `best.ckpt` be if none does?

In [ ]:
RUN_FINE_TUNING = True  # @param {type:"boolean"}
GRID_EPOCHS_CAP = 0  # @param {type:"integer"}
GRID_LEARNING_RATE = 0.0  # @param {type:"number"}
RANDOM_STATE = 42  # @param {type:"integer"}
run_stage('condition', run_fine_tuning=RUN_FINE_TUNING, grid_epochs_cap=GRID_EPOCHS_CAP, grid_learning_rate=GRID_LEARNING_RATE, random_state=RANDOM_STATE)

**What to notice:** `preregistered` and `settings_differing` at the top; the pretrained validation metrics; for each config its `val_log_loss_per_epoch` against `baseline_val_log_loss`, the optimizer steps it attempted, took and skipped (`optimizer_steps_attempted` / `optimizer_steps_taken` / `amp_skipped_steps`; fp16 on the GPU can skip a step whose gradients overflow), `weights_changed` and the two parameter deltas; `recommended_for_export` with its `selection_basis`; the `exploratory_readout`; and the grid hash. On a CPU runtime: `fine_tune.skipped` with the reason.

For orientation only — the **CPU development run** of this exact grid (2026-10-19, CPU, validation only, fresh test masked, grid hash `11cb0310…`; recorded in `docs/release-verification.md`, not a run of this notebook on a T4): the pretrained model scored validation log loss 0.2634 and accuracy 0.903; against tabicl's baseline of 0.2701 (measured with 2 ensemble members), C1 ran 8 epochs at 0.2712–0.2762, C2 ran 3 epochs at 0.2864 / 0.3340 / 0.3027, and C3 ran 3 epochs at 0.2702 / 0.2701 / 0.2700 — within ±0.0001 of the baseline. **No epoch beat the baseline by more than tabicl's `min_delta` (1e-4)**, so tabicl kept the pretrained weights in every config, and no config beat the pretrained model by the pre-registered 0.005 tolerance.

The **hosted run of record** (2026-10-20, Tesla T4, commit `9ea0888`, grid hash `4b1bf536…`; recorded in `docs/release-verification.md`) differed from the CPU run in one config: on the T4, C3 (learning rate 1e-6) ran all 10 epochs (60 optimizer steps taken, 0 skipped), its weights changed (`weights_changed: True`, parameter delta 0.059), and its validation log loss ended at 0.2628 against the pretrained 0.2634 — an improvement of about 0.0006, far below the 0.005 tolerance. No config beat the pretrained model by more than 0.005, so the pretrained model was kept. A T4 run of this notebook prints a grid hash that need not equal the record's, even at the same revision (issue #28); its selection and fresh-test results should match the record. Read your own run's numbers.

**Checkpoint:** suppose a config reports `weights_changed: False` while its last-epoch parameter delta is not 0. What happened? And what would it mean if a config instead reported `weights_changed: True`?

<details>
<summary>Check your reasoning (open after answering)</summary>

`weights_changed: False` with a non-zero last-epoch delta: the gradient steps did move the weights, but no epoch's weights beat the validation baseline by more than `min_delta` — some were worse, some (like C3 in the CPU development run) were within a ten-thousandth of it — so tabicl's early stopping restored the pretrained state and wrote it as `best.ckpt`. That is the procedure working as designed: selection on held-out data keeps an adaptation only when it measurably helps. A restored config can never be selected as "fine-tuned".

`weights_changed: True`: some epoch did beat the baseline by more than `min_delta`, and tabicl kept those weights (C3 on the hosted T4 run of record is this case). That is still not enough to be selected: the config must also beat the pretrained model's validation log loss by more than 0.005. Either way the validation partition chose, and the fresh test, untouched by that choice, says how the chosen model does on new data. A strong in-context model often leaves little for fine-tuning to add on a few thousand rows; a null result is a finding, reported as one.

</details>

## 7. Classical baselines, blending and the evaluation report · [Evaluation practice]

LightGBM and a random forest are fitted on the exact same encoded support rows and scored on the exact same validation partition, so the comparison is fair. Their probability columns are mapped onto the TabICLv2 class order with `align_probabilities` **before** any blending. A convex blend of TabICLv2 and LightGBM probabilities is chosen by maximising validation accuracy over a 101-point grid (ties broken toward the foundation model); it is evaluated in memory only, its tied weight range is printed, and the exported bundle in Section 9 is the unchanged selected model. Each model is reported with its validation error count beside the one-row resolution ($1/N$). Latencies are medians of five warmed runs on this runtime.

`evaluation_report` carries the selected model's validation metrics, the majority-class baseline, the classical baselines, the error counts, the blend, the whole fine-tuning grid and its hash, with the verdict `sample-sanity`. It is written to `outputs/tabiclv2_classifier_evaluation_report.json`. The MAGIC fresh test is **not** read here (Section 7b). For the exploratory samples their test partition is printed with its label (for Breast Cancer: *exposed development test*).

**Predict before running:** on 1,000 validation rows one row is 0.001 of accuracy. Can a blend chosen on those rows be trusted to beat TabICLv2 by a few rows?

In [ ]:
run_stage('report')

**What to notice:** `one_row_resolution_pct: 0.1`; the TabICLv2, LightGBM and random-forest lines on validation with their error counts; the blend's tied weight range; and the report's `verdict: sample-sanity`.

**Checkpoint:** the blend scores a few rows better than TabICLv2 on validation. What do you report?

<details>
<summary>Check your reasoning (open after answering)</summary>

That the blend's validation gain is selection-biased: its weight was chosen to maximise exactly that number. Only data untouched by the choice can say whether it helps, and the fresh test is reserved for the pre-registered model comparison, not for tuning a blend. Report the blend as an in-memory demonstration and export the selected model.

</details>

## 7b. The fresh test: scored only under the pre-registered settings · [Evaluation practice]

The 1,500 fresh-test rows answer one pre-registered question: **how does the selected model compare with the pretrained model on rows that no choice has seen?** The primary endpoint is the fresh-test **log loss difference Δ = selected − pretrained**, with a paired bootstrap 95% interval over rows (seed 20261019, 10,000 resamples). Verdict: *improved* if the interval's upper bound is below 0 and Δ ≤ −0.005; *worse* if its lower bound is above 0; otherwise *no detectable difference*. **If the pretrained model was selected, the verdict is "same model (Δ = 0)"**: no difference was measured, because none can exist. Accuracy (Wilson interval) and ROC-AUC are secondary, and a sensitivity readout repeats Δ without the 4 fresh-test rows that have an exact feature duplicate in the support or validation rows (3 and 1). Power, from the dataset research's planning estimates: about ±0.017 on a single accuracy, about ±0.011 on a paired accuracy difference, about ±0.017 on a paired log-loss difference.

**Score-once, fail-closed.** Before reading the test the stage recomputes, from the recorded settings, that the run is `preregistered` (every fixed setting, BYOD off, a Tesla T4), that the grid is complete and its hash matches, that the selected checkpoint's digest is the one Section 6 recorded, and that the archive digest matches. If any setting differs it writes `outputs/fresh-test/tabiclv2_classifier_fresh_test_exploratory_<n>.json` (`test_look: not scored (exploratory)`, with the differing settings) and reads nothing. The evaluation of record already exists — the first pre-registered scoring, on a hosted T4 on 2026-10-20 (commit `9ea0888`), labelled **"pre-registered evaluation (this runtime)"** — and this notebook carries it, so every pre-registered scoring now, the first in a new runtime included, writes `_reproduction_<n>`, is labelled **"reproduction"** and prints `matches_record` with the fields that `differs` from the record. The selection, the device and every fresh-test value must match exactly. The grid hash is printed as `grid_hash` with the record's value, this run's value and whether they are `equal`, labelled **informational**: it never decides `matches_record`, whatever the stage runner (issue #28), and neither does the printed `stage_runner_sha256`. `docs/release-verification.md` lists the record and its reproductions. Set the environment variable `DIMER_MASK_FRESH_TEST=1` (for example `%env DIMER_MASK_FRESH_TEST=1`) to skip the fresh test entirely; it is re-read on every stage call. The mask stops the fresh test from being read, not from being downloaded: the pinned archive holds all three CSVs, so the data stage still fetches and caches it (digest-checked), but `test.csv` is never extracted, parsed or scored under the mask.

**Predict before running:** if Section 6 kept the pretrained model, what will Δ and its interval be?

In [ ]:
run_stage('fresh_test')

**What to notice:** the mask line; `test_look`; `selected`; the primary endpoint with its interval and verdict; the secondary accuracy and ROC-AUC; and the sensitivity readout. On an exploratory run: the list of `settings_differing` and no test numbers at all.

**Checkpoint:** why does the notebook refuse to score the fresh test when you change only `GRID_EPOCHS_CAP`?

<details>
<summary>Check your reasoning (open after answering)</summary>

Because a test you can re-score under any setting stops being a test: try enough settings, keep the best number, and the result is chosen on the test. Fixing every setting in advance and scoring only that combination keeps the fresh-test number an honest estimate. Exploration belongs on validation.

</details>

## 8. New-data inference with class probabilities · [Engineering]

By default the `predict` stage scores the first eight validation rows — already scored in Section 7, so this demonstrates the output contract, not a new evaluation — and writes their unlabelled copy, with their original `row_id`, to `outputs/tabiclv2_classifier_new_rows.csv`. Tick `RUN_NEW_DATA_INFERENCE` and set `NEW_DATA_PATH` (or, in Colab, leave it empty to upload) to score your own unlabelled CSV: `read_inference_csv` and `validate_inputs(..., target_column=None, feature_columns=...)` require a unique header, no pre-existing `prediction`/`probability_*` columns and every feature present; extra columns are kept in the output and not passed to the model. The output adds `prediction` (the `argmax` label) and one `probability_<class>` column per class **in the fitted class order**; the probabilities are uncalibrated and no threshold is shipped. `outputs/tabiclv2_classifier_result.json` records the predictions, metrics, baselines, the fine-tuning record, the evaluation report, the input manifest, the sample digest, the notebook's source, the model identity, revision and licence, and the runtime identity.

In [ ]:
RUN_NEW_DATA_INFERENCE = False  # @param {type:"boolean"}
NEW_DATA_PATH = ''  # @param {type:"string"}

new_data_file = ''
if RUN_NEW_DATA_INFERENCE:
    new_data_file = NEW_DATA_PATH or next(iter(upload_files('one unlabelled CSV').values()))
run_stage('predict', new_data_path=new_data_file)

**What to notice:** eight rows with their `row_id`, the `prediction` and one `probability_<class>` column per class, summing to 1 per row (they are renormalised after the deviation is checked).

## 9. Export a DIMER-style serving bundle, then prove a fresh reload · [Engineering]

The ZIP carries the minimum serving contract the DIMER pipeline expects: `checkpoints/best.ckpt`, `training_context.parquet`, and `artifact.json`. The training context is *required*: TabICL is still an in-context learner at serve time, so whoever loads the bundle must hand the model the same rows you evaluated with, and the context inherits the source data's licensing obligations (MAGIC: CC BY 4.0, attribution required). `artifact.json` records the feature and target columns, the class labels, the base-model identity and digest, the selected mode and its basis, the metrics, the inference settings, per-file sizes and SHA-256 digests, the payload allowlist and the producer runtime. The `export` stage prints the ZIP's SHA-256 — the **trusted digest** to hand over with the ZIP, which whoever loads it checks before opening it (Section 10 shows that check on an artifact made elsewhere).

The `reload` stage then runs in a **fresh process**: it extracts the ZIP with `safe_extract_zip`, verifies the allowlist, sizes and digests with `verify_artifact_bundle`, rebuilds the classifier from the bundle alone, and checks that its labels equal and its probabilities agree with the exporting process's on eight validation rows (`rtol=1e-5`, `atol=1e-7`).

**Predict before running:** if Section 6 kept the pretrained model, which checkpoint is in the bundle, and what does Section 10 check about such a checkpoint before it loads one?

In [ ]:
run_stage('export')
run_stage('reload')

**What to notice:** the ZIP path, its `zip_sha256`, the `mode` and the checkpoint size; then `labels_identical: True` and the `PASS` line.

**Checkpoint:** the bundle's checkpoint is code-capable (a PyTorch/Lightning checkpoint). What protects the person who loads it?

<details>
<summary>Check your reasoning (open after answering)</summary>

For a `pretrained` bundle, the external-artifact section (Section 10) requires the checkpoint's SHA-256 to equal the pinned base checkpoint's, so it is byte for byte the file the model's publisher released and Section 3 verified. For a `fine-tuned` bundle no such reference exists: the only protection is that the ZIP's digest matches a trusted digest received from the producer through a separate channel (`EXPECTED_ZIP_SHA256`). So: load fine-tuned bundles only from producers you trust, with their printed digest.

</details>

## Interpretation and limits

`prediction` is the `argmax` of uncalibrated class probabilities over the fitted class set; the package ships no threshold, and any deployment cut-off must be chosen on the caller's own labelled, domain-representative data. The evaluation report's `sample-sanity` verdict names what it is: one seeded split of a public sample with no dispersion estimate. MAGIC is a simulated physics table in the TALENT benchmark, which the TabICLv2 paper evaluated on [1]; TabICLv2 is pretrained on synthetic tables only, so that is evaluation exposure, not training leakage. It is not in TabArena or OpenML-CC18. The fine-tuning selection and the blend weight are chosen on validation and are selection-biased; only the fresh test, scored under the pre-registered settings, is an unbiased estimate, and its recorded evaluation and reproductions are listed in `docs/release-verification.md`. The evaluation of record (2026-10-20, hosted T4) kept the pretrained model, so its fresh test reads "same model (Δ = 0)": accuracy 0.8993 (Wilson 95 % 0.8831–0.9136) and ROC AUC 0.95 on 1,500 rows. A null fine-tuning result here is evidence about this model on this table, not a general rule.

**Earlier-run reference (Breast Cancer, previous recipe).** In the Colab T4 run of blob `051c81e9` (2026-10-10, recorded in `docs/release-verification.md`), the earlier default — the Breast Cancer table with a 114-row holdout and a 114-row test — gave the pretrained model holdout accuracy 0.9825 (112/114); the old three-epoch fine-tune took one non-zero step and was discarded by tabicl, so the candidate equalled the pretrained model; LightGBM scored 0.9649 and a random forest 0.9561 on that holdout. That test was scored in recorded runs and is an exposed development test; no setting of this notebook was chosen on it. These numbers are history, not results of this revision.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned checkpoint and data, validate the demonstrated tables with class coverage, condition on support rows, run a pre-registered fine-tuning grid with validation-based selection on a GPU, score the fresh test only under the pre-registered settings, compute sample metrics against trivial and classical baselines, score rows with class probabilities under an explicit `argmax` rule, export a DIMER-style serving bundle and rebuild an equivalent classifier from that bundle alone in a fresh process, and validate, rebuild and serve a pinned artifact that an earlier recorded run produced — without the repository being reachable. It does **not** establish benchmark superiority, domain generalisation, fairness, robustness, probability calibration, production safety, or deployment fitness.

## Conclusion · [Evaluation practice]

Write three to five sentences, using the numbers your run printed:

1. **Result:** the selected model's validation accuracy and log loss beside the majority baseline, LightGBM and the random forest.
2. **Adaptation:** what each grid config did on validation (curve, steps, parameter delta), and which model was selected and why.
3. **Fresh test:** the primary endpoint with its interval and verdict, and its `test_look`.
4. **Reuse and limits:** what the Section 9 reload proved, what Section 10 (next) adds with an artifact made elsewhere, and the one limitation you would fix first.

<details>
<summary>Sample conclusion (open after writing yours)</summary>

A sample of the reasoning, written for the case the hosted run of record showed (2026-10-20, T4); replace it with your own numbers: on MAGIC's 1,000 validation rows the pretrained TabICLv2 model already beat LightGBM and the random forest on log loss (0.2634 against 0.3412 and 0.3234). Fine-tuning moved the weights in one config — C3 at learning rate 1e-6 lowered validation log loss by about 0.0006 — but no config beat the pretrained model by the 0.005 tolerance, so the pretrained model was kept: the fresh test then reports "same model (Δ = 0)" (accuracy 0.8993, Wilson 95 % 0.8831–0.9136) — nothing to compare, which is itself the finding that fine-tuning this in-context model on 3,000 rows did not measurably help.If instead a config was selected, report the fresh-test Δ with its interval and verdict, and say whether the interval excludes 0. The bundle rebuilt an identical classifier in a fresh process. Before deployment I would evaluate on a domain-representative table and calibrate the probabilities.

</details>

## 10. Using an artifact made elsewhere · [Engineering]

So far this notebook has scored data with the model it adapted in this run, and the fresh reload showed that this run's export survives serialization. This section does something different: the reload shows that this run's export survives serialization, and this section shows that an artifact made elsewhere can be validated and served here. It downloads a pinned artifact that an earlier, recorded run of this notebook produced, verifies its SHA-256 before opening it, shows its provenance and format, rebuilds serving state from the files alone, and scores sample rows that follow the same data contract as the rest of this notebook. **This run's own export (Section 9) is never used here.**

**This section needs only the setup cells (Sections 1–3).** It reads no partition, encoder, model or file that Sections 4–9 created, so if you already hold a compatible artifact you can run Sections 1–3 and then this section alone. Its outputs go to `outputs/external_artifact/`, apart from the main path's files.

**The pinned sample artifact.** By default the `external_artifact` stage downloads `tabiclv2_classifier_artifact.zip` from this repository's GitHub release `sample-artifact-magic-v1` (no account or token). It is the deterministic export of a recorded hosted T4 run of this notebook's default path (the MAGIC sample) at commit `383ca04`; the release notes name the run record, and the asset is never replaced (a changed artifact gets a new tag). The stage prints the URL, the pinned SHA-256, the producer and the format `tabicl-dimer-classifier-v1` **before** it downloads anything, then refuses the file unless its SHA-256 equals the pinned value. Only then does `safe_extract_zip` copy the members one by one (no absolute paths, `..`, symlinks or oversize archives); `validate_artifact_runtime` checks the format and the `tabicl`/`torch` versions; the manifest must name the pinned base checkpoint; `verify_artifact_bundle` checks the member allowlist, sizes and digests; and the manifest's features, target and class labels must equal the MAGIC default path's. Any failed check stops the run, and no prediction file is written.

**Trust boundary.** The artifact's `checkpoints/best.ckpt` is deserialised by `tabicl` (a PyTorch/Lightning checkpoint, which can carry code), so loading an artifact means trusting its producer. The pinned SHA-256 binds this download to the file the maintainer published; for a `pretrained` artifact the checkpoint must also be, byte for byte, the base checkpoint whose digest Section 3 verified. Nothing is deserialised before these checks pass. For a `fine-tuned` artifact of your own, the only binding is a digest you received from its producer through a separate channel (`EXPECTED_ZIP_SHA256`).

**Optional: your own artifact (off by default).** Set `ARTIFACT_SOURCE` to `My own artifact ZIP` and give `ARTIFACT_ZIP_PATH` (or leave it empty in Colab to upload) and the producer's `EXPECTED_ZIP_SHA256`. Without a digest the checks establish internal consistency only, and the stage says so.

In [ ]:
ARTIFACT_SOURCE = 'Pinned sample artifact'  # @param ["Pinned sample artifact", "My own artifact ZIP"]
ARTIFACT_ZIP_PATH = ''  # @param {type:"string"}
EXPECTED_ZIP_SHA256 = ''  # @param {type:"string"}


def external_upload(what):
    """Colab upload for the optional branches of this section; elsewhere, set the path field."""
    try:
        from google.colab import files
    except ImportError as exc:
        raise Refused('(no file)', f'No path was given for {what}, and the upload dialog exists only in Google Colab: set the path field, then run this cell again.') from exc
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise Refused(', '.join(sorted(uploaded)) or '(no file)', f'Expected exactly one file for {what}; received {len(uploaded)}. Run this cell again.')
    ((upload_name, upload_payload),) = uploaded.items()
    upload_path = ROOT / 'inputs' / Path(upload_name).name
    upload_path.parent.mkdir(parents=True, exist_ok=True)
    upload_path.write_bytes(upload_payload)
    return str(upload_path)


if ARTIFACT_SOURCE == 'Pinned sample artifact':
    run_stage('external_artifact', source='pinned')
else:
    run_stage('external_artifact', source='own', zip_path=ARTIFACT_ZIP_PATH or external_upload('your artifact ZIP'), expected_zip_sha256=EXPECTED_ZIP_SHA256)

**What to notice:** the provenance line is printed before the download; then `trusted_digest: verified (pinned SAMPLE_ARTIFACT_SHA256)`, the `mode`, the `checkpoint_binding` and the `data_contract`; the producer's runtime and notebook source; and the artifact's 10 MAGIC features, its `target` column and the class labels `g` and `h`.

**Reconstruct serving state locally.** The `external_reconstruct` stage rebuilds the classifier from the artifact's files alone, in a new process: its checkpoint (with `allow_auto_download=False`), its `training_context.parquet` — the fitted support rows TabICL conditions on — and the categorical encoders recorded in `artifact.json`. Nothing is refitted from the rows you are about to score.

In [ ]:
run_stage('external_reconstruct')

**Score sample rows, or your own.** By default the `external_predict` stage takes the first eight rows of the pinned MAGIC validation file, removes their label, and scores them: they follow the artifact's data contract and are not among its support rows (the 1,500-row fresh test is never read). The rows pass the same `read_inference_csv` / `validate_inputs` checks as Section 8, a probe with a missing feature must be refused, and non-numeric values in a numeric feature are refused naming the column. The predictions, the input manifest and the artifact's provenance go to `outputs/external_artifact/tabiclv2_classifier_external_predictions.csv` and `outputs/external_artifact/tabiclv2_classifier_external_result.json`.

**Optional: your own rows (BYOD, off by default).** Tick `USE_OWN_ROWS` and set `EXTERNAL_ROWS_PATH` (or leave it empty in Colab to upload) to an unlabelled CSV with the artifact's feature columns. With your own artifact, your own rows are required: the MAGIC rows follow only the pinned artifact's contract.

In [ ]:
USE_OWN_ROWS = False  # @param {type:"boolean"}
EXTERNAL_ROWS_PATH = ''  # @param {type:"string"}

external_rows = ''
if USE_OWN_ROWS:
    external_rows = EXTERNAL_ROWS_PATH or external_upload('one unlabelled CSV')
run_stage('external_predict', rows_path=external_rows)

**What to notice:** eight rows with `row_id`, `prediction` and `probability_g` / `probability_h` summing to 1; the two output paths under `outputs/external_artifact/`; and the artifact's `zip_sha256`, equal to the pinned digest.

**Checkpoint:** if you set `ARTIFACT_SOURCE` to this run's own `outputs/tabiclv2_classifier_artifact.zip`, would that be external-artifact evidence?

<details>
<summary>Check your reasoning (open after answering)</summary>

No. An artifact this run created and then reloaded is a fresh-boundary test, which Section 9 already performed. External evidence needs an artifact made outside this run with documented provenance, which is why the default is a pinned release asset from an earlier recorded run, checked by its SHA-256 before it is opened.

</details>

## 11. Optional activity: a three-class table with 36 holdout rows · [Concept]

**Predict → Change → Run → Observe → Explain.** **Predict:** on `Sample: Wine` (178 rows, three classes, a 60/20/20 split leaves 36 holdout rows), what is the one-row resolution, and will one-vs-rest ROC-AUC be more or less informative than accuracy? **Change:** tick `RUN_ACTIVITY` (keep `ACTIVITY_DATA_SOURCE = 'Sample: Wine'`). **Run** this cell. **Observe** the activity record beside the canonical one. **Explain** what 36 rows can and cannot show. The activity runs the pretrained model only, writes to `outputs/activity/`, and stops if any canonical output changed.

In [ ]:
RUN_ACTIVITY = True  # @param {type:"boolean"}
ACTIVITY_DATA_SOURCE = 'Sample: Wine'  # @param ["Sample: Wine", "Sample: Palmer Penguins"]
if RUN_ACTIVITY:
    run_stage('activity', data_source=ACTIVITY_DATA_SOURCE)
else:
    print('Optional activity skipped: tick RUN_ACTIVITY to run it. The canonical outputs are complete.')

**What to notice (if you ran it):** `one_row_resolution` 0.0278 against the canonical 0.001; the majority baseline near 0.39; TabICLv2's accuracy and ROC-AUC on 36 rows. `canonical_outputs_unchanged: True`.

<details>
<summary>Check your reasoning (open after running)</summary>

With 36 rows one mistake moves accuracy by 2.8 points, so models that differ by one row are indistinguishable. ROC-AUC uses the full probability ranking, so it can separate models whose `argmax` labels agree, but on a table this easy it saturates near 1 too. Compare MAGIC's 1,000 validation and 1,500 fresh-test rows: that is why the default table changed.

</details>

## Troubleshooting · [Engineering]

| Symptom | Likely cause | What to do |
|---|---|---|
| Section 1 stops with `This notebook needs a Linux x86_64 runtime` | a local Windows or macOS kernel, or an ARM machine | Use Google Colab, Kaggle, or a Linux x86_64 Jupyter kernel. |
| `No CUDA GPU detected`; later `Fine-tuning skipped: no CUDA device` | a CPU runtime | Everything else runs, as an exploratory run. For the documented runtime choose *Runtime → Change runtime type → T4 GPU*. |
| `Not enough free disk` | the isolated environment needs about 8 GB | Start a fresh runtime; an environment built from the same lock is reused. |
| `Carried file integrity failure` | a carried file was edited in the notebook | Open a fresh copy from the repository. |
| `uv … mismatch`, `URLError`, or `CalledProcessError` from `uv` | network or a transient PyPI error | Re-run the Section 2 install cell. Never remove a pin or a hash. |
| `The run directory … has no carried files, or the isolated environment is gone` | Section 1 run with `NEW_RUN_DIRECTORY` ticked | Run Sections 1–3 again, or *Run all*. |
| `RuntimeError: Stage '…' failed (exit 1): …` | the stage's own error follows the colon | Find it below; fix the cause and re-run from that cell. |
| `Refused: <file name>: …` | an input was refused (one line, no traceback; your data, rows or artifact ZIP) | Fix the named file or field and run that cell again (for Section 4 data, re-run the Section 4 cell); nothing from the refused input was kept. |
| A Hub download error in Section 3, or `… sha256 … != manifest` | a transient failure or a corrupted download | Re-run Section 3; delete the partial file under `weights/` if the digest fails. Never edit the manifest. |
| `MAGIC sample archive digest mismatch` / `Palmer Penguins sample archive digest mismatch` | the pinned archive could not be fetched intact | Retry; never change the pinned digest. |
| `Trusted digest mismatch` in Section 10 | the downloaded or supplied ZIP is not the pinned (or expected) artifact | Re-run the Section 10 cell to download again; never change the pinned digest. With your own ZIP, ask its producer for the digest. |
| `does not follow this notebook's default data contract (SART10)` | the artifact was built from another dataset or an earlier default | Use an artifact exported from this notebook's current default path. |
| `external_artifact.json is missing` | a later Section 10 cell was run before the first one | Run the Section 10 cells in order (setup Sections 1–3 first). |
| `… does not have a deterministic implementation …` in Section 6 | an operation without a deterministic CUDA kernel | Report it with the log; do not switch the determinism flags off for a run of record. |
| `CUDA out of memory` in Section 6 | a large BYOD table | Untick `RUN_FINE_TUNING` (the run becomes exploratory), or restart on a fresh T4. |
| `fresh test not scored: pre-scoring check(s) failed` | Section 6 was not run after Section 4, or a file changed | Re-run from Section 4; a failed check is not a look. |
| `test_look: not scored (exploratory)` | a form field, the data source, BYOD or the device differs from the pre-registration | Expected: exploratory runs are compared on validation only. |
| `the target column … is not in the header` | your label column has another name | Set `TARGET_COLUMN`. |
| `column … is numeric except for N value(s)` | stray text in a numeric column | Fix the values, or list the column in `TEXT_COLUMNS`. |
| `need at least 50 labelled rows` / `Every class needs >= 2 rows` | a table below the floors | Supply more rows or merge rare classes. |
| `target classes unseen in training` | a validation/test class missing from the support rows | Re-split so every class is in the support partition. |
| `No path was given …` outside Colab, or `The upload was cancelled or empty` | no file supplied | Set the path field(s), or run the cell again and choose files. |

**Next experiments (all exploratory, validation only):** set `GRID_LEARNING_RATE` or `GRID_EPOCHS_CAP` and compare the curves; switch `DATA_SOURCE` to `Sample: Palmer Penguins` (categorical features); upload your own pre-split partitions; in a separate session, set `ARTIFACT_SOURCE` to your own ZIP in Section 10 and give it this run's `outputs/tabiclv2_classifier_artifact.zip` with its printed digest (an exercise of the optional branch, not external-artifact evidence).

## References

[1] J. Qu et al., "TabICLv2," arXiv:2602.11139, 2026. [Online]. Available: https://arxiv.org/abs/2602.11139

[2] J. Qu et al., "TabICL: A tabular foundation model for in-context learning on large data," arXiv:2502.05564, 2025. [Online]. Available: https://arxiv.org/abs/2502.05564

[3] R. Bock, "MAGIC Gamma Telescope," UCI Machine Learning Repository, 2004, doi: 10.24432/C52C8B. [Online]. Available: https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope

[4] soda-inria, "tabicl," software, version 2.1.1. [Online]. Available: https://github.com/soda-inria/tabicl

[5] jingang, "TabICL model checkpoints," Hugging Face. [Online]. Available: https://huggingface.co/jingang/TabICL

Repository documents:

- Repository model card: https://github.com/kurtvalcorza/tabicl-classifier-pipeline/blob/main/MODEL_CARD.md
- Fine-tuning pre-registration: https://github.com/kurtvalcorza/tabicl-classifier-pipeline/blob/main/docs/magic-finetune-preregistration.md
- Release verification: https://github.com/kurtvalcorza/tabicl-classifier-pipeline/blob/main/docs/release-verification.md

[6] K. Valcorza, "tabicl-classifier-pipeline: README, model card, weight provenance, sample dataset card, and the MAGIC pre-registration," GitHub repository. [Online]. Available: https://github.com/kurtvalcorza/tabicl-classifier-pipeline (see `README.md`, `MODEL_CARD.md`, `docs/WEIGHTS.md`, `examples/sample-data/DATASET_CARD.md`, `docs/magic-finetune-preregistration.md`)